In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import ticker
import pandas as pd
import numpy as np
import pyam
import colorbm as cbm
from waterfall_ax import WaterfallChart
from matplotlib.colors import ListedColormap
from pyam.plotting import add_net_values_to_bar_plot
from matplotlib.ticker import MultipleLocator 
pd.set_option("display.max_columns",400)
pd.set_option("display.max_rows",400)
pd.set_option("display.precision",0)
colors = pyam.plotting.PYAM_COLORS
sns.set_theme(font_scale=1.2)
sns.set_style("ticks")

plt.rcParams['font.family']=["SimHei"]
plt.rcParams['axes.unicode_minus']=False
# %config InlineBackend.figure_format='svg'#输出矢量图设置

In [ ]:
Data = pyam.IamDataFrame("figuredata20240315.xlsx",meta_sheet_name=False)
Data.append(pyam.IamDataFrame("GDP.xlsx",meta_sheet_name=False),inplace=True)
Data.filter(year=[2031,2032],keep=False)
ImgPath="C:\\Users\\44728\\OneDrive\\博五\\博士论文\\result\\"
Data.divide("Final Energy|Total Electricity","Final Energy","Electrification Rate",
            ignore_units="%",append=True)
renewable=["Primary Energy|Biomass","Primary Energy|Geothermal","Primary Energy|Hydro",
           "Primary Energy|Ocean","Primary Energy|Solar","Primary Energy|Wind"]
Fossil=["Primary Energy|Coal","Primary Energy|Oil","Primary Energy|Natural Gas"]
NonFossil=["Primary Energy|Nuclear","Primary Energy|Biomass","Primary Energy|Geothermal",
           "Primary Energy|Hydro","Primary Energy|Ocean","Primary Energy|Solar","Primary Energy|Wind"]
Data.divide("Primary Energy|Coal","Primary Energy","Coal Rate",ignore_units="%",append=True)
Data.divide(renewable,"Primary Energy","Renewable Rate",ignore_units="%",append=True)
Data.divide(NonFossil,"Primary Energy","Non-Fossil Rate",ignore_units="%",append=True)
Data.scenario

In [ ]:
#Sub-Annual Data
Data_subannual=pyam.IamDataFrame("figuredata20240315_subannual.xlsx")
Data_subannual.filter(year=[2031,2032],keep=False)
Scenario_ts=["ndc-ts","ndc-ts_2c-cn60","ndc-ts_2c-cn60-lm"]
S_name=["REF",'CN60','CN60-LM']
len_scen=len(Scenario_ts)
timeshort=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]

timeslice_share=[0.011872146,0.029680365,0.011872146,0.017808219,0.030136986,0.075342466,0.030136986,0.045205479,
                 0.011872146,0.029680365,0.011872146,0.017808219,0.029680365,0.074200913,0.029680365,0.044520548,
                 0.011872146,0.029680365,0.011872146,0.017808219,0.029680365,0.074200913,0.029680365,0.044520548,
                 0.011872146,0.029680365,0.011872146,0.017808219,0.029223744,0.073059361,0.029223744,0.043835616]


Data_subannual

# aggregate timeslice to seasonal level
Data_season=Data_subannual.aggregate_time(Data_subannual.variable,column='timeslice',value="3Fall",components="F*")
Data_season.append(Data_subannual.aggregate_time(Data_subannual.variable,column='timeslice',value="4Winter",components="W*"),inplace=True)
Data_season.append(Data_subannual.aggregate_time(Data_subannual.variable,column='timeslice',value="1Spring",components="R*"),inplace=True)
Data_season.append(Data_subannual.aggregate_time(Data_subannual.variable,column='timeslice',value="2Summer",components="S*"),inplace=True)
Data_season.to_excel("Tempdata_season.xlsx")
Data_season=pyam.IamDataFrame("Tempdata_season.xlsx")

def normal_to_24hours_marginal(Data):
    Timeslice_order=['RWeM','RWeD','RWeE','RWeN','RWdM','RWdD','RWdE','RWdN',
                     'SWeM','SWeD','SWeE','SWeN','SWdM','SWdD','SWdE','SWdN',
                     'FWeM','FWeD','FWeE','FWeN','FWdM','FWdD','FWdE','FWdN',
                     'WWeM','WWeD','WWeE','WWeN','WWdM','WWdD','WWdE','WWdN']
    weight=[4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6]
    count=5
    for i in range(0,32):
        for j in range(0,weight[i]):
            para=count%24
            match para:
                case 0:
                    Data.divide(Timeslice_order[i],1,'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True,append=True)
                case 1:
                    Data.divide(Timeslice_order[i],1,'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True,append=True)
                case 2:
                    Data.divide(Timeslice_order[i],1,'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True,append=True)
                case 3:
                    Data.divide(Timeslice_order[i],1,'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True,append=True)
                case 4:
                    Data.divide(Timeslice_order[i],1,'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True,append=True)
                case _:
                    Data.divide(Timeslice_order[i],1,'N_'+str(count).zfill(3),axis='timeslice',ignore_units=True,append=True)
            count=count+1
    return Data.filter(timeslice="N_*") 


def normal_to_24hours_consistent(Data):
    Timeslice_order=['RWeM','RWeD','RWeE','RWeN','RWdM','RWdD','RWdE','RWdN',
                     'SWeM','SWeD','SWeE','SWeN','SWdM','SWdD','SWdE','SWdN',
                     'FWeM','FWeD','FWeE','FWeN','FWdM','FWdD','FWdE','FWdN',
                     'WWeM','WWeD','WWeE','WWeN','WWdM','WWdD','WWdE','WWdN']
    weight=[4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6]
    count=5
    for i in range(0,32):
        for j in range(0,weight[i]):
            para=count%24
            match para:
                case 0:
                    Data.divide(Timeslice_order[i],timeslice_share[i]*8760,'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True,append=True)
                case 1:
                    Data.divide(Timeslice_order[i],timeslice_share[i]*8760,'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True,append=True)
                case 2:
                    Data.divide(Timeslice_order[i],timeslice_share[i]*8760,'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True,append=True)
                case 3:
                    Data.divide(Timeslice_order[i],timeslice_share[i]*8760,'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True,append=True)
                case 4:
                    Data.divide(Timeslice_order[i],timeslice_share[i]*8760,'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True,append=True)
                case _:
                    Data.divide(Timeslice_order[i],timeslice_share[i]*8760,'N_'+str(count).zfill(3),axis='timeslice',ignore_units=True,append=True)
            count=count+1
    return Data.filter(timeslice="N_*")  


def normal_to_24hours_consistent_TRA(Data):
    Timeslice_order=['RWeM','RWeD','RWeE','RWeN','RWdM','RWdD','RWdE','RWdN',
                     'SWeM','SWeD','SWeE','SWeN','SWdM','SWdD','SWdE','SWdN',
                     'FWeM','FWeD','FWeE','FWeN','FWdM','FWdD','FWdE','FWdN',
                     'WWeM','WWeD','WWeE','WWeN','WWdM','WWdD','WWdE','WWdN']
    weight=[4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6,4,10,4,6]
    Fast=[0.54565544,1.61560169,1.46996775,0.89184021,0.35775642,0.19970871,0.39307380,1.14977484,
          2.25744264,1.00962918,1.30906399,1.49878165,1.54335286,1.43664348,1.20254018,0.89151428,
          0.57302386,0.32375419,0.21169633,0.43707774,0.79645113,1.26250403,1.50396710,1.11917849]
    Slow=[1.05271605,1.05549768,1.00421571,0.91341357,0.79627756,1.41971197,1.12990437,0.85095333,
          0.59943033,0.91751747,0.54356495,0.31053759,0.23017187,0.30736703,0.54018500,0.91985043,
          1.43075072,2.05043596,2.74961899,0.76868000,0.93221835,1.08488580,1.21421585,1.17787943]
    count=5
    for i in range(0,32):
        for j in range(0,weight[i]):
            para=count%24
            match para:
                case 0:
                    Data.append(Data.filter(variable="*TRA_F*").divide(Timeslice_order[i],timeslice_share[i]*8760/Fast[para],'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
                    Data.append(Data.filter(variable="*TRA_S*").divide(Timeslice_order[i],timeslice_share[i]*8760/Slow[para],'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
                case 1:
                    Data.append(Data.filter(variable="*TRA_F*").divide(Timeslice_order[i],timeslice_share[i]*8760/Fast[para],'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
                    Data.append(Data.filter(variable="*TRA_S*").divide(Timeslice_order[i],timeslice_share[i]*8760/Slow[para],'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
                case 2:
                    Data.append(Data.filter(variable="*TRA_F*").divide(Timeslice_order[i],timeslice_share[i]*8760/Fast[para],'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
                    Data.append(Data.filter(variable="*TRA_S*").divide(Timeslice_order[i],timeslice_share[i]*8760/Slow[para],'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
                case 3:
                    Data.append(Data.filter(variable="*TRA_F*").divide(Timeslice_order[i],timeslice_share[i]*8760/Fast[para],'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
                    Data.append(Data.filter(variable="*TRA_S*").divide(Timeslice_order[i],timeslice_share[i]*8760/Slow[para],'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
                case 4:
                    Data.append(Data.filter(variable="*TRA_F*").divide(Timeslice_order[i],timeslice_share[i]*8760/Fast[para],'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
                    Data.append(Data.filter(variable="*TRA_S*").divide(Timeslice_order[i],timeslice_share[i]*8760/Slow[para],'N_'+str(count-24).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
                case _:
                    Data.append(Data.filter(variable="*TRA_F*").divide(Timeslice_order[i],timeslice_share[i]*8760/Fast[para],'N_'+str(count).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
                    Data.append(Data.filter(variable="*TRA_S*").divide(Timeslice_order[i],timeslice_share[i]*8760/Slow[para],'N_'+str(count).zfill(3),axis='timeslice',ignore_units=True),inplace=True)
            count=count+1
    return Data.filter(timeslice="N_*")

In [ ]:
# # calorific value calculation TO coal equivalent calculation
# Coalpower_coeff2019=2.4321 #41.12%
# Coalpower_coeff2020=2.4246 #41.24%
# Data.multiply('Primary Energy|Biomass',Coalpower_coeff2019,'PE(equivalent)|Biomass',append=True)
# Data.multiply('Primary Energy|Biomass|w/ CCS',Coalpower_coeff2019,'PE(equivalent)|Biomass|w/ CCS',append=True)
# Data.multiply('Primary Energy|Biomass|w/o CCS',Coalpower_coeff2019,'PE(equivalent)|Biomass|w/o CCS',append=True)
# Data.multiply('Primary Energy|Coal',1,'PE(equivalent)|Coal',append=True)
# Data.multiply('Primary Energy|Coal|w/ CCS',1,'PE(equivalent)|Coal|w/ CCS',append=True)
# Data.multiply('Primary Energy|Coal|w/o CCS',1,'PE(equivalent)|Coal|w/o CCS',append=True)
# Data.multiply('Primary Energy|Geothermal',Coalpower_coeff2019,'PE(equivalent)|Geothermal',append=True)
# Data.multiply('Primary Energy|Hydro',Coalpower_coeff2019,'PE(equivalent)|Hydro',append=True)
# Data.multiply('Primary Energy|Natural gas',1,'PE(equivalent)|Natural gas',append=True)
# Data.multiply('Primary Energy|Natural gas|w/ CCS',1,'PE(equivalent)|Natural gas|w/ CCS',append=True)
# Data.multiply('Primary Energy|Natural gas|w/o CCS',1,'PE(equivalent)|Natural gas|w/o CCS',append=True)
# Data.multiply('Primary Energy|Nuclear',Coalpower_coeff2019,'PE(equivalent)|Nuclear',append=True)
# Data.multiply('Primary Energy|Ocean',Coalpower_coeff2019,'PE(equivalent)|Ocean',append=True)
# Data.multiply('Primary Energy|Oil',1,'PE(equivalent)|Oil',append=True)
# Data.multiply('Primary Energy|Oil|w/ CCS',1,'PE(equivalent)|Oil|w/ CCS',append=True)
# Data.multiply('Primary Energy|Oil|w/o CCS',1,'PE(equivalent)|Oil|w/o CCS',append=True)
# Data.multiply('Primary Energy|Solar',Coalpower_coeff2019,'PE(equivalent)|Solar',append=True)
# Data.multiply('Primary Energy|Wind',Coalpower_coeff2019,'PE(equivalent)|Wind',append=True)
# Data.aggregate('PE(equivalent)',append=True)
# renewable_equivalent=["PE(equivalent)|Biomass","PE(equivalent)|Geothermal","PE(equivalent)|Hydro",
#            "PE(equivalent)|Ocean","PE(equivalent)|Solar","PE(equivalent)|Wind"]
# Fossil_equivalent=["PE(equivalent)|Coal","PE(equivalent)|Oil","PE(equivalent)|Natural Gas"]
# NonFossil_equivalent=["PE(equivalent)|Nuclear","PE(equivalent)|Biomass","PE(equivalent)|Geothermal",
#            "PE(equivalent)|Hydro","PE(equivalent)|Ocean","PE(equivalent)|Solar","PE(equivalent)|Wind"]

# Data.divide("PE(equivalent)|Coal","PE(equivalent)","Coal Rate(equivalent)",ignore_units="%",append=True)
# Data.divide(renewable_equivalent,"PE(equivalent)","Renewable Rate(equivalent)",ignore_units="%",append=True)
# Data.divide(NonFossil_equivalent,"PE(equivalent)","Non-Fossil Rate(equivalent)",ignore_units="%",append=True)

In [ ]:
#GAINS结果

HealthImpact=pd.read_excel(r"C:\Users\44728\OneDrive\博五\博士论文\gains\HealthImpact2024.xlsx")
#HealthImpact.drop(['2025','2035','2040','2045'],axis=1,inplace=True)
HealthImpact=HealthImpact[HealthImpact['scenario']!='CN60-CLE']
HealthImpact=HealthImpact.melt(id_vars=['scenario','Region','Variable'],var_name='时间',value_name='value')
HealthImpact.columns=['情景','地区','Variable','时间','value']
HealthImpact.replace('CN50-MFR','CN60-SDG',inplace=True)
HealthImpact.replace('CN50-CLE','CN60',inplace=True)

Death=HealthImpact[HealthImpact['Variable']!='Mean PM2.5Concentration']
Pop=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\gains\分省人口.xlsx')
Pop=Pop.melt(id_vars="地区",var_name='时间',value_name='value')
Data_GAINS=pyam.IamDataFrame(Pop,model='China TIMES 2.0',scenario='NDC',region='地区',variable='Population',Unit='million',year='时间')
Data_GAINS.append(pyam.IamDataFrame(Pop,model='China TIMES 2.0',scenario='CN60',region='地区',variable='Population',Unit='million',year='时间'),inplace=True)
Data_GAINS.append(pyam.IamDataFrame(Pop,model='China TIMES 2.0',scenario='CN60-SDG',region='地区',variable='Population',Unit='million',year='时间'),inplace=True)
Data_GAINS.append(pyam.IamDataFrame(Death,model='China TIMES 2.0',scenario='情景',region='地区',variable='Variable',Unit='p',year='时间'),inplace=True)
Data_GAINS.divide('Premature deaths','Population','每百万人死亡人数',ignore_units="人/百万人",append=True)
Province_order=Death.地区[0:31]
Data_GAINS=Data_GAINS.filter(variable='每百万人死亡人数').as_pandas()
Data_GAINS.columns=['模型','情景','地区','Variable','unit','时间','value']
Data_GAINS.value=Data_GAINS.value/100

ByRegion=pd.read_excel(r"C:\Users\44728\OneDrive\博五\博士论文\gains\EMISSIONS\ByRegion.xlsx")
ByRegion.drop(2015,axis=1,inplace=True)
ByRegion=ByRegion[ByRegion['scenario']!='CN60-CLE']
ByRegion.replace("CN50-MFR","CN60-SDG",inplace=True)
ByRegion.replace("CN50-CLE","CN60",inplace=True)
ByRegion=ByRegion.melt(id_vars=['scenario','Emissions(kt)','region'],var_name='时间',value_name='value')
ByRegion.value=ByRegion.value/1000
ByRegion=ByRegion[ByRegion['region']=='Sum']
ByRegion.columns=['情景','污染物','region','时间','value']
#成本收益分析
cost1=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\gains\COSTS\cost-brief.xlsx',0)
cost2=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\gains\COSTS\cost-brief.xlsx',1)
cost3=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\gains\COSTS\cost-brief.xlsx',2)
cost=pd.concat([cost1,cost2,cost3]).drop(2015,axis=1)
Cost=pyam.IamDataFrame(cost,model='China TIMES 2.0',scenario='情景',region='地区',variable='Variable',Unit='million Euro')
Cost.convert_unit('million Euro','billion $',1/1000*7.5/6,inplace=True)
Cost=Cost.append(pyam.IamDataFrame(Death,model='China TIMES 2.0',scenario='情景',region='地区',variable='Variable',Unit='p',year='时间'))
Cost.convert_unit('p','billion $',-1/1000,inplace=True)
Cost.rename(variable={'Premature deaths':'收益'},inplace=True)
Cost.aggregate_region('收益','Sum',append=True)
Cost.subtract('CN60-SDG','CN60',name="控制措施",axis='scenario',ignore_units=True,append=True)
Cost.subtract('CN60','NDC',name="协同效应",axis='scenario',ignore_units=True,append=True)
Cost.subtract('收益','成本',name="净收益",axis='variable',ignore_units=True,append=True)
Cost.multiply('成本',-1,'成本增加',ignore_units=True,append=True)
Cost.append(pyam.IamDataFrame(Pop,model='China TIMES 2.0',scenario='NDC',region='地区',variable='Population',Unit='million',year='时间'),inplace=True)
Cost.append(pyam.IamDataFrame(Pop,model='China TIMES 2.0',scenario='CN60',region='地区',variable='Population',Unit='million',year='时间'),inplace=True)
Cost.append(pyam.IamDataFrame(Pop,model='China TIMES 2.0',scenario='CN60-SDG',region='地区',variable='Population',Unit='million',year='时间'),inplace=True)
Cost.append(pyam.IamDataFrame(Pop,model='China TIMES 2.0',scenario='控制措施',region='地区',variable='Population',Unit='million',year='时间'),inplace=True)
Cost.append(pyam.IamDataFrame(Pop,model='China TIMES 2.0',scenario='协同效应',region='地区',variable='Population',Unit='million',year='时间'),inplace=True)
Cost.divide('净收益','Population','人均净收益',ignore_units="千美元/人",append=True)
Cost.divide('收益','Population','人均收益',ignore_units="千美元/人",append=True)
Cost.divide('成本增加','Population','人均成本增加',ignore_units="千美元/人",append=True)

In [ ]:
color_map = {
 'Capacity|Electricity|Biomass-Coal|w/ CCS':'#A1D99B',
 'Capacity|Electricity|Biomass-Coal|w/o CCS':'#C7E9C0',
 'Capacity|Electricity|Biomass|w/ CCS':'#31A354',
 'Capacity|Electricity|Biomass|w/o CCS':'#74C476',
 'Capacity|Electricity|Coal|w/ CCS':'#636363',
 'Capacity|Electricity|Coal|w/o CCS':'#969696',
 'Capacity|Electricity|Gas|w/ CCS':'#FD8D3C',
 'Capacity|Electricity|Gas|w/o CCS':'#FDAE6B',
 'Capacity|Electricity|Geothermal':'#F7B6D2',
 'Capacity|Electricity|Hydro':'#9EDAE5',
 'Capacity|Electricity|Nuclear':'#918DB8',
 'Capacity|Electricity|Ocean':'#9ECAE1',
 'Capacity|Electricity|Oil|w/ CCS':'#DBB48C',
 'Capacity|Electricity|Oil|w/o CCS':'#FDD0A2',
 'Capacity|Electricity|Solar':'#EDD783',
 'Capacity|Electricity|Wind':'#6BAED6',
    'Capacity|Electricity|Air-cooling':'#74C476',
    'Capacity|Electricity|Once-Through':'#FD8D3C',
    'Capacity|Electricity|Recirculating':'#9EDAE5',
 'Secondary Energy|Electricity|Biomass-Coal|w/ CCS':'#A1D99B',
 'Secondary Energy|Electricity|Biomass-Coal|w/o CCS':'#C7E9C0',
 'Secondary Energy|Electricity|Biomass|w/ CCS':'#31A354',
 'Secondary Energy|Electricity|Biomass|w/o CCS':'#74C476',
 'Secondary Energy|Electricity|Coal|w/ CCS':'#636363',
 'Secondary Energy|Electricity|Coal|w/o CCS':'#969696',
 'Secondary Energy|Electricity|Gas|w/ CCS':'#FD8D3C',
 'Secondary Energy|Electricity|Gas|w/o CCS':'#FDAE6B',
 'Secondary Energy|Electricity|Geothermal':'#F7B6D2',
 'Secondary Energy|Electricity|Hydro':'#9EDAE5',
 'Secondary Energy|Electricity|Nuclear':'#918DB8',
 'Secondary Energy|Electricity|Ocean':'#9ECAE1',
 'Secondary Energy|Electricity|Oil|w/ CCS':'#DBB48C',
 'Secondary Energy|Electricity|Oil|w/o CCS':'#FDD0A2',
 'Secondary Energy|Electricity|Solar':'#EDD783',
 'Secondary Energy|Electricity|Wind':'#6BAED6',
       'Secondary Energy|Hydrogen|Distribute|Solar':'#EDD783',
       'Secondary Energy|Hydrogen|Distribute|Onshore Wind':'#6BAED6',
       'Secondary Energy|Hydrogen|Distribute|Offshore Wind':'#5991B3',
       'Secondary Energy|Hydrogen|Electricity':'#3182BD',
       'Secondary Energy|Hydrogen|Biomass|w/o CCS':'#74C476',
       'Secondary Energy|Hydrogen|Biomass|w/ CCS':'#31A354',
       'Secondary Energy|Hydrogen|Gas|w/o CCS':'#FDAE6B',
       'Secondary Energy|Hydrogen|Gas|w/ CCS':'#FD8D3C',
       'Secondary Energy|Hydrogen|Oil|w/o CCS':'#FDD0A2',
       'Secondary Energy|Hydrogen|Oil|w/ CCS':'#DBB48C',
       'Secondary Energy|Hydrogen|Coal|w/o CCS':'#969696',
       'Secondary Energy|Hydrogen|Coal|w/ CCS':'#636363',
    'Primary Energy|Solar':'#EDD783',
    'Primary Energy|Wind':'#6BAED6',
    'Primary Energy|Ocean':'#9ECAE1',
    'Primary Energy|Geothermal':'#F7B6D2',
    'Primary Energy|Hydro':'#9EDAE5',
    'Primary Energy|Biomass|w/o CCS':'#74C476',
    'Primary Energy|Biomass|w/ CCS':'#31A354',
    'Primary Energy|Nuclear':'#918DB8',
    'Primary Energy|Natural gas|w/o CCS':'#FDAE6B',
    'Primary Energy|Natural gas|w/ CCS':'#FD8D3C',
    'Primary Energy|Oil|w/o CCS':'#FDD0A2',
    'Primary Energy|Oil|w/ CCS':'#AD6B5D',
    'Primary Energy|Coal|w/o CCS':'#969696',
    'Primary Energy|Coal|w/ CCS':'#636363',
        'PE(equivalent)|Solar':'#EDD783',
        'PE(equivalent)|Wind':'#6BAED6',
        'PE(equivalent)|Ocean':'#9ECAE1',
        'PE(equivalent)|Geothermal':'#F7B6D2',
        'PE(equivalent)|Hydro':'#9EDAE5',
        'PE(equivalent)|Biomass|w/o CCS':'#74C476',
        'PE(equivalent)|Biomass|w/ CCS':'#31A354',
        'PE(equivalent)|Nuclear':'#918DB8',
        'PE(equivalent)|Natural gas|w/o CCS':'#FDAE6B',
        'PE(equivalent)|Natural gas|w/ CCS':'#FD8D3C',
        'PE(equivalent)|Oil|w/o CCS':'#FDD0A2',
        'PE(equivalent)|Oil|w/ CCS':'#AD6B5D',
        'PE(equivalent)|Coal|w/o CCS':'#969696',
        'PE(equivalent)|Coal|w/ CCS':'#636363',
 'Final Energy|Total Biomass':'#2CA02C',
 'Final Energy|Total Coal':'#636363',
 'Final Energy|Total Electricity':'#74C476',
 'Final Energy|Total Gas':'#FD8D3C',
 'Final Energy|Total Geothermal':'#F7B6D2',
 'Final Energy|Total Heat':'#F26A67',
 'Final Energy|Total Hydrogen':'#3182BD',
 'Final Energy|Total Liquids':'#AD6B5D',
 'Final Energy|Total Solar':'#EDD783',
         'Final Energy|Industry|Coal':'#636363',
         'Final Energy|Industry|Electricity':'#74C476',
         'Final Energy|Industry|Gas':'#FD8D3C',
         'Final Energy|Industry|Heat':'#F26A67',
         'Final Energy|Industry|Hydrogen':'#3182BD',
         'Final Energy|Industry|Liquids':'#AD6B5D',
         'Final Energy|Building|Biomass':'#2CA02C',
         'Final Energy|Building|Coal':'#636363',
         'Final Energy|Building|Electricity':'#74C476',
         'Final Energy|Building|Gas':'#FD8D3C',
         'Final Energy|Building|Geothermal':'#F7B6D2',
         'Final Energy|Building|Heat':'#F26A67',
         'Final Energy|Building|Hydrogen':'#3182BD',
         'Final Energy|Building|Liquids':'#AD6B5D',
         'Final Energy|Building|Solar':'#EDD783',
         'Final Energy|Transport|Coal':'#636363',
         'Final Energy|Transport|Electricity':'#74C476',
         'Final Energy|Transport|Gas':'#FD8D3C',
         'Final Energy|Transport|Hydrogen':'#3182BD',
         'Final Energy|Transport|Liquids':'#AD6B5D',
'Final Energy|Industry|IIS|Coal':'#969696',
'Final Energy|Industry|IIS|Coal CCS':'#636363',
'Final Energy|Industry|IIS|Electricity':'#74C476',
'Final Energy|Industry|IIS|Gas':'#FDAE6B',
'Final Energy|Industry|IIS|Gas CCS':'#FD8D3C',
'Final Energy|Industry|IIS|Heat':'#F26A67',
'Final Energy|Industry|IIS|Hydrogen':'#3182BD',
'Final Energy|Industry|IIS|Liquids':'#FDD0A2',
'Final Energy|Industry|IIS|Liquids CCS':'#AD6B5D',

'Final Energy|Industry|ILP|Coal':'#969696',
'Final Energy|Industry|ILP|Coal CCS':'#636363',
'Final Energy|Industry|ILP|Electricity':'#74C476',
'Final Energy|Industry|ILP|Gas':'#FDAE6B',
'Final Energy|Industry|ILP|Gas CCS':'#FD8D3C',
'Final Energy|Industry|ILP|Heat':'#F26A67',
'Final Energy|Industry|ILP|Hydrogen':'#3182BD',
'Final Energy|Industry|ILP|Liquids':'#FDD0A2',
'Final Energy|Industry|ILP|Liquids CCS':'#AD6B5D',
 'Final Energy|Industry|INF(Al)|Coal':'#969696',
 'Final Energy|Industry|INF(Al)|Electricity':'#74C476',
 'Final Energy|Industry|INF(Al)|Gas':'#FDAE6B',
 'Final Energy|Industry|INF(Al)|Heat':'#F26A67',
 'Final Energy|Industry|INF(Al)|Liquids':'#FDD0A2',
 'Final Energy|Industry|INF(Cu)|Coal':'#969696',
 'Final Energy|Industry|INF(Cu)|Electricity':'#74C476',
 'Final Energy|Industry|INF(Cu)|Gas':'#FDAE6B',
 'Final Energy|Industry|INF(Cu)|Heat':'#F26A67',
 'Final Energy|Industry|INF(Cu)|Liquids':'#FDD0A2',
 'Final Energy|Industry|INF(Pb)|Coal':'#969696',
 'Final Energy|Industry|INF(Pb)|Electricity':'#74C476',
 'Final Energy|Industry|INF(Pb)|Gas':'#FDAE6B',
 'Final Energy|Industry|INF(Zn)|Coal':'#969696',
 'Final Energy|Industry|INF(Zn)|Electricity':'#74C476',
 'Final Energy|Industry|INF(Zn)|Gas':'#FDAE6B',
 'Final Energy|Industry|ICH(C2H4)|Liquids':'#FDD0A2',
 'Final Energy|Industry|ICH(NH3)|Coal':'#969696',
 'Final Energy|Industry|ICH(NH3)|Coal CCS':'#636363',
 'Final Energy|Industry|ICH(NH3)|Electricity':'#74C476',
 'Final Energy|Industry|ICH(NH3)|Gas':'#FDAE6B',
 'Final Energy|Industry|ICH(NH3)|Gas CCS':'#FD8D3C',
 'Final Energy|Industry|ICH(NH3)|Heat':'#F26A67',
 'Final Energy|Industry|ICH(NH3)|Hydrogen':'#3182BD',
 'Final Energy|Industry|ICH(NH3)|Liquids':'#FDD0A2',
 'Final Energy|Industry|ICH(NH3)|Liquids CCS':'#AD6B5D',
 'Final Energy|Industry|ICH(Na2CO3)|Electricity':'#74C476',
 'Final Energy|Industry|ICH(Na2CO3)|Heat':'#F26A67',
 'Final Energy|Industry|ICH(NaOH)|Electricity':'#74C476',
 'Final Energy|Industry|ICH(NaOH)|Heat':'#F26A67',
 'Final Energy|Industry|IBU(Cement)|Coal':'#969696',
 'Final Energy|Industry|IBU(Cement)|Coal CCS':'#636363',
 'Final Energy|Industry|IBU(Cement)|Electricity':'#74C476',
 'Final Energy|Industry|IBU(Glass)|Coal':'#969696',
 'Final Energy|Industry|IBU(Glass)|Electricity':'#74C476',
 'Final Energy|Industry|IBU(Glass)|Liquids':'#FDD0A2',
    'Final Energy|Agriculture':'#47A359',
    'Final Energy|Transport':'#6BAED6',
    'Final Energy|Building':'#9E9AC8',
    'Final Energy|Industry':'#FDD0A2',
 'Emissions|CO2|AGR':'#47A359',
 'Emissions|CO2|BLD':'#9E9AC8',
 'Emissions|CO2|ELC':'#BDBDBD',
 'Emissions|CO2|IND':'#FD8D3C',
 'Emissions|CO2|INDPROC':'#FDD0A2',
 'Emissions|CO2|LUCF':'#A1D99B',
 'Emissions|CO2|TRA':'#6BAED6',
 'Emissions|CO2|UPS':'#636363',
         'Emissions|GHG|AGR':'#47A359',
         'Emissions|GHG|BLD':'#9E9AC8',
         'Emissions|GHG|ELC':'#BDBDBD',
         'Emissions|GHG|IND':'#FD8D3C',
         'Emissions|GHG|LUCF':'#A1D99B',
         'Emissions|GHG|TRA':'#6BAED6',
         'Emissions|GHG|UPS':'#636363',
     'Emissions|CH4|AGR':'#47A359',
     'Emissions|CH4|BLD':'#9E9AC8',
     'Emissions|CH4|ELC':'#BDBDBD',
     'Emissions|CH4|IND':'#FD8D3C',
     'Emissions|CH4|TRA':'#6BAED6',
     'Emissions|CH4|UPS':'#636363',
 'Emissions|N2O|AGR':'#47A359',
 'Emissions|N2O|BLD':'#9E9AC8',
 'Emissions|N2O|ELC':'#BDBDBD',
 'Emissions|N2O|IND':'#FD8D3C',
 'Emissions|N2O|TRA':'#6BAED6',
 'Emissions|N2O|UPS':'#636363',
     'Emissions|CH4|LUCF|AGR|Enteric':'#588556',
     'Emissions|CH4|LUCF|AGR|Manure':'#6FA86D',
     'Emissions|CH4|LUCF|AGR|Rice':'#89CF86',
     'Emissions|CH4|LUCF|Savannah':'#CF98BF',
     'Emissions|CH4|LUCF|Waste':'#9BA3C2',
     'Emissions|N2O|LUCF|AGR|Enteric':'#588556',
     'Emissions|N2O|LUCF|AGR|Manure':'#6FA86D',
     'Emissions|N2O|LUCF|AGR|Rice':'#89CF86',
     'Emissions|N2O|LUCF|Savannah':'#CF98BF',
     'Emissions|N2O|LUCF|Waste':'#9BA3C2',
           'Emissions|CCS|CO2|Upstream|DAC':'#F7B6D2',
           'Emissions|CCS|CO2|Upstream|Biomass':'#74C476',
           'Emissions|CCS|CO2|Electricity|Biomass':'#31A354',
           'Emissions|CCS|CO2|Upstream|Fossil':'#969696',
           'Emissions|CCS|CO2|Electricity|Coal':'#636363',
           'Emissions|CCS|CO2|Electricity|Oil':'#AD6B5D',   
           'Emissions|CCS|CO2|Electricity|Gas':'#FD8D3C',
           'Emissions|CCS|CO2|Upstream|Heat':'#F26A67',
           'Emissions|CCS|CO2|Industry|Cement':'#CF4141',   
           'Emissions|CCS|CO2|Industry|IIS':'#6BAED6',
           'Emissions|CCS|CO2|Industry|NH3':'#EDD783',
 'BLD_Heat|COM|C|Coal':'#636363',
 'BLD_Heat|COM|C|Electricity':'#6BAED6',
 'BLD_Heat|COM|C|Gas':'#FD8D3C',
 'BLD_Heat|COM|C|Geothermal':'#F7B6D2',
 'BLD_Heat|COM|C|Heat':'#F26A67',
 'BLD_Heat|COM|C|Oil':'#AD6B5D',
 'BLD_Heat|COM|HSCW|Coal':'#636363',
 'BLD_Heat|COM|HSCW|Electricity':'#6BAED6',
 'BLD_Heat|COM|HSCW|Gas':'#FD8D3C',
 'BLD_Heat|COM|HSCW|Geothermal':'#F7B6D2',
 'BLD_Heat|COM|HSCW|Heat':'#F26A67',
 'BLD_Heat|COM|HSCW|Oil':'#AD6B5D',
 'BLD_Heat|COM|HSWW|Coal':'#636363',
 'BLD_Heat|COM|HSWW|Electricity':'#6BAED6',
 'BLD_Heat|COM|HSWW|Gas':'#FD8D3C',
 'BLD_Heat|COM|HSWW|Geothermal':'#F7B6D2',
 'BLD_Heat|COM|HSWW|Heat':'#F26A67',
 'BLD_Heat|COM|HSWW|Oil':'#AD6B5D',
 'BLD_Heat|COM|M|Coal':'#636363',
 'BLD_Heat|COM|M|Electricity':'#6BAED6',
 'BLD_Heat|COM|M|Gas':'#FD8D3C',
 'BLD_Heat|COM|M|Geothermal':'#F7B6D2',
 'BLD_Heat|COM|M|Heat':'#F26A67',
 'BLD_Heat|COM|M|Oil':'#AD6B5D',
 'BLD_Heat|COM|SC|Coal':'#636363',
 'BLD_Heat|COM|SC|Electricity':'#6BAED6',
 'BLD_Heat|COM|SC|Gas':'#FD8D3C',
 'BLD_Heat|COM|SC|Geothermal':'#F7B6D2',
 'BLD_Heat|COM|SC|Heat':'#F26A67',
 'BLD_Heat|COM|SC|Oil':'#AD6B5D',
 'BLD_Heat|RUL|C|Biogas':'#74C476',
 'BLD_Heat|RUL|C|Biomass':'#2CA02C',
 'BLD_Heat|RUL|C|Coal':'#636363',
 'BLD_Heat|RUL|C|Electricity':'#6BAED6',
 'BLD_Heat|RUL|C|Gas':'#FD8D3C',
 'BLD_Heat|RUL|C|Geothermal':'#F7B6D2',
 'BLD_Heat|RUL|HSCW|Biogas':'#74C476',
 'BLD_Heat|RUL|HSCW|Biomass':'#2CA02C',
 'BLD_Heat|RUL|HSCW|Coal':'#636363',
 'BLD_Heat|RUL|HSCW|Electricity':'#6BAED6',
 'BLD_Heat|RUL|HSCW|Gas':'#FD8D3C',
 'BLD_Heat|RUL|HSCW|Geothermal':'#F7B6D2',
 'BLD_Heat|RUL|HSWW|Biogas':'#74C476',
 'BLD_Heat|RUL|HSWW|Biomass':'#2CA02C',
 'BLD_Heat|RUL|HSWW|Coal':'#636363',
 'BLD_Heat|RUL|HSWW|Electricity':'#6BAED6',
 'BLD_Heat|RUL|HSWW|Gas':'#FD8D3C',
 'BLD_Heat|RUL|HSWW|Geothermal':'#F7B6D2',
 'BLD_Heat|RUL|M|Biogas':'#74C476',
 'BLD_Heat|RUL|M|Biomass':'#2CA02C',
 'BLD_Heat|RUL|M|Coal':'#636363',
 'BLD_Heat|RUL|M|Electricity':'#6BAED6',
 'BLD_Heat|RUL|M|Gas':'#FD8D3C',
 'BLD_Heat|RUL|M|Geothermal':'#F7B6D2',
 'BLD_Heat|RUL|SC|Biogas':'#74C476',
 'BLD_Heat|RUL|SC|Biomass':'#2CA02C',
 'BLD_Heat|RUL|SC|Coal':'#636363',
 'BLD_Heat|RUL|SC|Electricity':'#6BAED6',
 'BLD_Heat|RUL|SC|Gas':'#FD8D3C',
 'BLD_Heat|RUL|SC|Geothermal':'#F7B6D2',
 'BLD_Heat|URB|C|Coal':'#636363',
 'BLD_Heat|URB|C|Electricity':'#6BAED6',
 'BLD_Heat|URB|C|Gas':'#FD8D3C',
 'BLD_Heat|URB|C|Geothermal':'#F7B6D2',
 'BLD_Heat|URB|C|Heat':'#F26A67',
 'BLD_Heat|URB|C|Oil':'#AD6B5D',
 'BLD_Heat|URB|HSCW|Coal':'#636363',
 'BLD_Heat|URB|HSCW|Electricity':'#6BAED6',
 'BLD_Heat|URB|HSCW|Gas':'#FD8D3C',
 'BLD_Heat|URB|HSCW|Geothermal':'#F7B6D2',
 'BLD_Heat|URB|HSCW|Heat':'#F26A67',
 'BLD_Heat|URB|HSCW|Oil':'#AD6B5D',
 'BLD_Heat|URB|HSWW|Coal':'#636363',
 'BLD_Heat|URB|HSWW|Electricity':'#6BAED6',
 'BLD_Heat|URB|HSWW|Gas':'#FD8D3C',
 'BLD_Heat|URB|HSWW|Geothermal':'#F7B6D2',
 'BLD_Heat|URB|HSWW|Heat':'#F26A67',
 'BLD_Heat|URB|HSWW|Oil':'#AD6B5D',
 'BLD_Heat|URB|M|Coal':'#636363',
 'BLD_Heat|URB|M|Electricity':'#6BAED6',
 'BLD_Heat|URB|M|Gas':'#FD8D3C',
 'BLD_Heat|URB|M|Geothermal':'#F7B6D2',
 'BLD_Heat|URB|M|Heat':'#F26A67',
 'BLD_Heat|URB|M|Oil':'#AD6B5D',
 'BLD_Heat|URB|SC|Coal':'#636363',
 'BLD_Heat|URB|SC|Electricity':'#6BAED6',
 'BLD_Heat|URB|SC|Gas':'#FD8D3C',
 'BLD_Heat|URB|SC|Geothermal':'#F7B6D2',
 'BLD_Heat|URB|SC|Heat':'#F26A67',
 'BLD_Heat|URB|SC|Oil':'#AD6B5D',
 'BLD_Heat|RUL|C|Oil':'#AD6B5D',
 'BLD_Heat|RUL|HSCW|Oil':'#AD6B5D',
 'BLD_Heat|RUL|HSWW|Oil':'#AD6B5D',
 'BLD_Heat|RUL|M|Oil':'#AD6B5D',
 'BLD_Heat|RUL|SC|Oil':'#AD6B5D',
    
'COM_LED':'#74C476',
'COM_FLU':'#6BAED6',
'COM_HAL':'#F26A67',
'COM_INC':'#636363',
'RUL_LED':'#74C476',
'RUL_FLU':'#6BAED6',
'RUL_HAL':'#F26A67',
'RUL_INC':'#636363',
'URB_LED':'#74C476',
'URB_FLU':'#6BAED6',
'URB_HAL':'#F26A67',
'URB_INC':'#636363',

'RUL_BGS':'#31A354',
'RUL_BSL':'#2CA02C',
'RUL_COA':'#636363',
'RUL_ELC':'#F26A67',
'RUL_GAS':'#FD8D3C',
'RUL_GEO':'#F7B6D2',
'RUL_OIL':'#AD6B5D',
'RUL_SOL':'#EDD783',
'COM_COA':'#636363',
'COM_ELC':'#F26A67',
'COM_GAS':'#FD8D3C',
'COM_GEO':'#F7B6D2',
'COM_OIL':'#AD6B5D',
'COM_SOL':'#EDD783',
'URB_COA':'#636363',
'URB_ELC':'#F26A67',
'URB_GAS':'#FD8D3C',
'URB_GEO':'#F7B6D2',
'URB_OIL':'#AD6B5D',
'URB_SOL':'#EDD783',    
    'Storage|Hydrogen|Discharge':'#802828',
    'Storage|Hydrogen|Charge':'#CF4141',   
    'Storage|Electricity|Charge':'#802828',
    'Storage|Electricity|Discharge':'#CF4141',

 'LUCF|AGR|Demand|Bioenergy|First-generation':'#c82827',
 'LUCF|AGR|Demand|Bioenergy|Second-generation':'#e89c9c',
 'LUCF|AGR|Demand|Food|Crop':'#57a035',
 'LUCF|AGR|Demand|Food|Livestock':'#b1cee2',
 'LUCF|AGR|Demand|Feed|Crop':'#bbe08e',
 'LUCF|AGR|Demand|Industrial use|Crop':'#84b7ad',
 'LUCF|AGR|Demand|Industrial use|Livestock':'#4677b2',
    
 'LUCF|AGR|Production|Energy crop':'#dd8554',
 'LUCF|AGR|Production|Non-energy crop':'#eddc92',
 'LUCF|AGR|Production|Livestock':'#6da9eb',

'LUCF|FOOD|CROP':"#eddc92",
'LUCF|FOOD|LIVESTOCK':"#6da9eb",
'V2G-Charge':'#81d57c',
'V2G-Discharge':'#62a25e',
'Load Management|TRA|V2G|Charge':'#81d57c',
'Load Management|TRA|V2G|Discharge':'#62a25e',
"Load Management|TRA_Shift":'#8c564b',
'Load Management|TRA_F':'#3478b1',
'Load Management|TRA_S':'#a6dae4',
'Water Demand|Water Withdrawal|UPS|Ecoflow':"#66c2a5",
'Water Demand|Water Withdrawal|AGR':"#bcdf8f",
'Water Demand|Water Withdrawal|UPS|Energy':"#e79d9b",
'Water Demand|Water Withdrawal|ELC':"#f0c077",
'Water Demand|Water Withdrawal|IND':"#4676b2",
'Water Demand|Water Withdrawal|BLD':"#b1cee2",
'UPSWW|Ecoflow':"#66c2a5",
'AGRWW':"#bcdf8f",
'UPSWW|Energy':"#e79d9b",
'ELCWW':"#f0c077",
'INDWW':"#4676b2",
'BLDWW':"#b1cee2",

'H2OGFW':"#a9caea",
'H2OSFW':"#7ab8d2",
'H2OSALT':"#f3cd86",
'H2OWASTE':"#c297c0",
'Water Supply|Ground Water':"#a9caea",
'Water Supply|Surface Water':"#7ab8d2",
'Water Supply|Salt Water':"#f3cd86",
'Water Supply|Waste Water':"#c297c0",
}
pyam.run_control().update({'color': {'variable': color_map}})

In [ ]:
#读取AR6数据库
df=pyam.read_iiasa(name='ar6-public',variable=['Emissions|CO2'],region='China')
df=df.filter(year=range(2015,2105,5),Category=['C2','C3','C4','C6','C7'])

In [ ]:
cmap_co2tot={ 'ndc-ts':'#880015','ndc-ts_2c-cn60':'#2CA02C'}
pyam.run_control().update({'color': {'scenario': cmap_co2tot}})
cmap_co2totAR6={ 'C2':'AR6-C2','C3':'AR6-C3','C4':'AR6-C4','C6':'AR6-C6','C7':'AR6-C7'}
pyam.run_control().update({'color': {'Category': cmap_co2totAR6}})

fig, ax =plt.subplots()
CO2total = Data.filter(variable = ['Emissions|CO2|TRA','Emissions|CO2|BLD','Emissions|CO2|AGR','Emissions|CO2|INDPROC','Emissions|CO2|IND','Emissions|CO2|UPS','Emissions|CO2|ELC'],scenario=['ndc-ts','ndc-ts_2c-cn60']).aggregate("Emissions|CO2").filter(keep=False,year=[2022,2031,2032])
CO2total = CO2total.convert_unit('Mt', to='亿吨', factor=0.01)
CO2total.plot(ax=ax,color='scenario',title=False,linewidth=2)
#AR6范围
df=df.convert_unit('Mt CO2/yr','亿吨',0.01)
df.filter(scenario='E*').plot(ax=ax,color='Category',alpha=0,fill_between=dict(alpha=0.3),title=False,legend=False,final_ranges=dict(linewidth=4))
plt.legend(['REF','CN60'],loc='lower left',frameon=False)
plt.xlim([2010,2110])
plt.ylabel("二氧化碳排放（亿吨CO$_2$）")
plt.xlabel("时间")
ax.text(2101,30,"C2",color='#778663')
ax.text(2101,50,"C3",color='#6F7899')
ax.text(2101,70,"C4",color='#A7C682')
ax.text(2101,90,"C6",color='#FAC182')
ax.text(2101,110,"C7",color='#F18872')


plt.tight_layout()
# plt.savefig(ImgPath+"chap3_CO2_AR6"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
CO2 = Data.filter(variable = "Emissions|CO2|*",level=0).filter(keep=False,variable = "*LUCF*")
CO2 = CO2.convert_unit('Mt', to='亿吨', factor=0.01)
L=['净排放','电力','上游','工业','工业过程','农业', '建筑','交通']
CO2_order=['Emissions|CO2|TRA','Emissions|CO2|BLD','Emissions|CO2|AGR','Emissions|CO2|INDPROC','Emissions|CO2|IND','Emissions|CO2|UPS','Emissions|CO2|ELC']
L.reverse()
#CO2 emissions FFI-NDC
scen='ndc-ts'
Fig, ax = plt.subplots()
CO2.filter(scenario = scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,title=False,total = dict(lw=2),order=CO2_order,legend=False)
# handles,l= ax.get_legend_handles_labels()
# ax.legend(handles=handles,labels=L,loc=1,ncol=1,frameon=False)
plt.xlabel("时间")
plt.ylabel("化石燃料和工业过程二氧化碳排放（亿吨CO$_2$）")
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_CO2_FFI-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

#CO2 emissions FFI-CN60
Fig, ax = plt.subplots()
scen='ndc-ts_2c-cn60'
CO2.filter(scenario = scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(title=False,total = dict(lw=2),order=CO2_order,ax=ax)
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=handles,labels=L,loc=1,ncol=1,frameon=False)
plt.xlabel("时间")
plt.ylabel("化石燃料和工业过程二氧化碳排放（亿吨CO$_2$）")
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_CO2_FFI-CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# CO2_LULUCF = Data.filter(variable = "Emissions|CO2|*",level=0)
# CO2_LULUCF = CO2_LULUCF.convert_unit('Mt', to='亿吨', factor=0.01)
# L=[ '净排放','土地利用','电力','上游','工业','工业过程','农业', '建筑','交通']
# L.reverse()
# CO2_LULUCF_order=['Emissions|CO2|TRA','Emissions|CO2|BLD','Emissions|CO2|AGR','Emissions|CO2|INDPROC','Emissions|CO2|IND','Emissions|CO2|UPS','Emissions|CO2|ELC','Emissions|CO2|LUCF']
# # CO2 emissions FFI with LULUCF-NDC
# scen='ndc-ts'
# Fig, ax = plt.subplots()
# CO2_LULUCF.filter(scenario = scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,title=False,total = dict(lw=2),order=CO2_LULUCF_order,legend=False)
# # handles,l= ax.get_legend_handles_labels()
# # ax.legend(handles=handles,labels=L,loc=1,ncol=2,frameon=False)
# plt.xlabel("时间")
# plt.ylabel("净二氧化碳排放（亿吨CO$_2$）")
# plt.tight_layout()
# plt.savefig(ImgPath+"chap3_CO2_LULUCF-NDC"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

# # CO2 emissions FFI with LULUCF-CN60
# scen='ndc-ts_2c-cn60'
# Fig, ax = plt.subplots()
# CO2_LULUCF.filter(scenario = scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,title=False,total = dict(lw=2),order=CO2_LULUCF_order)
# handles,l= ax.get_legend_handles_labels()
# ax.legend(handles=handles,labels=L,loc=1,ncol=1,frameon=False)

# plt.xlabel("时间")
# plt.ylabel("净二氧化碳排放（亿吨CO$_2$）")
# plt.tight_layout()
# plt.savefig(ImgPath+"chap3_CO2_LULUCF-CN60"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:
CH4=Data.filter(variable = "Emissions|CH4|*").filter(keep=False,variable = "Emissions|CH4|LUCF|AGR").filter(keep=False,variable = "Emissions|CH4|LUCF")
CH4 = CH4.convert_unit('kt', to='万吨', factor=0.1)
L=['上游','电力','工业','农业', '建筑','交通','肠道发酵','土地肥料','水稻种植','草地', '废弃物']
L.reverse()
CH4_order=['Emissions|CH4|LUCF|Waste','Emissions|CH4|LUCF|Savannah','Emissions|CH4|LUCF|AGR|Rice','Emissions|CH4|LUCF|AGR|Manure','Emissions|CH4|LUCF|AGR|Enteric','Emissions|CH4|TRA','Emissions|CH4|BLD','Emissions|CH4|AGR','Emissions|CH4|IND','Emissions|CH4|ELC','Emissions|CH4|UPS']
#CH4 emissions-NDC
scen='ndc-ts'
Fig, ax = plt.subplots()
CH4.filter(scenario = scen).filter(keep=False,year=[2022,2028,2031,2032,2070]).plot.stack(ax=ax,title=False,order=CH4_order,legend=False)
# handles,l= ax.get_legend_handles_labels()
# ax.legend(handles=handles,labels=L,loc=1,ncol=2,frameon=False)
plt.xlabel("时间")
plt.ylabel("甲烷排放（万吨CH$_4$）")
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_CH4-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

#CH4 emissions-CN60
scen='ndc-ts_2c-cn60'
Fig, ax = plt.subplots()
CH4.filter(scenario = scen).filter(keep=False,year=[2022,2028,2031,2032,2050]).plot.stack(ax=ax,title=False,order=CH4_order)
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=handles,labels=L,loc=1,ncol=2,frameon=False)
plt.xlabel("时间")
plt.ylabel("甲烷排放（万吨CH$_4$）")
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_CH4_CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
N2O=Data.filter(variable = "Emissions|N2O|*").filter(keep=False,variable = "Emissions|N2O|LUCF|AGR").filter(keep=False,variable = "Emissions|N2O|LUCF")
N2O = N2O.convert_unit('kt', to='万吨', factor=0.1)
L=['上游','电力','工业','农业', '建筑','交通','肠道发酵','土地肥料','水稻种植','草地', '废弃物']
L.reverse()
N2O_order=['Emissions|N2O|LUCF|Waste','Emissions|N2O|LUCF|Savannah','Emissions|N2O|LUCF|AGR|Rice','Emissions|N2O|LUCF|AGR|Manure','Emissions|N2O|LUCF|AGR|Enteric','Emissions|N2O|TRA','Emissions|N2O|BLD','Emissions|N2O|AGR','Emissions|N2O|IND','Emissions|N2O|ELC','Emissions|N2O|UPS']
#N2O emissions-NDC
scen='ndc-ts'
Fig, ax = plt.subplots()
N2O.filter(scenario = scen).filter(keep=False,year=[2022,2031,2032,2050]).plot.stack(ax=ax,title=False,order=N2O_order,legend=False)
# handles,l= ax.get_legend_handles_labels()
# ax.legend(handles=handles,labels=L,loc=1,ncol=2,frameon=False)
plt.xlabel("时间")
plt.ylabel("氧化亚氮排放（万吨N$_2$O）")
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_N2O-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

#N2O emissions-CN60
scen='ndc-ts_2c-cn60'
Fig, ax = plt.subplots()
N2O.filter(scenario = scen).filter(keep=False,year=[2022,2031,2032,2050]).plot.stack(ax=ax,title=False,order=N2O_order)
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=handles,labels=L,loc=1,ncol=2,frameon=False,columnspacing=0.5)
plt.xlabel("时间")
plt.ylabel("氧化亚氮排放（万吨N$_2$O）")
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_N2O_CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
GHG=Data.filter(variable = "Emissions|GHG|*")
GHG = GHG.convert_unit('kt', to='亿吨', factor=0.00001)
L=[ '净排放','上游','电力','工业','农业', '建筑','交通','土地利用']
L.reverse()
GHG_order=['Emissions|GHG|LUCF','Emissions|GHG|TRA','Emissions|GHG|BLD','Emissions|GHG|AGR','Emissions|GHG|IND','Emissions|GHG|ELC','Emissions|GHG|UPS']
#GHG emissions-NDC
scen='ndc-ts'
Fig, ax = plt.subplots()
GHG.filter(scenario = scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,title=False,order=GHG_order,legend=False,total = dict(lw=2))
# handles,l= ax.get_legend_handles_labels()
# ax.legend(handles=handles,labels=L,loc=1,frameon=False)
plt.xlabel("时间")
plt.ylabel("温室气体排放（亿吨CO$_2$eq）")
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_GHG-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

#GHG emissions-CN60
scen='ndc-ts_2c-cn60'
Fig, ax = plt.subplots()
GHG.filter(scenario = scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,title=False,order=GHG_order,total = dict(lw=2))
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=handles,labels=L,loc=1,frameon=False)
plt.xlabel("时间")
plt.ylabel("温室气体排放（亿吨CO$_2$eq）")
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_GHG_CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
CCS=Data.filter(variable="Emissions|CCS*").filter(keep=False,year=[2031,2032]).convert_unit('Mt','亿吨',0.01)
# CCS
bar_order=['Emissions|CCS|CO2|Upstream|DAC',
           'Emissions|CCS|CO2|Upstream|Biomass',
           'Emissions|CCS|CO2|Electricity|Biomass',          
           'Emissions|CCS|CO2|Upstream|Fossil',
           'Emissions|CCS|CO2|Electricity|Coal',
           'Emissions|CCS|CO2|Electricity|Oil',          
           'Emissions|CCS|CO2|Electricity|Gas',
           'Emissions|CCS|CO2|Upstream|Heat',           
           'Emissions|CCS|CO2|Industry|Cement',
           'Emissions|CCS|CO2|Industry|IIS',
           'Emissions|CCS|CO2|Industry|NH3']
L=['上游-DACS （碳移除技术）','上游-BECCS（碳移除技术）',"电力-BECCS（碳移除技术）",'上游-CCS','电力-煤CCS','电力-油CCS','电力-气CCS','热力-CCS','水泥-CCS','钢铁-CCS','制氨-CCS']
scen=['ndc-ts_2c-cn60']
S_name=['CN60']


Fig, ax = plt.subplots(figsize=(10,5))
CCS.filter(scenario=scen,year=range(2030,2101)).plot.stack(ax=ax,x='year',title=False,
        legend=False,order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(rotation=0)
plt.xlabel("时间")
plt.ylabel("二氧化碳捕集量（亿吨CO$_2$）")    
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=handles,labels=L,ncol=1,frameon=False,loc=2,labelspacing=0.1)
# plt.legend(L,ncol=1,loc=2)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_CO2_CCS"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
PE=Data.filter(variable="Primary Energy|*")
PE=PE.convert_unit("PJ","亿吨标煤",1/0.293076/10000)
PE_order=['Primary Energy|Solar','Primary Energy|Wind','Primary Energy|Ocean','Primary Energy|Hydro','Primary Energy|Biomass|w/ CCS','Primary Energy|Biomass|w/o CCS','Primary Energy|Geothermal','Primary Energy|Nuclear','Primary Energy|Natural gas|w/ CCS','Primary Energy|Natural gas|w/o CCS','Primary Energy|Oil|w/ CCS','Primary Energy|Oil|w/o CCS','Primary Energy|Coal|w/ CCS','Primary Energy|Coal|w/o CCS']
L=['煤','煤CCS','油','油CCS','气','气CCS','核能','地热能','生物质','生物质CCS','水能','海洋能','风能','太阳能']
secondaryy="Renewable Rate"
# Primary Energy-NDC
scen='ndc-ts'
fig, ax = plt.subplots()
PE.filter(scenario=scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,title=False,order=PE_order,cmap=cbm.pal("npg").as_cmap)
ax.set_xlabel("时间")
ax.set_ylabel('一次能源供应（亿吨标煤）')
ax.legend(L,ncol=5,bbox_to_anchor=(1.1,-0.15),frameon=False,fontsize=12,columnspacing=1)
ax2=ax.twinx()
format_args = dict(color="black", linestyle="-")
Data.filter(scenario=scen,variable=secondaryy).plot(
ax=ax2, legend=True, title=None,**format_args)
ax2.set_ylim(0, 1)
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(1,0))
ax2.set_ylabel("可再生能源比例（%）")
ax2.legend(['可再生能源比例'],frameon=False)

# plt.savefig(ImgPath+"chap3_PE-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# Primary Energy-CN60
scen='ndc-ts_2c-cn60'
fig, ax = plt.subplots()
PE.filter(scenario=scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,title=False,order=PE_order,cmap=cbm.pal("npg").as_cmap)
ax.set_xlabel("时间")
ax.set_ylabel('一次能源供应（亿吨标煤）')
ax.legend(L,ncol=5,bbox_to_anchor=(1.1,-0.15),frameon=False,fontsize=12,columnspacing=1)
ax2=ax.twinx()
format_args = dict(color="black", linestyle="-")
Data.filter(scenario=scen,variable=secondaryy).plot(
ax=ax2, legend=True, title=None,**format_args)

ax2.set_ylim(0, 1)
ax2.set_ylabel("可再生能源比例（%）")
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(1,0))
ax2.legend(['可再生能源比例'],frameon=False)

# plt.savefig(ImgPath+"chap3_PE-CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# PE_equivalent=Data.filter(variable="PE(equivalent)|*")
# PE_equivalent=PE_equivalent.convert_unit("PJ","亿吨标煤",1/0.293076/10000)
# PE_equivalent_order=['PE(equivalent)|Solar','PE(equivalent)|Wind','PE(equivalent)|Ocean','PE(equivalent)|Hydro','PE(equivalent)|Biomass|w/ CCS','PE(equivalent)|Biomass|w/o CCS','PE(equivalent)|Geothermal','PE(equivalent)|Nuclear','PE(equivalent)|Natural gas|w/ CCS','PE(equivalent)|Natural gas|w/o CCS','PE(equivalent)|Oil|w/ CCS','PE(equivalent)|Oil|w/o CCS','PE(equivalent)|Coal|w/ CCS','PE(equivalent)|Coal|w/o CCS']
# L=['煤','煤CCS','油','油CCS','气','气CCS','核能','地热能','生物质','生物质CCS','水能','海洋能','风能','太阳能']
# secondaryy="Renewable Rate(equivalent)"
# # Primary Energy-NDC
# scen='ndc-ts'
# fig, ax = plt.subplots()
# PE_equivalent.filter(scenario=scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,title=False,order=PE_equivalent_order,cmap=cbm.pal("npg").as_cmap)
# ax.set_xlabel("时间")
# ax.set_ylabel('一次能源供应（亿吨标煤）')
# ax.legend(L,ncol=5,bbox_to_anchor=(1.1,-0.15),frameon=False,fontsize=12,columnspacing=1)
# ax2=ax.twinx()
# format_args = dict(color="black", linestyle="-")
# Data.filter(scenario=scen,variable=secondaryy).plot(
# ax=ax2, legend=False, title=None,**format_args)
# ax2.set_ylim(0, 1)
# ax2.set_ylabel("可再生能源比例（%）")
# ax2.yaxis.set_major_formatter(ticker.PercentFormatter(1,0))
# ax2.legend(['可再生能源比例'],frameon=False,loc='upper left')

# plt.savefig(ImgPath+"chap3_PE_equivalent-NDC"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

# # Primary Energy-CN60
# scen='ndc-ts_2c-cn60'
# fig, ax = plt.subplots()
# PE_equivalent.filter(scenario=scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,title=False,order=PE_equivalent_order,cmap=cbm.pal("npg").as_cmap)
# ax.set_xlabel("时间")
# ax.set_ylabel('一次能源供应（亿吨标煤）')
# ax.legend(L,ncol=5,bbox_to_anchor=(1.1,-0.15),frameon=False,fontsize=12,columnspacing=1)
# ax2=ax.twinx()
# format_args = dict(color="black", linestyle="-")
# Data.filter(scenario=scen,variable=secondaryy).plot(
# ax=ax2, legend=False, title=None,**format_args)
# ax2.set_ylim(0, 1)
# ax2.set_ylabel("可再生能源比例（%）")
# ax2.yaxis.set_major_formatter(ticker.PercentFormatter(1,0))
# ax2.legend(['可再生能源比例'],frameon=False,loc='upper left')

# plt.savefig(ImgPath+"chap3_PE_equivalent-CN60"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:
# Energy independency -Oil
OIL=Data.filter(variable="Import|Crude Oil").append(Data.filter(variable="Primary Energy|Oil"))
OIL.subtract("Primary Energy|Oil","Import|Crude Oil",'EXT Oil',append=True)
OIL=OIL.convert_unit("PJ","亿吨标煤",1/0.293076/10000)

OIL_order=['Import|Crude Oil','EXT Oil']
L=["进口","国内"]
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2020,2035,2060,2100]
L.reverse()
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3.5))
Fig6=plt.figure(figsize = (5, 3.5))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)

ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

OIL.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=OIL_order,cmap=sns.color_palette("tab20",as_cmap=True));
plt.xticks(label="2020年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("原油供应量（亿吨标煤）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    OIL.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=OIL_order,cmap=sns.color_palette("tab20",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("原油供应量（亿吨标煤）")
handles,l= ax6[3].get_legend_handles_labels()
ax6[3].legend(handles=reversed(handles),labels=L,loc=1,frameon=False)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_OIL_Import"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# Energy independency -Natural Gas
NGA=Data.filter(variable="Import|Natural Gas").append(Data.filter(variable="Primary Energy|Natural gas"))
NGA.subtract("Primary Energy|Natural gas","Import|Natural Gas",'EXT NGA',append=True)
NGA=NGA.convert_unit("PJ","亿吨标煤",1/0.293076/10000)

NGA_order=['Import|Natural Gas','EXT NGA']
L=["进口","国内"]
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2020,2035,2060,2100]
L.reverse()
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3.5))
Fig6=plt.figure(figsize = (5, 3.5))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)

ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

NGA.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=NGA_order,cmap=sns.color_palette("tab20",as_cmap=True));
plt.xticks(label="2020年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("天然气供应量（亿吨标煤）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    NGA.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=NGA_order,cmap=sns.color_palette("tab20",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("天然气供应量（亿吨标煤）")
handles,l= ax6[3].get_legend_handles_labels()
ax6[3].legend(handles=reversed(handles),labels=L,loc=1,frameon=False)
plt.tight_layout()
plt.savefig(ImgPath+"chap3_NGA_Import"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Final energy -sector
FE_S=Data.filter(variable="Final Energy*",level=1).filter(keep=False,variable = "*Total*")
FE_S=FE_S.convert_unit("PJ","亿吨标煤",1/0.293076/10000)
FE_S_order=['Final Energy|Agriculture','Final Energy|Transport','Final Energy|Building','Final Energy|Industry']
L=['工业',"建筑","交通","农业"]

# Final energy -sector-NDC
scen='ndc-ts'
Fig, ax = plt.subplots()
FE_S.filter(scenario=scen).filter(keep=False,year=[2031,2032]).plot.stack(
        title=False,legend=True,order=FE_S_order,ax=ax)
ax.legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
plt.xlabel("时间")
plt.ylabel("分部门终端能源消费（亿吨标煤）")

# plt.savefig(ImgPath+"chap3_FE_SECTOR-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# Final energy -sector-CN60
scen='ndc-ts_2c-cn60'
Fig, ax = plt.subplots()
FE_S.filter(scenario=scen).filter(keep=False,year=[2031,2032,2055]).plot.stack(ax=ax,
        title=False,legend=True,order=FE_S_order)
ax.legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
plt.xlabel("时间")
plt.ylabel("分部门终端能源消费（亿吨标煤）")

# plt.savefig(ImgPath+"chap3_FE_SECTOR-CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
FE_F=Data.filter(variable="Final Energy*",level=1).filter(variable = "*Total*")
FE_F=FE_F.convert_unit("PJ","亿吨标煤",1/0.293076/10000)
FE_F_order=['Final Energy|Total Electricity','Final Energy|Total Hydrogen','Final Energy|Total Solar','Final Energy|Total Geothermal','Final Energy|Total Biomass','Final Energy|Total Heat','Final Energy|Total Gas','Final Energy|Total Liquids','Final Energy|Total Coal',]
L=['煤',"油品","燃气","热","生物质","地热","太阳能","氢能","电"]
secondaryy="Electrification Rate"

# Final energy -fuel-NDC
scen='ndc-ts'
fig, ax = plt.subplots()
FE_F.filter(scenario=scen).filter(keep=False,year=[2024,2025,2027,2028,2029,2031,2032]).plot.stack(ax=ax,title=False,order=FE_F_order)
ax.set_xlabel("时间")
ax.set_ylabel('分燃料终端能源消费（亿吨标煤）')
ax.legend(L,ncol=5,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
ax2=ax.twinx()
format_args = dict(color="black", linestyle="-")
Data.filter(scenario=scen,variable=secondaryy).plot(
ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylim(0, 1)
ax2.set_ylabel("电气化率（%）")
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(1,0))
ax2.legend(['电气化率'],frameon=False)

# plt.savefig(ImgPath+"chap3_FE_FUEL-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()
# Final energy -fuel-CN60
scen='ndc-ts_2c-cn60'
fig, ax = plt.subplots()
FE_F.filter(scenario=scen).filter(keep=False,year=[2024,2025,2027,2028,2029,2031,2032,2045]).plot.stack(ax=ax,title=False,order=FE_F_order)
ax.set_xlabel("时间")
ax.set_ylabel('分燃料终端能源消费（亿吨标煤）')
ax.legend(L,ncol=5,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
ax2=ax.twinx()
format_args = dict(color="black", linestyle="-")
Data.filter(scenario=scen,variable=secondaryy).plot(
ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylim(0, 1)
ax2.set_ylabel("电气化率（%）")
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(1,0))
ax2.legend(['电气化率'],frameon=False)

# plt.savefig(ImgPath+"chap3_FE_FUEL-CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
ELC_cap=Data.filter(variable="Capacity|Electricity|*",level="1-").filter(keep=False,variable=["*Gas","*Coal","*Oil","*Biomass","Biomass-Coal","*Once*","*Recir*","*Air*"])
L=['煤','煤CCS','油', '油CCS', '气','气CCS','核电','地热','生物质-煤','生物质-煤CCS','生物质','生物质CCS','水电','海洋能','风能','太阳能']
ELC_Cap_order=['Capacity|Electricity|Solar', 'Capacity|Electricity|Wind', 'Capacity|Electricity|Ocean','Capacity|Electricity|Hydro', 'Capacity|Electricity|Biomass|w/ CCS','Capacity|Electricity|Biomass|w/o CCS','Capacity|Electricity|Biomass-Coal|w/ CCS', 'Capacity|Electricity|Biomass-Coal|w/o CCS','Capacity|Electricity|Geothermal','Capacity|Electricity|Nuclear',  'Capacity|Electricity|Gas|w/ CCS', 'Capacity|Electricity|Gas|w/o CCS', 'Capacity|Electricity|Oil|w/ CCS','Capacity|Electricity|Oil|w/o CCS','Capacity|Electricity|Coal|w/ CCS', 'Capacity|Electricity|Coal|w/o CCS']
# Electricity cap-NDC
scen='ndc-ts'
ELC_cap.filter(scenario=scen).filter(keep=False,year=[2031,2032]).plot.stack(
        title=False,legend=True,order=ELC_Cap_order)
plt.legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
plt.xlabel("时间")
plt.ylabel("电力装机容量（GW）")
plt.ylim([0,12500])
# plt.savefig(ImgPath+"chap3_ELC_CAP-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# Electricity cap-CN60
scen='ndc-ts_2c-cn60'
ELC_cap.filter(scenario=scen).filter(keep=False,year=[2031,2032]).plot.stack(
        title=False,legend=True,order=ELC_Cap_order)
plt.legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
plt.xlabel("时间")
plt.ylabel("电力装机容量（GW）")
plt.ylim([0,12500])
# plt.savefig(ImgPath+"chap3_ELC_CAP-CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
ELC_gen=Data.filter(variable="Secondary Energy|Electricity|*",level="1-").filter(keep=False,variable=["*Gas","*Coal","*Oil","*Biomass","Biomass-Coal"])
ELC_gen=ELC_gen.convert_unit('TWh','PWh')
L=['煤','煤CCS','油', '油CCS', '气','气CCS','核电','地热','生物质-煤','生物质-煤CCS','生物质','生物质CCS','水电','海洋能','风能','太阳能']
ELC_gen_order=['Secondary Energy|Electricity|Solar', 'Secondary Energy|Electricity|Wind', 'Secondary Energy|Electricity|Ocean','Secondary Energy|Electricity|Hydro', 'Secondary Energy|Electricity|Biomass|w/ CCS','Secondary Energy|Electricity|Biomass|w/o CCS','Secondary Energy|Electricity|Biomass-Coal|w/ CCS', 'Secondary Energy|Electricity|Biomass-Coal|w/o CCS','Secondary Energy|Electricity|Geothermal','Secondary Energy|Electricity|Nuclear',  'Secondary Energy|Electricity|Gas|w/ CCS', 'Secondary Energy|Electricity|Gas|w/o CCS', 'Secondary Energy|Electricity|Oil|w/ CCS', 'Secondary Energy|Electricity|Oil|w/o CCS','Secondary Energy|Electricity|Coal|w/ CCS', 'Secondary Energy|Electricity|Coal|w/o CCS']

# Electricity Generation-NDC
scen='ndc-ts'
ELC_gen.filter(scenario=scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(
        title=False,legend=True,order=ELC_gen_order)
plt.legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
plt.xlabel("时间")
plt.ylabel("发电量（PWh）")
plt.ylim([0,21])
# plt.savefig(ImgPath+"chap3_ELC_Gen-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# Electricity Generation-CN60
scen='ndc-ts_2c-cn60'
ELC_gen.filter(scenario=scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(
        title=False,legend=True,order=ELC_gen_order)
plt.legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
plt.xlabel("时间")
plt.ylabel("发电量（PWh）")
plt.ylim([0,21])
# plt.savefig(ImgPath+"chap3_ELC_Gen-CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
H2N_GEN=Data.filter(variable="Secondary Energy|Hydrogen|*").filter(keep=False,variable=["*Biomass",'*Gas','*Oil','*Coal'])
L=['网电','陆上风电','海上风电','光伏','生物质','生物质CCS']
H2N_GEN_order=['Secondary Energy|Hydrogen|Biomass|w/ CCS',
               'Secondary Energy|Hydrogen|Biomass|w/o CCS',
               'Secondary Energy|Hydrogen|Distribute|Solar',
               'Secondary Energy|Hydrogen|Distribute|Onshore Wind',
               'Secondary Energy|Hydrogen|Distribute|Offshore Wind',
               'Secondary Energy|Hydrogen|Electricity']
H2N_GEN=H2N_GEN.convert_unit('PJ','万吨氢气',0.83333333)
L.reverse()
# Hydrogen generation-NDC
scen='ndc-ts'
Fig, ax = plt.subplots()
H2N_GEN.filter(scenario=scen).filter(keep=False,year=[2031,2032]).plot.stack(ax=ax,
        title=False,legend=False,order=H2N_GEN_order)
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=handles,labels=L,loc=2,frameon=False)
plt.xlabel("时间")
plt.ylabel("氢能生产量（万吨H$_2$）")
plt.ylim([0,6000])
# plt.savefig(ImgPath+"chap3_Hydrogen-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# Hydrogen generation-CN60
scen='ndc-ts_2c-cn60'
Fig, ax = plt.subplots()
H2N_GEN.filter(scenario=scen).filter(keep=False,year=[2031,2032]).plot.stack(ax=ax,
        title=False,legend=True,order=H2N_GEN_order)
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=handles,labels=L,loc=2,frameon=False)
plt.xlabel("时间")
plt.ylabel("氢能生产量（万吨H$_2$）")
plt.ylim([0,6000])
# plt.savefig(ImgPath+"chap3_Hydrogen-CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Industry production
IND_P=Data.filter(variable='Energy Service|IND*',scenario='ndc-ts')
L=['钢铁','水泥','平板玻璃','造纸',
   '铝','铜','锌','铅',
   '乙烯','纯碱','烧碱','合成氨'
  ]
IND_order=[
    'Energy Service|IND|IIS','Energy Service|IND|IBU|Cement','Energy Service|IND|IBU|Glass','Energy Service|IND|ILP', 
    'Energy Service|IND|INF|Al','Energy Service|IND|INF|Cu','Energy Service|IND|INF|Zn','Energy Service|IND|INF|Pb',
    'Energy Service|IND|ICH|C2H4','Energy Service|IND|ICH|Na2CO3','Energy Service|IND|ICH|NaOH','Energy Service|IND|ICH|NH3'
    ]


FigIND, axIND = plt.subplots(3, 4, figsize = (15, 10),sharex=False)
for j in range (0,3):
    for i in range (0,4):
        FigIND=IND_P.filter(variable=IND_order[4*j+i]).plot(x="year",y='value',title=False,ax = axIND[j][i],legend=False,linewidth=3)
        axIND[j][i].set_xlabel("时间",size=16)
        axIND[j][i].set_ylabel("")
        if (j==0)&(i==2):
            axIND[j][i].set_title(L[4*j+i]+"（百万箱）",size=18)
        
        else:
            axIND[j][i].set_title(L[4*j+i]+"（百万吨）",size=18)
plt.subplots_adjust(wspace=0, hspace=0)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_IND_Production"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
# # Final energy -IND
# FE_IND_S=Data.filter(variable="Final Energy|Industry*",level=1).filter(keep=False,variable = "*Total*").filter(keep=False,year=2022)
# FE_IND_S=FE_IND_S.convert_unit("PJ","亿吨标煤",1/0.293076/10000)
# FE_IND_S_order=['Final Energy|Industry|Electricity','Final Energy|Industry|Hydrogen','Final Energy|Industry|Heat','Final Energy|Industry|Gas','Final Energy|Industry|Liquids','Final Energy|Industry|Coal',]
# L=['煤',"油品","燃气","热","氢能","电"]
# FE_GDP=Data.divide("Final Energy|Industry","GDP|Secondary",'Energy Intensity of Industry')
# FE_GDP=FE_GDP.filter(scenario=['ndc-ts','ndc-ts_2c-cn60']).convert_unit('PJ / USD / billion','吨标煤/万元',1/0.293076/(10/6.9))
# # Final energy -IND-NDC
# scen='ndc-ts'
# Fig, ax = plt.subplots()
# FE_IND_S.filter(scenario=scen).filter(keep=False,year=[2024,2025,2031,2032]).plot.stack(ax=ax,
#         title=False,legend=False,order=FE_IND_S_order)
# handles,l= ax.get_legend_handles_labels()
# ax.legend(handles=reversed(handles),labels=L,ncol=6,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
# plt.xlabel("时间")
# plt.ylabel("工业部门终端能源消费（亿吨标煤）")
# ax2=ax.twinx()
# format_args = dict(color="black", linestyle="-")
# FE_GDP.filter(scenario=['ndc-ts']).filter(keep=False,year=[2022,2023,2031,2032]).plot(ax=ax2, legend=False, title=None,**format_args)
# ax2.set_ylabel("工业单位增加值能耗（吨标煤/万元）")
# ax2.legend(['工业单位增加值能耗'],frameon=False)
# ax2.set_ylim([0,30])
# plt.savefig(ImgPath+"chap3_FE_IND-NDC"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

# # Final energy -IND-CN60
# scen='ndc-ts_2c-cn60'
# Fig, ax = plt.subplots()
# FE_IND_S.filter(scenario=scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,
#         title=False,legend=True,order=FE_IND_S_order)
# handles,l= ax.get_legend_handles_labels()
# ax.legend(handles=reversed(handles),labels=L,ncol=6,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)

# plt.xlabel("时间")
# plt.ylabel("工业部门终端能源消费（亿吨标煤）")
# ax2=ax.twinx()
# format_args = dict(color="black", linestyle="-")
# FE_GDP.filter(scenario=['ndc-ts_2c-cn60']).filter(keep=False,year=[2022,2031,2032]).plot(ax=ax2, legend=False, title=None,**format_args)
# ax2.set_ylabel("工业单位增加值能耗（吨标煤/万元）")
# ax2.legend(['工业单位增加值能耗'],frameon=False)
# ax2.set_ylim([0,30])

# plt.savefig(ImgPath+"chap3_FE_IND-CN60"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:
# CO2IND = Data.filter(variable = "Emissions|CO2|IND|*",level=0)
# CO2IND = CO2IND.convert_unit('Mt', to='亿吨', factor=0.01)
# L=['钢铁','建材','化工','造纸','有色','其他']
# CO2IND_order=[ 
#     'Emissions|CO2|IND|IIS',
#     'Emissions|CO2|IND|IBU',
#     'Emissions|CO2|IND|ICH',
#     'Emissions|CO2|IND|ILP',
#     'Emissions|CO2|IND|INF',
#     'Emissions|CO2|IND|IOI']
# CO2IND_order.reverse()
# L.reverse()
# #CO2 emissions IND-NDC
# scen='ndc-ts'
# Fig, ax = plt.subplots()
# CO2IND.filter(scenario = scen).filter(keep=False,year=[2022,2026,2031,2032]).plot.stack(ax=ax,title=False,total = dict(lw=2),order=CO2IND_order,legend=False)
# # handles,l= ax.get_legend_handles_labels()
# # ax.legend(handles=handles,labels=L,loc=1,ncol=1,frameon=False)
# plt.xlabel("时间")
# plt.ylabel("工业部门二氧化碳排放（亿吨CO$_2$）")
# plt.tight_layout()
# plt.savefig(ImgPath+"chap3_CO2_IND-NDC"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

# #CO2 emissions IND-CN60
# Fig, ax = plt.subplots()
# scen='ndc-ts_2c-cn60'
# CO2IND.filter(scenario = scen).filter(keep=False,year=[2022,2026,2031,2032,2045]).plot.stack(title=False,total = dict(lw=2),order=CO2IND_order,ax=ax)
# handles,l= ax.get_legend_handles_labels()
# ax.legend(handles=handles,labels=L,loc=1,ncol=1,frameon=False)
# plt.xlabel("时间")
# plt.ylabel("工业部门二氧化碳排放（亿吨CO$_2$）")
# plt.tight_layout()
# plt.savefig(ImgPath+"chap3_CO2_IND-CN60"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:

CO2_IND=Data.filter(variable=["Emissions|CO2|IND|*","Emissions|CO2|INDPROC|IBU","Emissions|CO2|INDPROC|IIS"],scenario=["ndc-ts","ndc-ts_2c-cn60"]).filter(variable=["*IOI","*ILP","*INF","*ICH"],keep=False).filter(keep=False,year=[2022,2023,2031,2032])
CO2_IND.add('Emissions|CO2|IND|IBU','Emissions|CO2|INDPROC|IBU','建材',append=True)
CO2_IND.add('Emissions|CO2|IND|IIS','Emissions|CO2|INDPROC|IIS','钢铁',append=True)
CO2_IND=CO2_IND.filter(keep=False, variable=['Emissions|CO2|IND|IBU','Emissions|CO2|INDPROC|IBU','Emissions|CO2|IND|IIS','Emissions|CO2|INDPROC|IIS']).convert_unit("Mt","亿吨",0.01).rename(scenario={"ndc-ts":"REF","ndc-ts_2c-cn60":"CN60"}).as_pandas()
g=sns.catplot(CO2_IND,x='year',y='value',col="variable", hue="scenario",col_wrap=3,native_scale=True,sharey=False,kind="point",marker="",legend_out=False,legend='full',palette='Set1',col_order=['钢铁','建材'])
g.set_xlabels("")
g.set_ylabels("CO$_2$排放量（亿吨CO$_2$）",size=16)
g.set_titles(col_template="{col_name}",size=18)
g.add_legend(frameon=False)
g.tight_layout()

# plt.savefig(ImgPath+"chap3_IND_CO2"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
FE_IND_S=Data.filter(scenario='ndc-ts_2c-cn60',variable="Final Energy|Industry|I*")
FE_IND_S=FE_IND_S.convert_unit("PJ","亿吨标煤",1/0.293076/10000)


Fig6, ax6 = plt.subplots(3, 4, figsize = (16, 10), sharey = False)
#钢铁
xorder=['Final Energy|Industry|IIS|Coal','Final Energy|Industry|IIS|Coal CCS','Final Energy|Industry|IIS|Gas','Final Energy|Industry|IIS|Heat','Final Energy|Industry|IIS|Electricity','Final Energy|Industry|IIS|Hydrogen']
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|IIS|*").filter(keep=False,year=[2031,2032]).plot.stack(title="钢铁",ax=ax6[0][0],legend=False,order=xorder)
ax6[0][0].set_xlabel("")
ax6[0][0].set_ylabel("总能耗（亿吨标煤）")
P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|IIS")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|IIS'),inplace=True)
P1=P1.divide("Final Energy|Industry|IIS",'Energy Service|IND|IIS','单耗',ignore_units=True).filter(keep=False,year=[2031,2032])
ax2=ax6[0][0].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/吨）")
ax2.legend(['单耗'],frameon=False,loc=1)
ax2.set_ylim([0,500])
#水泥
xorder=['Final Energy|Industry|IBU(Cement)|Coal','Final Energy|Industry|IBU(Cement)|Coal CCS','Final Energy|Industry|IBU(Cement)|Electricity',]
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|IBU(Cement)|*").filter(keep=False,year=[2023,2031,2032,2050,2055]).plot.stack(title="水泥",ax=ax6[0][1],legend=False,order=xorder)
ax6[0][1].set_xlabel("")
ax6[0][1].set_ylabel("总能耗（亿吨标煤）")

P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|IBU(Cement)")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|IBU|Cement'),inplace=True)
P1=P1.divide("Final Energy|Industry|IBU(Cement)",'Energy Service|IND|IBU|Cement','单耗',ignore_units=True).filter(year=[2031,2032],keep=False)
ax2=ax6[0][1].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/吨）")
ax2.legend(['单耗'],frameon=False)
ax2.set_ylim([0,100])


#玻璃
xorder=['Final Energy|Industry|IBU(Glass)|Coal', 'Final Energy|Industry|IBU(Glass)|Electricity','Final Energy|Industry|IBU(Glass)|Liquids']
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|IBU(Glass)|*").filter(keep=False,year=[2021,2022,2031,2032,2040]).convert_unit("亿吨标煤","Mtce",100).plot.stack(title="平板玻璃",ax=ax6[0][2],legend=False,order=xorder)
ax6[0][2].set_xlabel("")
ax6[0][2].set_ylabel("总能耗（百万吨标煤）")
P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|IBU(Glass)")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|IBU|Glass'),inplace=True)
P1=P1.divide("Final Energy|Industry|IBU(Glass)",'Energy Service|IND|IBU|Glass','单耗',ignore_units=True).filter(keep=False,year=[2031,2032])
ax2=ax6[0][2].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/标准箱）")
ax2.legend(['单耗'],frameon=False)
ax2.set_ylim([0,12])

#造纸
xorder=['Final Energy|Industry|ILP|Coal','Final Energy|Industry|ILP|Gas','Final Energy|Industry|ILP|Heat','Final Energy|Industry|ILP|Electricity']
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|ILP|*").filter(keep=False,year=[2020,2023,2031,2032]).convert_unit("亿吨标煤","Mtce",100).plot.stack(title="造纸",ax=ax6[0][3],legend=False,order=xorder)
ax6[0][3].set_xlabel("")
ax6[0][3].set_ylabel("总能耗（百万吨标煤）")
P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|ILP")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|ILP'),inplace=True)
P1=P1.divide("Final Energy|Industry|ILP",'Energy Service|IND|ILP','单耗',ignore_units=True).filter(keep=False,year=[2020,2021,2031,2032])
ax2=ax6[0][3].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/吨）")
ax2.legend(['单耗'],frameon=False)
ax2.set_ylim([0,250])

#铝
xorder=['Final Energy|Industry|INF(Al)|Coal','Final Energy|Industry|INF(Al)|Liquids','Final Energy|Industry|INF(Al)|Gas','Final Energy|Industry|INF(Al)|Heat','Final Energy|Industry|INF(Al)|Electricity']
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|INF(Al)|*").filter(keep=False,year=[2031,2032]).convert_unit("亿吨标煤","Mtce",100).plot.stack(title="一次铝",ax=ax6[1][0],legend=False,order=xorder)
ax6[1][0].set_xlabel("")
ax6[1][0].set_ylabel("总能耗（百万吨标煤）")
P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|INF(Al)")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|INF|Al'),inplace=True)
P1=P1.divide("Final Energy|Industry|INF(Al)",'Energy Service|IND|INF|Al','单耗',ignore_units=True).filter(keep=False,year=[2031,2032])
ax2=ax6[1][0].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/吨）")
ax2.legend(['单耗'],frameon=False,loc=1)
ax2.set_ylim([0,2500])

#铜
xorder=['Final Energy|Industry|INF(Cu)|Coal','Final Energy|Industry|INF(Cu)|Liquids','Final Energy|Industry|INF(Cu)|Gas','Final Energy|Industry|INF(Cu)|Heat','Final Energy|Industry|INF(Cu)|Electricity']
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|INF(Cu)|*").filter(keep=False,year=[2031,2032]).convert_unit("亿吨标煤","Mtce",100).plot.stack(title="精炼铜",ax=ax6[1][1],legend=False,order=xorder)
ax6[1][1].set_xlabel("")
ax6[1][1].set_ylabel("总能耗（百万吨标煤）")
P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|INF(Cu)")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|INF|Cu'),inplace=True)
P1=P1.divide("Final Energy|Industry|INF(Cu)",'Energy Service|IND|INF|Cu','单耗',ignore_units=True).filter(keep=False,year=[2031,2032])
ax2=ax6[1][1].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/吨）")
ax2.legend(['单耗'],frameon=False,loc=1)
ax2.set_ylim([0,500])
#锌
xorder=[ 'Final Energy|Industry|INF(Zn)|Coal','Final Energy|Industry|INF(Zn)|Gas','Final Energy|Industry|INF(Zn)|Electricity']
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|INF(Zn)|*").filter(keep=False,year=[2031,2032]).convert_unit("亿吨标煤","Mtce",100).plot.stack(title="锌",ax=ax6[1][2],legend=False,order=xorder)
ax6[1][2].set_xlabel("")
ax6[1][2].set_ylabel("总能耗（百万吨标煤）")
P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|INF(Zn)")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|INF|Zn'),inplace=True)
P1=P1.divide("Final Energy|Industry|INF(Zn)",'Energy Service|IND|INF|Zn','单耗',ignore_units=True).filter(keep=False,year=[2031,2032])
ax2=ax6[1][2].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/吨）")
ax2.legend(['单耗'],frameon=False,loc=1)
ax2.set_ylim([0,1500])
#铅
xorder=['Final Energy|Industry|INF(Pb)|Coal','Final Energy|Industry|INF(Pb)|Gas','Final Energy|Industry|INF(Pb)|Electricity']
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|INF(Pb)|*").filter(keep=False,year=[2031,2032]).convert_unit("亿吨标煤","Mtce",100).plot.stack(title="铅",ax=ax6[1][3],legend=False,order=xorder)
ax6[1][3].set_xlabel("")
ax6[1][3].set_ylabel("总能耗（百万吨标煤）")
P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|INF(Pb)")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|INF|Pb'),inplace=True)
P1=P1.divide("Final Energy|Industry|INF(Pb)",'Energy Service|IND|INF|Pb','单耗',ignore_units=True).filter(keep=False,year=[2031,2032])
ax2=ax6[1][3].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/吨）")
ax2.legend(['单耗'],frameon=False,loc=1)
ax2.set_ylim([0,600])

#乙烯
xorder=['Final Energy|Industry|ICH(C2H4)|Liquids']
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|ICH(C2H4)|*").filter(keep=False,year=[2031,2032]).convert_unit("亿吨标煤","Mtce",100).plot.stack(title="乙烯",ax=ax6[2][0],legend=False,order=xorder)
ax6[2][0].set_xlabel("")
ax6[2][0].set_ylabel("总能耗（百万吨标煤）")

P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|ICH(C2H4)")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|ICH|C2H4'),inplace=True)
P1=P1.divide("Final Energy|Industry|ICH(C2H4)",'Energy Service|IND|ICH|C2H4','单耗',ignore_units=True).filter(year=[2031,2032],keep=False)
ax2=ax6[2][0].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/吨）")
ax2.legend(['单耗'],frameon=False,loc=1)
ax2.set_ylim([0,650])


#纯碱
xorder=['Final Energy|Industry|ICH(Na2CO3)|Heat','Final Energy|Industry|ICH(Na2CO3)|Electricity']
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|ICH(Na2CO3)|*").filter(keep=False,year=[2031,2032]).convert_unit("亿吨标煤","Mtce",100).plot.stack(title="纯碱",ax=ax6[2][1],legend=False,order=xorder)
ax6[2][1].set_xlabel("")
ax6[2][1].set_ylabel("总能耗（百万吨标煤）")

P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|ICH(Na2CO3)")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|ICH|Na2CO3'),inplace=True)
P1=P1.divide("Final Energy|Industry|ICH(Na2CO3)",'Energy Service|IND|ICH|Na2CO3','单耗',ignore_units=True).filter(keep=False,year=[2031,2032])
ax2=ax6[2][1].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/吨）")
ax2.legend(['单耗'],frameon=False,loc=1)
ax2.set_ylim([0,250])

#烧碱
xorder=['Final Energy|Industry|ICH(NaOH)|Heat','Final Energy|Industry|ICH(NaOH)|Electricity']
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|ICH(NaOH)|*").filter(keep=False,year=[2031,2032]).convert_unit("亿吨标煤","Mtce",100).plot.stack(title="烧碱",ax=ax6[2][2],legend=False,order=xorder)
ax6[2][2].set_xlabel("")
ax6[2][2].set_ylabel("总能耗（百万吨标煤）")
P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|ICH(NaOH)")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|ICH|NaOH'),inplace=True)
P1=P1.divide("Final Energy|Industry|ICH(NaOH)",'Energy Service|IND|ICH|NaOH','单耗',ignore_units=True).filter(keep=False,year=[2031,2032])
ax2=ax6[2][2].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/吨）")
ax2.legend(['单耗'],frameon=False,loc=1)
ax2.set_ylim([0,450])

#合成氨
xorder=['Final Energy|Industry|ICH(NH3)|Coal','Final Energy|Industry|ICH(NH3)|Coal CCS','Final Energy|Industry|ICH(NH3)|Liquids',
 'Final Energy|Industry|ICH(NH3)|Liquids CCS','Final Energy|Industry|ICH(NH3)|Gas','Final Energy|Industry|ICH(NH3)|Gas CCS','Final Energy|Industry|ICH(NH3)|Heat','Final Energy|Industry|ICH(NH3)|Electricity','Final Energy|Industry|ICH(NH3)|Hydrogen']
L=['煤',"煤CCS",'油品','油品CCS','燃气','燃气CCS','热','电能','氢能']
xorder.reverse()
FE_IND_S.filter(variable="Final Energy|Industry|ICH(NH3)|*").filter(keep=False,year=[2022,2023,2031,2032]).convert_unit("亿吨标煤","Mtce",100).plot.stack(title="合成氨",ax=ax6[2][3],legend=False,order=xorder)
ax6[2][3].set_xlabel("")
ax6[2][3].set_ylabel("总能耗（百万吨标煤）")

P1=Data.filter(scenario='ndc-ts_2c-cn60').aggregate(variable="Final Energy|Industry|ICH(NH3)")
P1=P1.convert_unit("PJ","千吨标煤",1/0.293076*10)
P1.append(Data.filter(scenario='ndc-ts_2c-cn60',variable='Energy Service|IND|ICH|NH3'),inplace=True)
P1=P1.divide("Final Energy|Industry|ICH(NH3)",'Energy Service|IND|ICH|NH3','单耗',ignore_units=True).filter(keep=False,year=[2022,2031,2032,2060])
ax2=ax6[2][3].twinx()
format_args = dict(color="black", linestyle="-")
P1.plot(ax=ax2, legend=False, title=None,**format_args)
ax2.set_ylabel("单耗（千克标煤/吨）")
ax2.legend(['单耗'],frameon=False)
ax2.set_ylim([0,1400])

for i in range (0,3):
    for j in range (0,4):
        ax6[i][j].xaxis.set_major_locator(MultipleLocator(20))
    

plt.subplots_adjust(wspace=0, hspace=0)
plt.tight_layout()
handles,l= ax6[2][3].get_legend_handles_labels()
ax6[2][3].legend(handles=reversed(handles),labels=L,frameon=False,fontsize=12,ncol=9,loc='upper center',bbox_to_anchor=(-2,-0.1))
# plt.savefig(ImgPath+"chap3_IND_ENERGY"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
CEM=Data.filter(variable="IND|IBU|CEM|*")
CEM=CEM.convert_unit("Mt","亿吨",1/100)
# Cement
bar_order=['IND|IBU|CEM|CCS',
           'IND|IBU|CEM|Multichannel',
           'IND|IBU|CEM|Highratio',
           'IND|IBU|CEM|Large',
           'IND|IBU|CEM|Medium',
           'IND|IBU|CEM|Small']
bar_order.reverse()
L=["小型",'中型','大型',"高固气比",'多通道','CCS']
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2020,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

CEM.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2020年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("水泥产量（亿吨）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    CEM.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("水泥产量（亿吨）")
ax6[2].legend(L,ncol=3,loc='upper center',bbox_to_anchor=(0.2,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_IND_CEM"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
NH3=Data.filter(variable="IND|ICH|NH3|*")
# NH3
bar_order=['IND|ICH|NH3|CCS',
           'IND|ICH|NH3|Hydrogen',
           'IND|ICH|NH3|Natural gas',
           'IND|ICH|NH3|Oil',
           'IND|ICH|NH3|Coal Anthracite',
           'IND|ICH|NH3|Coal Pulverized']

L=["粉煤",'无烟煤','石油','天然气','氢能','CCS']
bar_order.reverse()
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2020,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

NH3.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2_r",as_cmap=True));
plt.xticks(label="2020年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("合成氨产量（百万吨）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    NH3.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2_r",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("合成氨产量（百万吨）")
ax6[2].legend(L,ncol=3,loc='upper center',bbox_to_anchor=(0.2,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_IND_NH3"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
IRON=Data.filter(variable="IND|IIS|IRON|*")
IRON=IRON.convert_unit("Mt","亿吨",1/100)
# IRON
bar_order=['IND|IIS|IRON|Hydrogen',
           'IND|IIS|IRON|CCS',
           'IND|IIS|IRON|DTRT',
           'IND|IIS|IRON|WTRT',
           'IND|IIS|IRON|NTRT']

L=["无除尘",'湿式除尘','干式除尘',"CCS技术",'氢能炼铁']
L.reverse()
bar_order.reverse()
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2020,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

IRON.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2_r",as_cmap=True));
plt.xticks(label="2020年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("生铁产量（亿吨）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    IRON.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2_r",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("生铁产量（亿吨）")
handles,l= ax6[3].get_legend_handles_labels()
ax6[2].legend(handles=reversed(handles),labels=L,ncol=3,loc='upper center',bbox_to_anchor=(0.2,-0.1),frameon=False,fontsize=12,columnspacing=1)

# plt.savefig(ImgPath+"chap3_IND_IRON"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
STEEL=Data.filter(variable="IND|IIS|STEEL|*")
STEEL=STEEL.convert_unit("Mt","亿吨",1/100)
# STEEL
bar_order=['IND|IIS|STEEL|Electricity',
           'IND|IIS|STEEL|Oxygen']

L=["转炉炼钢",'电炉炼钢']

l.reverse()
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2020,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

STEEL.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2020年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("粗钢产量（亿吨）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    STEEL.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("粗钢产量（亿吨）")
handles,l= ax6[2].get_legend_handles_labels()
ax6[2].legend(handles=reversed(handles),labels=L,ncol=1,loc='upper center',bbox_to_anchor=(0.2,-0.1),frameon=False,fontsize=12,columnspacing=1)

# plt.savefig(ImgPath+"chap3_IND_STEEL"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Building Area 
BLD_A=Data.filter(variable="Building Area*")
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60',"ndc-ts_2c-cn60-noCF"]
S_name=["REF情景",'CN60情景',"CN60-noCF情景"]
Fig10, ax10 = plt.subplots(3,3, figsize = (15, 15), sharey = True)
for i in range (0,3):
    BLD_A.filter(scenario=scen_sdg[i],variable="*Commercial|*").filter(keep=False,year=[2031,2032]).plot.stack(
        legend=False,title=S_name[i]+" 公共建筑",ax = ax10[i][0],cmap=cbm.pal("npg").as_cmap,
        order=sorted(BLD_A.filter(scenario=scen_sdg[i],variable="*Commercial|*").variable,reverse=True))
    ax10[i][0].set_xlabel("时间")
    ax10[i][0].set_ylabel("房屋面积（亿平方米）")
    BLD_A.filter(scenario=scen_sdg[i],variable="*Rural|*").filter(keep=False,year=[2031,2032]).plot.stack(
        legend=False,title=S_name[i]+" 农村建筑",ax = ax10[i][1],cmap=cbm.pal("npg").as_cmap,
        order=sorted(BLD_A.filter(scenario=scen_sdg[i],variable="*Rural|*").variable,reverse=True))
    ax10[i][1].set_xlabel("时间")
    ax10[i][1].set_ylabel("房屋面积（亿平方米）")
    BLD_A.filter(scenario=scen_sdg[i],variable="*Urban|*").filter(keep=False,year=[2031,2032]).plot.stack(
        legend=False,title=S_name[i]+" 城镇建筑",ax = ax10[i][2],cmap=cbm.pal("npg").as_cmap,
        order=sorted(BLD_A.filter(scenario=scen_sdg[i],variable="*Urban|*").variable,reverse=True))
    ax10[i][2].set_xlabel("时间")
    ax10[i][2].set_ylabel("房屋面积（亿平方米）")
handles,l= ax10[0][0].get_legend_handles_labels()
L=["1990标准","2005标准","2015标准","2020标准","低能耗","近零能耗"]
L.reverse()
ax10[0][0].legend(handles,L,ncol=1,loc='best',frameon=False)
ax10[1][0].legend(handles,L,ncol=1,loc='best',frameon=False)
ax10[2][0].legend(handles,L,ncol=1,loc='best',frameon=False)

handles,l= ax10[0][1].get_legend_handles_labels()
L=["1980标准","1990标准","2000标准","2010标准","2020标准","低能耗","近零能耗"]
L.reverse()
ax10[0][1].legend(handles,L,ncol=1,loc='best',frameon=False)
ax10[1][1].legend(handles,L,ncol=1,loc='best',frameon=False)
ax10[2][1].legend(handles,L,ncol=1,loc='best',frameon=False)
handles,l= ax10[0][2].get_legend_handles_labels()
L=["1980标准","1990标准","2000标准","2010标准","2020标准","低能耗","近零能耗"]
L.reverse()
ax10[0][2].legend(handles,L,ncol=1,loc='right',bbox_to_anchor=(0.95,0.5))
ax10[1][2].legend(handles,L,ncol=1,loc='right',bbox_to_anchor=(0.95,0.5))
ax10[2][2].legend(handles,L,ncol=1,loc='right',bbox_to_anchor=(0.95,0.5))

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_Area"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Final energy -BLD
FE_BLD_S=Data.filter(variable="Final Energy|Building*",level=1)
FE_BLD_S=FE_BLD_S.convert_unit("PJ","亿吨标煤",1/0.293076/10000)
FE_BLD_S_order=['Final Energy|Building|Electricity','Final Energy|Building|Hydrogen','Final Energy|Building|Solar','Final Energy|Building|Geothermal','Final Energy|Building|Biomass','Final Energy|Building|Heat','Final Energy|Building|Gas','Final Energy|Building|Liquids','Final Energy|Building|Coal']
L=['煤',"油品","燃气","热","生物质","地热","太阳能","氢能","电"]

# Final energy -BLD-NDC
scen='ndc-ts'
Fig, ax = plt.subplots()
FE_BLD_S.filter(scenario=scen).filter(keep=False,year=[2025,2028,2029,2030,2031,2032]).plot.stack(ax=ax,
        title=False,legend=True,order=FE_BLD_S_order)
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=reversed(handles),labels=L,ncol=5,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
plt.xlabel("时间")
plt.ylabel("建筑部门终端能源消费（亿吨标煤）")

# plt.savefig(ImgPath+"chap3_FE_BLD-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# Final energy -BLD-CN60
scen='ndc-ts_2c-cn60'
Fig, ax = plt.subplots()
FE_BLD_S.filter(scenario=scen).filter(keep=False,year=[2025,2028,2029,2030,2031,2032,2040]).plot.stack(ax=ax,
        title=False,legend=True,order=FE_BLD_S_order)
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=reversed(handles),labels=L,ncol=5,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
plt.xlabel("时间")
plt.ylabel("建筑部门终端能源消费（亿吨标煤）")

# plt.savefig(ImgPath+"chap3_FE_BLD-CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Building Heat
BLD_Heat=Data.filter(variable="BLD_Heat*",year=2060)
BLD_Heat_20=pyam.IamDataFrame(Data.filter(variable="BLD_Heat*",scenario='ndc-ts',year=2020).rename(scenario={'ndc-ts':'2020年'}).as_pandas(False).replace(2020,2060))
BLD_Heat.append(BLD_Heat_20,inplace=True)
BLD_Heat_30=pyam.IamDataFrame(Data.filter(variable="BLD_Heat*",scenario='ndc-ts',year=2030).rename(scenario={'ndc-ts':'2030年'}).as_pandas(False).replace(2030,2060))
BLD_Heat.append(BLD_Heat_30,inplace=True)
scen_sdg=['2020年','2030年',"ndc-ts",'ndc-ts_2c-cn60','ndc-ts_2c-cn60-noCF']
S_name=['2020年','2030年',"REF",'CN60',"CN60-noCF"]
BLD_Heat=BLD_Heat.convert_unit("PJ","万吨标煤",1/0.293076)
Fig9, ax9 = plt.subplots(3, 5, figsize = (12, 10), sharey = False, sharex=True)
BLD_Heat.filter(variable="*COM|SC*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[0][0],legend=False,title="严寒公共建筑",width=0.8)
ax9[0][0].set_ylabel("空间供暖用能（万吨标煤）")
BLD_Heat.filter(variable="*COM|C*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[0][1],legend=False,title="寒冷公共建筑",width=0.8)
ax9[0][1].set_ylabel(None)
BLD_Heat.filter(variable="*COM|HSCW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[0][2],legend=False,title="夏热冬冷公共建筑",width=0.8)
ax9[0][2].set_ylabel(None)
BLD_Heat.filter(variable="*COM|HSWW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[0][3],legend=False,title="夏热冬暖公共建筑",width=0.8)
ax9[0][3].set_ylabel(None)
BLD_Heat.filter(variable="*COM|M*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[0][4],legend=True,title="温和公共建筑",width=0.8)
ax9[0][4].set_ylabel(None)
handles,l= ax9[0][4].get_legend_handles_labels()
L=["煤","电","气","地热","热",'油']

ax9[0][4].legend(handles,L,loc='center left',bbox_to_anchor=(1,0.5),frameon=False)

BLD_Heat.filter(variable="*RUL|SC*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[1][0],legend=False,title="严寒农村建筑",width=0.8)
ax9[1][0].set_ylabel("空间供暖用能（万吨标煤）")
BLD_Heat.filter(variable="*RUL|C*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[1][1],legend=False,title="寒冷农村建筑",width=0.8)
ax9[1][1].set_ylabel(None)
BLD_Heat.filter(variable="*RUL|HSCW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[1][2],legend=False,title="夏热冬冷农村建筑",width=0.8)
ax9[1][2].set_ylabel(None)
BLD_Heat.filter(variable="*RUL|HSWW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[1][3],legend=False,title="夏热冬暖农村建筑",width=0.8)
ax9[1][3].set_ylabel(None)
BLD_Heat.filter(variable="*RUL|M*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[1][4],legend=True,title="温和农村建筑",width=0.8)
ax9[1][4].set_ylabel(None)
handles,l= ax9[1][4].get_legend_handles_labels()
L=["沼气","生物质","煤","电","气","地热",'油']

ax9[1][4].legend(handles,L,loc='center left',bbox_to_anchor=(1,0.5),frameon=False)

BLD_Heat.filter(variable="*URB|SC*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[2][0],legend=False,title="严寒城镇建筑",width=0.8)
ax9[2][0].set_ylabel("空间供暖用能（万吨标煤）")
BLD_Heat.filter(variable="*URB|C*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[2][1],legend=False,title="寒冷城镇建筑",width=0.8)
ax9[2][1].set_ylabel(None)
BLD_Heat.filter(variable="*URB|HSCW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[2][2],legend=False,title="夏热冬冷城镇建筑",width=0.8)
ax9[2][2].set_ylabel(None)
BLD_Heat.filter(variable="*URB|HSWW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[2][3],legend=False,title="夏热冬暖城镇建筑",width=0.8)
ax9[2][3].set_ylabel(None)
BLD_Heat.filter(variable="*URB|M*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[2][4],legend=True,title="温和城镇建筑",width=0.8)
ax9[2][4].set_ylabel(None)
handles,l= ax9[2][4].get_legend_handles_labels()
L=["煤","电","气","地热","热",'油']

ax9[2][4].legend(handles,L,loc='center left',bbox_to_anchor=(1,0.5),frameon=False)

for j in [0,1,2,3,4]:
    ax9[2][j].set_xlabel(None)
    ax9[2][j].set_xticklabels(S_name)
    ax9[0][j].xaxis.set_minor_locator(plt.NullLocator())
    ax9[1][j].xaxis.set_minor_locator(plt.NullLocator())
    ax9[2][j].xaxis.set_minor_locator(plt.NullLocator())
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_Heat"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
# Building Cool
BLD_Cool=Data.filter(variable="BLD_Cool*Electricity",year=2060)
BLD_Cool_20=pyam.IamDataFrame(Data.filter(variable="BLD_Cool*Electricity",scenario='ndc-ts',year=2020).rename(scenario={'ndc-ts':'2020年'}).as_pandas(False).replace(2020,2060))
BLD_Cool.append(BLD_Cool_20,inplace=True)
BLD_Cool_30=pyam.IamDataFrame(Data.filter(variable="BLD_Cool*Electricity",scenario='ndc-ts',year=2030).rename(scenario={'ndc-ts':'2030年'}).as_pandas(False).replace(2030,2060))
BLD_Cool.append(BLD_Cool_30,inplace=True)
BLD_Cool=BLD_Cool.convert_unit("PJ","万吨标煤",1/0.293076)
scen_sdg=['2020年','2030年',"ndc-ts",'ndc-ts_2c-cn60','ndc-ts_2c-cn60-noCF']
S_name=['2020年','2030年',"REF",'CN60',"CN60-noCF"]
l=["Coal","Electricity","Gas","Geothermal","Cool"]
Fig9, ax9 = plt.subplots(3, 5, figsize = (10, 10), sharey = False, sharex=True)
plt.ylabel(None)
BLD_Cool.filter(variable="*COM|SC*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[0][0],legend=False,title="严寒公共建筑",width=0.8)
ax9[0][0].set_ylabel("空间供冷用能（万吨标煤）")
BLD_Cool.filter(variable="*COM|C*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[0][1],legend=False,title="寒冷公共建筑",width=0.8)
ax9[0][1].set_ylabel(None)
BLD_Cool.filter(variable="*COM|HSCW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[0][2],legend=False,title="夏热冬冷公共建筑",width=0.8)
ax9[0][2].set_ylabel(None)
BLD_Cool.filter(variable="*COM|HSWW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[0][3],legend=False,title="夏热冬暖公共建筑",width=0.8)
ax9[0][3].set_ylabel(None)
BLD_Cool.filter(variable="*COM|M*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[0][4],legend=False,title="温和公共建筑",width=0.8)
ax9[0][4].set_ylabel(None)

BLD_Cool.filter(variable="*RUL|SC*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[1][0],legend=False,title="严寒农村建筑",width=0.8)
ax9[1][0].set_ylabel("空间供冷用能（万吨标煤）")
BLD_Cool.filter(variable="*RUL|C*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[1][1],legend=False,title="寒冷农村建筑",width=0.8)
ax9[1][1].set_ylabel(None)
BLD_Cool.filter(variable="*RUL|HSCW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[1][2],legend=False,title="夏热冬冷农村建筑",width=0.8)
ax9[1][2].set_ylabel(None)
BLD_Cool.filter(variable="*RUL|HSWW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[1][3],legend=False,title="夏热冬暖农村建筑",width=0.8)
ax9[1][3].set_ylabel(None)
BLD_Cool.filter(variable="*RUL|M*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[1][4],legend=False,title="温和农村建筑",width=0.8)
ax9[1][4].set_ylabel(None)

BLD_Cool.filter(variable="*URB|SC*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[2][0],legend=False,title="严寒城镇建筑",width=0.8)
ax9[2][0].set_ylabel("空间供冷用能（万吨标煤）")
BLD_Cool.filter(variable="*URB|C*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[2][1],legend=False,title="寒冷城镇建筑",width=0.8)
ax9[2][1].set_ylabel(None)
BLD_Cool.filter(variable="*URB|HSCW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[2][2],legend=False,title="夏热冬冷城镇建筑",width=0.8)
ax9[2][2].set_ylabel(None)
BLD_Cool.filter(variable="*URB|HSWW*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[2][3],legend=False,title="夏热冬暖城镇建筑",width=0.8)
ax9[2][3].set_ylabel(None)
BLD_Cool.filter(variable="*URB|M*",year=2060,scenario=scen_sdg).plot.bar(stacked=True, x="scenario",ax = ax9[2][4],legend=False,title="温和城镇建筑",width=0.8)
ax9[2][4].set_ylabel(None)

for j in [0,1,2,3,4]:
    ax9[2][j].set_xlabel(None)
    ax9[2][j].set_xticklabels(S_name)
    ax9[0][j].xaxis.set_minor_locator(plt.NullLocator())
    ax9[1][j].xaxis.set_minor_locator(plt.NullLocator())
    ax9[2][j].xaxis.set_minor_locator(plt.NullLocator())
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_Cool"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
COOK=Data.filter(variable='BLD_DMD|*|*|COOK|*')
COM_COA=COOK.filter(variable='BLD_DMD|COM|*|COOK|Coal').variable
COM_ELC=COOK.filter(variable='BLD_DMD|COM|*|COOK|Electricity').variable
COM_GAS=COOK.filter(variable='BLD_DMD|COM|*|COOK|Gas').variable
COM_OIL=COOK.filter(variable='BLD_DMD|COM|*|COOK|Liquid').variable
COM_SOL=COOK.filter(variable='BLD_DMD|COM|*|COOK|Solar').variable

RUL_BGS=COOK.filter(variable='BLD_DMD|RUL|*|COOK|Biogas').variable
RUL_BSL=COOK.filter(variable='BLD_DMD|RUL|*|COOK|Biomass').variable
RUL_COA=COOK.filter(variable='BLD_DMD|RUL|*|COOK|Coal').variable
RUL_ELC=COOK.filter(variable='BLD_DMD|RUL|*|COOK|Electricity').variable
RUL_GAS=COOK.filter(variable='BLD_DMD|RUL|*|COOK|Gas').variable
RUL_OIL=COOK.filter(variable='BLD_DMD|RUL|*|COOK|Liquid').variable
RUL_SOL=COOK.filter(variable='BLD_DMD|RUL|*|COOK|Solar').variable

URB_COA=COOK.filter(variable='BLD_DMD|URB|*|COOK|Coal').variable
URB_ELC=COOK.filter(variable='BLD_DMD|URB|*|COOK|Electricity').variable
URB_GAS=COOK.filter(variable='BLD_DMD|URB|*|COOK|Gas').variable
URB_OIL=COOK.filter(variable='BLD_DMD|URB|*|COOK|Liquid').variable
URB_SOL=COOK.filter(variable='BLD_DMD|URB|*|COOK|Solar').variable

COOK.aggregate('COM_COA',COM_COA,append=True)
COOK.aggregate('COM_ELC',COM_ELC,append=True)
COOK.aggregate('COM_GAS',COM_GAS,append=True)
COOK.aggregate('COM_OIL',COM_OIL,append=True)
COOK.aggregate('COM_SOL',COM_SOL,append=True)
COOK.aggregate('RUL_BGS',RUL_BGS,append=True)
COOK.aggregate('RUL_BSL',RUL_BSL,append=True)
COOK.aggregate('RUL_COA',RUL_COA,append=True)
COOK.aggregate('RUL_ELC',RUL_ELC,append=True)
COOK.aggregate('RUL_GAS',RUL_GAS,append=True)
COOK.aggregate('RUL_OIL',RUL_OIL,append=True)
COOK.aggregate('RUL_SOL',RUL_SOL,append=True)
COOK.aggregate('URB_COA',URB_COA,append=True)
COOK.aggregate('URB_ELC',URB_ELC,append=True)
COOK.aggregate('URB_GAS',URB_GAS,append=True)
COOK.aggregate('URB_OIL',URB_OIL,append=True)
COOK.aggregate('URB_SOL',URB_SOL,append=True)
COOK=COOK.filter(keep=False,variable='BLD*')
COOK=COOK.convert_unit("PJ","万吨标煤",1/0.293076)
# COOK-COM
COM_COOK=COOK.filter(variable='COM*')
bar_order=['COM_COA', 'COM_OIL', 'COM_GAS','COM_ELC', 'COM_SOL']
L=["煤",'油','气',"电",'太阳能']
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2019,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

COM_COOK.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2019年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("炊事能源服务需求（万吨标煤）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    COM_COOK.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("炊事能源服务需求（万吨标煤）")
ax6[2].legend(L,ncol=3,loc='upper center',bbox_to_anchor=(0.2,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_COOK_COM"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# COOK-RUL
RUL_COOK=COOK.filter(variable='RUL*')
bar_order=['RUL_COA','RUL_OIL','RUL_GAS','RUL_BGS','RUL_BSL', 'RUL_ELC',   'RUL_SOL']
L=["煤",'油','气','沼气','生物质',"电",'太阳能']
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2019,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

RUL_COOK.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2019年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("炊事能源服务需求（万吨标煤）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    RUL_COOK.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("炊事能源服务需求（万吨标煤）")
ax6[2].legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0.2,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_COOK_RUL"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# COOK-URB
URB_COOK=COOK.filter(variable='URB*')
bar_order=['URB_COA','URB_OIL', 'URB_GAS', 'URB_ELC',  'URB_SOL']
L=["煤",'油','气',"电",'太阳能']
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2019,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

URB_COOK.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2019年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("炊事能源服务需求（万吨标煤）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    URB_COOK.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("炊事能源服务需求（万吨标煤）")
ax6[2].legend(L,ncol=3,loc='upper center',bbox_to_anchor=(0.2,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_COOK_URB"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
LIGHT=Data.filter(variable='BLD_DMD|*|*|LIGHT|*')
COM_LED=LIGHT.filter(variable='BLD_DMD|COM|*|LIGHT|LED').variable
COM_FLU=LIGHT.filter(variable='BLD_DMD|COM|*|LIGHT|Fluorescent').variable
COM_HAL=LIGHT.filter(variable='BLD_DMD|COM|*|LIGHT|Halogen').variable
COM_INC=LIGHT.filter(variable='BLD_DMD|COM|*|LIGHT|Incandescent').variable

RUL_LED=LIGHT.filter(variable='BLD_DMD|RUL|*|LIGHT|LED').variable
RUL_FLU=LIGHT.filter(variable='BLD_DMD|RUL|*|LIGHT|Fluorescent').variable
RUL_HAL=LIGHT.filter(variable='BLD_DMD|RUL|*|LIGHT|Halogen').variable
RUL_INC=LIGHT.filter(variable='BLD_DMD|RUL|*|LIGHT|Incandescent').variable

URB_LED=LIGHT.filter(variable='BLD_DMD|URB|*|LIGHT|LED').variable
URB_FLU=LIGHT.filter(variable='BLD_DMD|URB|*|LIGHT|Fluorescent').variable
URB_HAL=LIGHT.filter(variable='BLD_DMD|URB|*|LIGHT|Halogen').variable
URB_INC=LIGHT.filter(variable='BLD_DMD|URB|*|LIGHT|Incandescent').variable

LIGHT.aggregate('COM_LED',COM_LED,append=True)
LIGHT.aggregate('COM_FLU',COM_FLU,append=True)
LIGHT.aggregate('COM_HAL',COM_HAL,append=True)
LIGHT.aggregate('COM_INC',COM_INC,append=True)

LIGHT.aggregate('RUL_LED',RUL_LED,append=True)
LIGHT.aggregate('RUL_FLU',RUL_FLU,append=True)
LIGHT.aggregate('RUL_HAL',RUL_HAL,append=True)
LIGHT.aggregate('RUL_INC',RUL_INC,append=True)

LIGHT.aggregate('URB_LED',URB_LED,append=True)
LIGHT.aggregate('URB_FLU',URB_FLU,append=True)
LIGHT.aggregate('URB_HAL',URB_HAL,append=True)
LIGHT.aggregate('URB_INC',URB_INC,append=True)

LIGHT=LIGHT.filter(keep=False,variable='BLD*')
LIGHT=LIGHT.convert_unit('100Tlm·h','Plm·h',1/10)
# LIGHT-COM
COM_LIGHT=LIGHT.filter(variable='COM*')
bar_order=['COM_INC', 'COM_FLU', 'COM_HAL','COM_LED']
L=["白炽灯",'荧光灯','卤素灯','LED灯']
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2019,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

COM_LIGHT.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2019年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("照明能源服务需求（Plm·h）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    COM_LIGHT.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("照明能源服务需求（Plm·h）")
ax6[2].legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0.1,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_LIGHT_COM"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# LIGHT-RUL
RUL_LIGHT=LIGHT.filter(variable='RUL*')
bar_order=['RUL_INC', 'RUL_FLU', 'RUL_HAL','RUL_LED']
L=["白炽灯",'荧光灯','卤素灯','LED灯']
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2019,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

RUL_LIGHT.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2019年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("照明能源服务需求（Plm·h）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    RUL_LIGHT.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("照明能源服务需求（Plm·h）")
ax6[2].legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0.1,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_LIGHT_RUL"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# LIGHT-URB
URB_LIGHT=LIGHT.filter(variable='URB*')
bar_order=['URB_INC', 'URB_FLU', 'URB_HAL','URB_LED']
L=["白炽灯",'荧光灯','卤素灯','LED灯']
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2019,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

URB_LIGHT.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2019年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("照明能源服务需求（Plm·h）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    URB_LIGHT.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("照明能源服务需求（Plm·h）")
ax6[2].legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0.1,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_LIGHT_URB"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
HW=Data.filter(variable='BLD_DMD|*|*|HW|*')
COM_COA=HW.filter(variable='BLD_DMD|COM|*|HW|Coal').variable
COM_ELC=HW.filter(variable='BLD_DMD|COM|*|HW|Electricity').variable
COM_GAS=HW.filter(variable='BLD_DMD|COM|*|HW|Gas').variable
COM_GEO=HW.filter(variable='BLD_DMD|COM|*|HW|Geothermal').variable
COM_OIL=HW.filter(variable='BLD_DMD|COM|*|HW|Liquid').variable
COM_SOL=HW.filter(variable='BLD_DMD|COM|*|HW|Solar').variable

RUL_BGS=HW.filter(variable='BLD_DMD|RUL|*|HW|Biogas').variable
RUL_BSL=HW.filter(variable='BLD_DMD|RUL|*|HW|Biomass').variable
RUL_COA=HW.filter(variable='BLD_DMD|RUL|*|HW|Coal').variable
RUL_ELC=HW.filter(variable='BLD_DMD|RUL|*|HW|Electricity').variable
RUL_GAS=HW.filter(variable='BLD_DMD|RUL|*|HW|Gas').variable
RUL_GEO=HW.filter(variable='BLD_DMD|RUL|*|HW|Geothermal').variable
RUL_OIL=HW.filter(variable='BLD_DMD|RUL|*|HW|Liquid').variable
RUL_SOL=HW.filter(variable='BLD_DMD|RUL|*|HW|Solar').variable

URB_COA=HW.filter(variable='BLD_DMD|URB|*|HW|Coal').variable
URB_ELC=HW.filter(variable='BLD_DMD|URB|*|HW|Electricity').variable
URB_GAS=HW.filter(variable='BLD_DMD|URB|*|HW|Gas').variable
URB_GEO=HW.filter(variable='BLD_DMD|URB|*|HW|Geothermal').variable
URB_OIL=HW.filter(variable='BLD_DMD|URB|*|HW|Liquid').variable
URB_SOL=HW.filter(variable='BLD_DMD|URB|*|HW|Solar').variable

HW.aggregate('COM_COA',COM_COA,append=True)
HW.aggregate('COM_ELC',COM_ELC,append=True)
HW.aggregate('COM_GAS',COM_GAS,append=True)
HW.aggregate('COM_GEO',COM_GEO,append=True)
HW.aggregate('COM_OIL',COM_OIL,append=True)
HW.aggregate('COM_SOL',COM_SOL,append=True)
HW.aggregate('RUL_BGS',RUL_BGS,append=True)
HW.aggregate('RUL_BSL',RUL_BSL,append=True)
HW.aggregate('RUL_COA',RUL_COA,append=True)
HW.aggregate('RUL_ELC',RUL_ELC,append=True)
HW.aggregate('RUL_GAS',RUL_GAS,append=True)
HW.aggregate('RUL_GEO',RUL_GEO,append=True)
HW.aggregate('RUL_OIL',RUL_OIL,append=True)
HW.aggregate('RUL_SOL',RUL_SOL,append=True)
HW.aggregate('URB_COA',URB_COA,append=True)
HW.aggregate('URB_ELC',URB_ELC,append=True)
HW.aggregate('URB_GAS',URB_GAS,append=True)
HW.aggregate('URB_GEO',URB_GEO,append=True)
HW.aggregate('URB_OIL',URB_OIL,append=True)
HW.aggregate('URB_SOL',URB_SOL,append=True)
HW=HW.filter(keep=False,variable='BLD*')
HW=HW.convert_unit("PJ","万吨标煤",1/0.293076)
# HW-COM
COM_HW=HW.filter(variable='COM*')
bar_order=['COM_COA', 'COM_OIL', 'COM_GAS','COM_ELC', 'COM_GEO', 'COM_SOL']
L=["煤",'油','气',"电",'地热','太阳能']
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2019,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

COM_HW.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2019年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("热水能源服务需求（万吨标煤）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    COM_HW.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("热水能源服务需求（万吨标煤）")
ax6[2].legend(L,ncol=3,loc='upper center',bbox_to_anchor=(0.2,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_HW_COM"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# HW-RUL
RUL_HW=HW.filter(variable='RUL*')
bar_order=['RUL_COA','RUL_OIL','RUL_GAS','RUL_BGS','RUL_BSL', 'RUL_ELC', 'RUL_GEO',  'RUL_SOL']
L=["煤",'油','气','沼气','生物质',"电",'地热','太阳能']
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2019,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

RUL_HW.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2019年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("热水能源服务需求（万吨标煤）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    RUL_HW.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("热水能源服务需求（万吨标煤）")
ax6[2].legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0.2,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_HW_RUL"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# HW-URB
URB_HW=HW.filter(variable='URB*')
bar_order=['URB_COA', 'URB_OIL', 'URB_GAS','URB_ELC', 'URB_GEO', 'URB_SOL']
L=["煤",'油','气',"电",'地热','太阳能']
scen_sdg=["ndc-ts",'ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2019,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

URB_HW.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2019年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("热水能源服务需求（万吨标煤）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    URB_HW.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("热水能源服务需求（万吨标煤）")
ax6[2].legend(L,ncol=3,loc='upper center',bbox_to_anchor=(0.2,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_BLD_HW_URB"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Transport turnover-F
TRA_F=Data.filter(variable="Energy Service|TRA|F*",level=1)
TRA_F=TRA_F.convert_unit('billion t-km','trillion t-km',1/1000)
L=['国内货运','国际空运','国际水运','管道运输','农村货运']
TRA_F_order=[ 'Energy Service|TRA|Freight|Rural',
    'Energy Service|TRA|Freight|Pipline',
    'Energy Service|TRA|Freight|International Ship',
    'Energy Service|TRA|Freight|International Air',
    'Energy Service|TRA|Freight|Domestic']
L.reverse()
Fig, ax = plt.subplots()
TRA_F.filter(scenario='ndc-ts').plot.stack(ax=ax,x='year',legend=True,order=TRA_F_order,cmap=sns.color_palette("Set2",as_cmap=True),title=False);
plt.xlabel("时间")
plt.ylabel("货运周转量（万亿吨公里）")
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=handles,labels=L,ncol=1,loc='lower right',bbox_to_anchor=(0.95,0.05))
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TRA_F_Turnover"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# Transport turnover-P
TRA_P=Data.filter(variable="Energy Service|TRA|P*",level=1)
TRA_P=TRA_P.convert_unit('billion p-km','trillion p-km',1/1000)
L=['城乡客运','公务客运','城间客运','国际空运']
L.reverse()
TRA_P_order=['Energy Service|TRA|Passenger|Urban',
             'Energy Service|TRA|Passenger|Business',
             'Energy Service|TRA|Passenger|Inter-city',
             'Energy Service|TRA|Passenger|International Air']
TRA_P_order.reverse()
Fig, ax = plt.subplots()
TRA_P.filter(scenario='ndc-ts').plot.stack(ax=ax,x='year',legend=True,order=TRA_P_order,cmap=sns.color_palette("Set2",as_cmap=True),title=False);
plt.xlabel("时间")
plt.ylabel("客运周转量（万亿人公里）")
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=handles,labels=L,ncol=1,loc='lower right',bbox_to_anchor=(0.95,0.05))

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TRA_P_Turnover"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Final energy -TRA
FE_TRA_S=Data.filter(variable="Final Energy|Transport*",level=1)
FE_TRA_S=FE_TRA_S.convert_unit("PJ","亿吨标煤",1/0.293076/10000)
FE_TRA_S_order=['Final Energy|Transport|Electricity','Final Energy|Transport|Hydrogen','Final Energy|Transport|Gas','Final Energy|Transport|Liquids','Final Energy|Transport|Coal']
L=['煤',"油品","燃气","氢能","电"]

# Final energy -TRA-NDC
scen='ndc-ts'
Fig, ax = plt.subplots()
FE_TRA_S.filter(scenario=scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,
        title=False,legend=True,order=FE_TRA_S_order)
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=reversed(handles),labels=L,ncol=6,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
plt.xlabel("时间")
plt.ylabel("交通部门终端能源需求（亿吨标煤）")

# plt.savefig(ImgPath+"chap3_FE_TRA-NDC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# Final energy -TRA-CN60
scen='ndc-ts_2c-cn60'
Fig, ax = plt.subplots()
FE_TRA_S.filter(scenario=scen).filter(keep=False,year=[2022,2031,2032]).plot.stack(ax=ax,
        title=False,legend=True,order=FE_TRA_S_order)
handles,l= ax.get_legend_handles_labels()
ax.legend(handles=reversed(handles),labels=L,ncol=6,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
plt.xlabel("时间")
plt.ylabel("交通部门终端能源消费（亿吨标煤）")

# plt.savefig(ImgPath+"chap3_FE_TRA-CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# TRA_F_DOM
bar_order=['TRA_F|DOM|AIR Total',
           'TRA_F|DOM|SHIP Total',
           'TRA_F|DOM|RAIL Total',
           'TRA_F|DOM|ROAD|MNT Total',
           'TRA_F|DOM|ROAD|LDT Total',
           'TRA_F|DOM|ROAD|MDT Total',
           'TRA_F|DOM|ROAD|HDT Total']
L=["大型货车",'中型货车','小型货车','微型货车','铁路','水运','航空']
bar_order.reverse()
TRA_F_DOM=Data.filter(variable=bar_order)
TRA_F_DOM=TRA_F_DOM.convert_unit('billion t-km','trillion t-km',1/1000)
scen_sdg=['ndc-ts','ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2020,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

TRA_F_DOM.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2020年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("货运周转量（万亿吨公里）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    TRA_F_DOM.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("货运周转量（万亿吨公里）")
ax6[2].legend(L,ncol=4,loc='upper center',bbox_to_anchor=(0,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TRA_F_DOM"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# TRA_P_URBAN
bar_order=['TRA_P|URBAN|METRO Total',
           'TRA_P|URBAN|TAXI Total',
           'TRA_P|URBAN|MOTOR Total',
           'TRA_P|URBAN|BUS Total',
           'TRA_P|URBAN|LDV Total']
L=["私家车",'公交车','摩托车','出租车','轨道交通']
bar_order.reverse()
TRA_P_URB=Data.filter(variable=bar_order)
TRA_P_URB=TRA_P_URB.convert_unit('billion p-km','trillion p-km',1/1000)
scen_sdg=['ndc-ts','ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2020,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

TRA_P_URB.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2020年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("客运周转量（万亿人公里）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    TRA_P_URB.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("客运周转量（万亿人公里）")
ax6[2].legend(L,ncol=3,loc='upper center',bbox_to_anchor=(0,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TRA_P_URB"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# TRA_P_INT
bar_order=[
    'TRA_P|INT|AIR Total',
    'TRA_P|INT|SHIP Total',
    'TRA_P|INT|RAIL Total',
    'TRA_P|INT|BUS|HDB Total',
    'TRA_P|INT|BUS|LDB Total',
    'TRA_P|INT|LDV Total']
L=["私家车",'中型客车','大型客车','铁路','水路','航空']
bar_order.reverse()
TRA_P_INT=Data.filter(variable=bar_order)
TRA_P_INT=TRA_P_INT.convert_unit('billion p-km','trillion p-km',1/1000)
scen_sdg=['ndc-ts','ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2020,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

TRA_P_INT.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2020年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("客运周转量（万亿人公里）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    TRA_P_INT.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("客运周转量（万亿人公里）")
ax6[2].legend(L,ncol=3,loc='upper center',bbox_to_anchor=(0,-0.1),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TRA_P_INT"+".pdf",bbox_inches='tight',dpi=300)
plt.show()




In [ ]:
# TRA_F_RUL
bar_order=[
    'TRA_F|RURAL|3W Total',
 'TRA_F|RURAL|4W Total']
L=["四轮低速汽车",'三轮低速汽车']
bar_order.reverse()
TRA_F_RUL=Data.filter(variable=bar_order)
TRA_F_RUL=TRA_F_RUL.convert_unit('billion t-km','0.1 billion p-km',10)
scen_sdg=['ndc-ts','ndc-ts_2c-cn60']
S_name=['REF','CN60']
Y=[2020,2035,2060,2100]
Fig6, ax6 = plt.subplots(1, 4, figsize = (5, 3), sharey = True)
Fig6=plt.figure(figsize = (5, 3))
grid = plt.GridSpec(1,7,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

TRA_F_RUL.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2020年",rotation=0)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("货运周转量（亿吨公里）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*2):(1+2*i)],sharey=ax6[0])
    TRA_F_RUL.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen_sdg);
    
for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("货运周转量（亿吨公里）")
plt.legend(L,bbox_to_anchor=(0,-0.1),ncol=1,frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TRA_F_RUL"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
#LDV
def sss(x,y,i):
    r=["","","","","","","",""]
    for k in range(0,8):
        r[k]=x[i]+y[k]
    return r

Y=[2019,2025,2035,2060]
Fig7, ax7 = plt.subplots(4, 4, figsize = (14, 3))
Fig7=plt.figure(figsize = (14, 3))
grid = plt.GridSpec(7,4,wspace=0.1,hspace=0)
F=['Diesel','Gasoline','LPG','Natural gas','Biodiesel','Biogasoline','Electricity', 'Hydrogen']
L=["柴油",'汽油','液化石油气','天然气','生物柴油','乙醇汽油','电力','氢能']
S=['TRA_P|URB|LDV|','TRA_P|INT|LDV|','TRA_P|BUSI|LDV|','TRA_P|URB|TAXI|']
scen_sdg=['ndc-ts','ndc-ts_2c-cn60']
S_name=['REF','CN60']
TRA_LDV=Data.filter(scenario=scen_sdg,variable=['TRA_P*|LDV|*','TRA_P*|TAXI|*']).filter(variable='*Total',keep=False)
TRA_LDV=TRA_LDV.convert_unit('billion p-km','trillion p-km',1/1000)
for i in range (0,4):
    ax7[0][i]=Fig7.add_subplot(grid[0,i],sharex=ax7[0][i],sharey=ax7[0][0])
    TRA_LDV.filter(variable=sss(S,F,i)).filter(year=2019,scenario=['ndc-ts']).rename(scenario={"ndc-ts":"2019年"}).plot.bar(stacked =True,y='value',x='scenario',bars='variable',width=0.8,
                                                bars_order=sss(S,F,i),orient='h',legend=False,ax=ax7[0][i],title=False,cmap=sns.color_palette("Set2_r",as_cmap=True))


for j in range (1,4):
    for i in range (0,4):  
        ax7[j][i]=Fig7.add_subplot(grid[(1+2*(j-1)):(1+2*j),i],sharex=ax7[0][i],sharey=ax7[j][0])
        TRA_LDV.filter(variable=sss(S,F,i)).filter(year=Y[j]).rename(scenario={"ndc-ts":"REF","ndc-ts_2c-cn60":"CN60"}).plot.bar(stacked =True,y='value',x='scenario',bars='variable',width=0.8,
                                                bars_order=sss(S,F,i),orient='h',legend=False,ax=ax7[j][i],title=False,cmap=sns.color_palette("Set2_r",as_cmap=True))

ax7[3][0].set_xlabel("城市小汽车（万亿人公里）")
ax7[3][1].set_xlabel("城际小汽车（万亿人公里）")
ax7[3][2].set_xlabel("城市出租车（万亿人公里）")
ax7[3][3].set_xlabel("公务小汽车（万亿人公里）")
ax7[0][0].set_ylabel("")
ax7[1][0].set_ylabel("2025年")
ax7[2][0].set_ylabel("2035年")
ax7[3][0].set_ylabel("2060年")
ax7[0][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[1][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[2][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[3][0].yaxis.set_minor_locator(plt.NullLocator())


ax7[3][2].legend(L,ncol=8,loc='upper center',bbox_to_anchor=(-0.2,-0.7),frameon=False,fontsize=12,columnspacing=1)
# plt.savefig(ImgPath+"chap3_TRA_LDV"+".pdf",bbox_inches='tight',dpi=300)
plt.show() 

In [ ]:
#air plane
def sss(x,y,i):
    r=["","","",""]
    for k in range(0,4):
        r[k]=x[i]+y[k]
    return r

Y=[2019,2035,2060,2100]
Fig7, ax7 = plt.subplots(4, 4, figsize = (14, 3))
Fig7=plt.figure(figsize = (14, 3))
grid = plt.GridSpec(7,4,wspace=0.1,hspace=0)
F=['Kerosene','Biofuel','Electricity','Hydrogen']
L=["航空煤油",'生物航油','电动飞机','氢能飞机']
S=['TRA_F|DOM|AIR|','TRA_F|INTL|AIR|','TRA_P|INT|AIR|','TRA_P|INTL|AIR|']
scen_sdg=['ndc-ts','ndc-ts_2c-cn60']
S_name=['REF','CN60']
TRA_Air=Data.filter(scenario=scen_sdg,variable='TRA_*AIR*').filter(variable='*Total',keep=False)
for i in range (0,4):
    ax7[0][i]=Fig7.add_subplot(grid[0,i],sharex=ax7[0][i],sharey=ax7[0][0])
    TRA_Air.filter(variable=sss(S,F,i)).filter(year=2019,scenario=['ndc-ts']).rename(scenario={"ndc-ts":"2019年"}).plot.bar(stacked =True,y='value',x='scenario',bars='variable',width=0.8,
                                                bars_order=sss(S,F,i),orient='h',legend=False,ax=ax7[0][i],title=False,cmap=sns.color_palette("Set2_r",as_cmap=True))


for j in range (1,4):
    for i in range (0,4):  
        ax7[j][i]=Fig7.add_subplot(grid[(1+2*(j-1)):(1+2*j),i],sharex=ax7[0][i],sharey=ax7[j][0])
        TRA_Air.filter(variable=sss(S,F,i)).filter(year=Y[j]).rename(scenario={"ndc-ts":"REF","ndc-ts_2c-cn60":"CN60"}).plot.bar(stacked =True,y='value',x='scenario',bars='variable',width=0.8,
                                                 bars_order=sss(S,F,i),orient='h',legend=False,ax=ax7[j][i],title=False,cmap=sns.color_palette("Set2_r",as_cmap=True))

ax7[3][0].set_xlabel("国内航空货运（十亿吨公里）")
ax7[3][1].set_xlabel("国际航空货运（十亿吨公里）")
ax7[3][2].set_xlabel("国内航空客运（十亿人公里）")
ax7[3][3].set_xlabel("国际航空客运（十亿人公里）")
ax7[0][0].set_ylabel("")
ax7[1][0].set_ylabel("2035年")
ax7[2][0].set_ylabel("2060年")
ax7[3][0].set_ylabel("2100年")
ax7[0][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[1][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[2][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[3][0].yaxis.set_minor_locator(plt.NullLocator())

plt.legend(L,ncol=4,bbox_to_anchor=(0, -0.7),frameon=False,fontsize=12,columnspacing=1)
# plt.savefig(ImgPath+"chap3_TRA_Air"+".pdf",bbox_inches='tight',dpi=300)
plt.show() 

In [ ]:
# truck
def sss(x,y,i):
    r=["","","","","","",""]
    for k in range(0,7):
        r[k]=x[i]+y[k]
    return r

Y=[2019,2035,2060,2100]
Fig7, ax7 = plt.subplots(4, 4, figsize = (14, 3))
Fig7=plt.figure(figsize = (14, 3))
grid = plt.GridSpec(7,4,wspace=0.1,hspace=0)
F=['Diesel','Gasoline','Natural gas','Biodiesel','Biogasoline','Electricity', 'Hydrogen']
L=["柴油",'汽油','天然气','生物柴油','乙醇汽油','电力','氢能']
S=['TRA_F|DOM|HDT|','TRA_F|DOM|MDT|','TRA_F|DOM|LDT|','TRA_F|DOM|MNT|']
scen_sdg=['ndc-ts','ndc-ts_2c-cn60']
S_name=['REF','CN60']
TRA_Truck=Data.filter(scenario=scen_sdg,variable='TRA_F|DOM|*T*').filter(variable='*Total',keep=False)

for i in range (0,4):
    ax7[0][i]=Fig7.add_subplot(grid[0,i],sharex=ax7[0][i],sharey=ax7[0][0])
    TRA_Truck.filter(variable=sss(S,F,i)).filter(year=2019,scenario=['ndc-ts']).rename(scenario={"ndc-ts":"2019年"}).plot.bar(stacked =True,y='value',x='scenario',bars='variable',width=0.8,
                                                bars_order=sss(S,F,i),orient='h',legend=False,ax=ax7[0][i],title=False,cmap=sns.color_palette("Set2_r",as_cmap=True))


for j in range (1,4):
    for i in range (0,4):  
        ax7[j][i]=Fig7.add_subplot(grid[(1+2*(j-1)):(1+2*j),i],sharex=ax7[0][i],sharey=ax7[j][0])
        TRA_Truck.filter(variable=sss(S,F,i)).filter(year=Y[j]).rename(scenario={"ndc-ts":"REF","ndc-ts_2c-cn60":"CN60"}).plot.bar(stacked =True,y='value',x='scenario',bars='variable',width=0.8,
                                                bars_order=sss(S,F,i),orient='h',legend=False,ax=ax7[j][i],title=False,cmap=sns.color_palette("Set2_r",as_cmap=True))

ax7[3][0].set_xlabel("大型卡车（十亿吨公里）")
ax7[3][1].set_xlabel("中型卡车（十亿吨公里）")
ax7[3][2].set_xlabel("轻型卡车（十亿吨公里）")
ax7[3][3].set_xlabel("微型卡车（十亿吨公里）")
ax7[0][0].set_ylabel("")
ax7[1][0].set_ylabel("2035年")
ax7[2][0].set_ylabel("2060年")
ax7[3][0].set_ylabel("2100年")
ax7[0][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[1][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[2][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[3][0].yaxis.set_minor_locator(plt.NullLocator())

plt.legend(L,ncol=7,bbox_to_anchor=(0.5, -0.7),frameon=False,fontsize=12,columnspacing=1)
# plt.savefig(ImgPath+"chap3_TRA_Truck"+".pdf",bbox_inches='tight',dpi=300)
plt.show() 

In [ ]:

# Electricity Consumption 2060 Timeslice hourly
FE_t1=Data_subannual.filter(scenario='ndc-ts',variable=["Electricity Consumption*"],year=[2019,2060],timeslice="*H*").convert_unit('PJ','GWh')
FE_t2=Data_subannual.filter(variable=["Electricity Consumption*"],year=2060,timeslice="*H*").convert_unit('PJ','GWh')
L=['农业','建筑-公共','建筑-农村','建筑-城镇','工业-非金属','工业-化工','工业-钢铁','工业-造纸','工业-有色','工业-其他','交通','上游']
Scenario_ts=['ndc-ts',"ndc-ts","ndc-ts_2c-cn60","ndc-ts_2c-cn60-lm"]
S_name=['REF',"REF",'CN60','CN60-LM']
Y=[2019,2060]
Figt1, axt1 = plt.subplots(1, 4, figsize = (13, 6), sharey = True)

for i in range (0,2):
    Figt1=FE_t1.filter(year=Y[i],scenario='ndc-ts').plot.bar(width=1,
        stacked=True,x="timeslice",title=str(Y[i])+'年REF情景',ax = axt1[i],legend=False,
        cmap=sns.color_palette("tab20",as_cmap=True))
    axt1[i].set_xticks(timeshort[::2])
    axt1[i].set_xticklabels(timeshort[::2],rotation=0)
    axt1[i].set_xlabel('小时')
    axt1[i].set_ylabel('用电负荷（GW）')
for i in range (2,4):
    Figt1=FE_t2.filter(year=2060,scenario=Scenario_ts[i]).plot.bar(width=1,
        stacked=True,x="timeslice",title='2060年'+S_name[i]+'情景',ax = axt1[i],legend=False,
        cmap=sns.color_palette("tab20",as_cmap=True))
    axt1[i].set_xticks(timeshort[::2])
    axt1[i].set_xticklabels(timeshort[::2],rotation=0)
    axt1[i].set_xlabel('小时')
    axt1[i].set_ylabel('用电负荷（GW）')
plt.tight_layout()
plt.legend(L,ncol=6,bbox_to_anchor=(0.2,-0.1),frameon=False,fontsize=12,columnspacing=1)
# plt.savefig(ImgPath+"chap3_TS_ELC_Demand_Hourly"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Electricity Generation - 2060 Timeslice hourly
ELC_t1=Data_subannual.filter(variable='Secondary Energy|E*',timeslice="*H*")
ELC_t1=ELC_t1.append(Data_subannual.filter(variable=['Storage|Electricity*'],timeslice="*H*").aggregate(["Storage|Electricity|Charge","Storage|Electricity|Discharge"])).convert_unit('TWh','GWh')
ELC_t1=ELC_t1.append(Data_subannual.filter(variable="Load Management|TRA|V2G*",timeslice="*H*").convert_unit('TWh','GWh'))

ELC_t11=ELC_t1.filter(year=[2019,2060],scenario='ndc-ts',timeslice="*H*")
ELC_t12=ELC_t1.filter(year=2060,timeslice="*H*")
L=['煤','煤CCS','油', '油CCS', '气','气CCS','核电','地热','生物质-煤','生物质-煤CCS','生物质','生物质CCS','水电','海洋能','风能','太阳能','储能放电','储能充电','V2G放电','V2G充电']
ELC_gen_order=['Load Management|TRA|V2G|Charge','Load Management|TRA|V2G|Discharge','Storage|Electricity|Charge','Storage|Electricity|Discharge','Secondary Energy|Electricity|Solar', 'Secondary Energy|Electricity|Wind', 'Secondary Energy|Electricity|Ocean','Secondary Energy|Electricity|Hydro', 'Secondary Energy|Electricity|Biomass|w/ CCS','Secondary Energy|Electricity|Biomass|w/o CCS','Secondary Energy|Electricity|Biomass-Coal|w/ CCS', 'Secondary Energy|Electricity|Biomass-Coal|w/o CCS','Secondary Energy|Electricity|Geothermal','Secondary Energy|Electricity|Nuclear',  'Secondary Energy|Electricity|Gas|w/ CCS', 'Secondary Energy|Electricity|Gas|w/o CCS', 'Secondary Energy|Electricity|Oil|w/ CCS','Secondary Energy|Electricity|Oil|w/o CCS','Secondary Energy|Electricity|Coal|w/ CCS', 'Secondary Energy|Electricity|Coal|w/o CCS']
ELC_gen_order.reverse()
Scenario_ts=['ndc-ts',"ndc-ts","ndc-ts_2c-cn60","ndc-ts_2c-cn60-lm"]
S_name=['REF',"REF",'CN60','CN60-LM']
Y=[2019,2060]
Figt2, axt2 = plt.subplots(1,4, figsize = (13, 6), sharey = True)

for i in range (0,2):
    ELC_t11.filter(year=Y[i],scenario='ndc-ts').plot.bar(width=0.8,bars_order=ELC_gen_order,
        x="timeslice",stacked=True,title=str(Y[i])+'年REF情景',ax = axt2[i],legend=False,cmap=sns.color_palette("tab20c_r",as_cmap=True))
    plt.legend(L,ncol=9,bbox_to_anchor=(0.8, -0.1))
    axt2[i].set_xticks(timeshort[::2])
    axt2[i].set_xticklabels(timeshort[::2],rotation=0)
    axt2[i].set_xlabel('小时')
    axt2[i].set_ylabel('发电负荷（GW）')
    add_net_values_to_bar_plot(axt2[i]) 

for i in range (2,4):
    ELC_t12.filter(year=2060,scenario=Scenario_ts[i]).plot.bar(width=0.8,bars_order=ELC_gen_order,
        x="timeslice",stacked=True,title='2060年'+S_name[i]+'情景',ax = axt2[i],legend=False,cmap=sns.color_palette("tab20c_r",as_cmap=True))
    plt.tight_layout()
    plt.legend(L,ncol=10,bbox_to_anchor=(1, -0.1),frameon=False,fontsize=12,columnspacing=1)
    axt2[i].set_xticks(timeshort[::2])
    axt2[i].set_xticklabels(timeshort[::2],rotation=0)
    axt2[i].set_xlabel('小时')
    axt2[i].set_ylabel('发电负荷（GW）')
    add_net_values_to_bar_plot(axt2[i]) 

# plt.savefig(ImgPath+"chap3_TS_ELC_Supply_Hourly"+".pdf",bbox_inches='tight',dpi=300)
plt.show()



In [ ]:

# # Electricity Consumption 2060 Timeslice normal
# FE_t2=Data_subannual.filter(scenario='ndc-ts_2c-cn60',variable=["Electricity Consumption*"],year=2060).filter(keep=False,timeslice="*H*").convert_unit('PJ','GWh')
# FE_t2=normal_to_24hours_consistent(FE_t2)
# L=['农业','建筑-公共','建筑-农村','建筑-城镇','工业-非金属','工业-化工','工业-钢铁','工业-造纸','工业-有色','工业-其他','交通','上游']
# L.reverse()

# Figt2, axt2 = plt.subplots(1, 1, figsize = (12, 4), sharey = True)

# FE_t2.plot.bar(width=1,stacked=True,x="timeslice",title=False,ax=axt2,cmap=sns.color_palette("tab20",as_cmap=True))
# axt2.xaxis.set_ticks(ticks=np.arange(0,192,24),labels=['   春季周末','  春季工作日','   夏季周末','  夏季工作日','   秋季周末','  秋季工作日','   冬季周末','  冬季工作日'],rotation=0,ha='left')
# axt2.xaxis.set_ticks(ticks=np.arange(0,192,1),minor=True)

# axt2.set_xlabel('小时')
# axt2.set_ylabel('用电负荷（GW）')
# handles,l= axt2.get_legend_handles_labels()
# axt2.legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.5),frameon=False,fontsize=12,columnspacing=1)
# axt2.xaxis.grid(True, which='major')

# plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TS_ELC_Demand_normal"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()


In [ ]:
# # Electricity Generation - 2060 Timeslice normal
# ELC_t1=Data_subannual.filter(variable='Secondary Energy|E*').filter(keep=False,timeslice="*H*")
# ELC_t1=ELC_t1.append(Data_subannual.filter(variable=['Storage|Electricity*']).filter(keep=False,timeslice="*H*").aggregate(["Storage|Electricity|Charge","Storage|Electricity|Discharge"])).convert_unit('TWh','GWh')
# ELC_t1=ELC_t1.append(Data_subannual.filter(variable="Load Management|TRA|V2G*").filter(keep=False,timeslice="*H*").convert_unit('TWh','GWh'))

# ELC_t12=ELC_t1.filter(year=2060,scenario="ndc-ts_2c-cn60").filter(keep=False,timeslice="*H*")

# ELC_t12=normal_to_24hours_consistent(ELC_t12)
# L=['煤','煤CCS','油', '油CCS', '气','气CCS','核电','地热','生物质-煤','生物质-煤CCS','生物质','生物质CCS','水电','海洋能','风能','太阳能','储能放电','储能充电','V2G放电','V2G充电']
# ELC_gen_order=['Load Management|TRA|V2G|Charge','Load Management|TRA|V2G|Discharge','Storage|Electricity|Charge','Storage|Electricity|Discharge','Secondary Energy|Electricity|Solar', 'Secondary Energy|Electricity|Wind', 'Secondary Energy|Electricity|Ocean','Secondary Energy|Electricity|Hydro', 'Secondary Energy|Electricity|Biomass|w/ CCS','Secondary Energy|Electricity|Biomass|w/o CCS','Secondary Energy|Electricity|Biomass-Coal|w/ CCS', 'Secondary Energy|Electricity|Biomass-Coal|w/o CCS','Secondary Energy|Electricity|Geothermal','Secondary Energy|Electricity|Nuclear',  'Secondary Energy|Electricity|Gas|w/ CCS', 'Secondary Energy|Electricity|Gas|w/o CCS', 'Secondary Energy|Electricity|Oil|w/ CCS','Secondary Energy|Electricity|Oil|w/o CCS','Secondary Energy|Electricity|Coal|w/ CCS', 'Secondary Energy|Electricity|Coal|w/o CCS']
# ELC_gen_order.reverse()
# Figt2, axt2 = plt.subplots(1, 1, figsize = (14, 4), sharey = True)

# ELC_t12.plot.bar(bars_order=ELC_gen_order,x="timeslice",title=False,ax=axt2,width=1,stacked=True,cmap=sns.color_palette("tab20c_r",as_cmap=True))
# axt2.xaxis.set_ticks(ticks=np.arange(0,192,24),labels=['   春季周末','  春季工作日','   夏季周末','  夏季工作日','   秋季周末','  秋季工作日','   冬季周末','  冬季工作日'],rotation=0,ha='left')
# axt2.xaxis.set_ticks(ticks=np.arange(0,192,1),minor=True)

# axt2.set_xlabel('小时')
# axt2.set_ylabel('发电负荷（GW）')
# axt2.legend(L,ncol=10,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
# axt2.xaxis.grid(True, which='major')


# plt.savefig(ImgPath+"chap3_TS_ELC_Supply_normal"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:
#Net load curve 2060 timeslice hourly
FE_t3=Data_subannual.filter(variable=["Electricity Consumption*"],timeslice="*H*",scenario=["ndc-ts","ndc-ts_2c-cn60","ndc-ts_2c-cn60-lm"]).convert_unit('PJ','GWh').aggregate('*').rename(variable={'*':"Load"})
ELC_t3=Data_subannual.filter(variable='Secondary Energy|E*',timeslice="*H*",scenario=["ndc-ts","ndc-ts_2c-cn60","ndc-ts_2c-cn60-lm"])
ELC_t3=ELC_t3.append(Data_subannual.filter(variable=['Storage|Electricity*'],timeslice="*H*",scenario=["ndc-ts","ndc-ts_2c-cn60","ndc-ts_2c-cn60-lm"]).aggregate(["Storage|Electricity|Charge","Storage|Electricity|Discharge"])).convert_unit('TWh','GWh')
ELC_t3=ELC_t3.append(Data_subannual.filter(variable="Load Management|TRA|V2G*",timeslice="*H*",scenario=["ndc-ts","ndc-ts_2c-cn60","ndc-ts_2c-cn60-lm"]).convert_unit('TWh','GWh'))

ELC_t3=ELC_t3.filter(timeslice="*H*")
ELC_NET=ELC_t3.append(FE_t3)
ELC_NET.add('Secondary Energy|Electricity|Wind','Secondary Energy|Electricity|Solar','WandS',append=True)
ELC_NET1=ELC_NET.subtract('Load','WandS','Net load').filter(year=[2020,2030,2040,2050,2060,2070,2085,2100]).rename(scenario={'ndc-ts':'REF','ndc-ts_2c-cn60':'CN60','ndc-ts_2c-cn60-lm':'CN60-LM'})
ELC_NET1=ELC_NET1.as_pandas()
ELC_NET1.rename(columns={'year':'时间'},inplace=True)
fig=sns.catplot(data=ELC_NET1,x='timeslice',y='value',hue='时间',legend='full',kind='point',col='scenario',palette='rocket_r',col_order=['REF','CN60','CN60-LM'])
fig.refline(y=0)
fig.set_xticklabels(timeshort,rotation=0)
fig.set_axis_labels("小时", "净负荷（GW）")
fig.despine(right=False,top=False)
fig.set_titles('{col_name}')

# plt.savefig(ImgPath+"chap3_TS_ELC_NET_LOAD"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
FE_t=Data_subannual.filter(year=2060,timeslice="*H*",variable="Storage|Electricity*").convert_unit('TWh','GWh')
ss1=['ndc-ts','ndc-ts_2c-cn60','ndc-ts_2c-cn60-lm']
S_name=["REF",'CN60','CN60-LM']
storage_order=[
 'Storage|Electricity|Charge|CAES',
 'Storage|Electricity|Charge|Pumped Hydro',
 'Storage|Electricity|Charge|Superconducting',
 'Storage|Electricity|Charge|Fly-wheels',
 'Storage|Electricity|Charge|Li-ion',
 'Storage|Electricity|Charge|Flow-cell',
 'Storage|Electricity|Charge|Molten Salt',
 'Storage|Electricity|Charge|Pb-Acid',

 'Storage|Electricity|Discharge|CAES',
 'Storage|Electricity|Discharge|Pumped Hydro',
 'Storage|Electricity|Discharge|Superconducting',
 'Storage|Electricity|Discharge|Fly-wheels',
 'Storage|Electricity|Discharge|Li-ion',
 'Storage|Electricity|Discharge|Flow-cell',
 'Storage|Electricity|Discharge|Molten Salt',
 'Storage|Electricity|Discharge|Pb-Acid',
]

l=['充电-压缩空气','充电-抽水蓄能','充电-超导储能','充电-飞轮储能','充电-锂电池','充电-液流电池','充电-熔盐电池','充电-铅酸电池',
   '放电-压缩空气','放电-抽水蓄能','放电-超导储能','放电-飞轮储能','放电-锂电池','放电-液流电池','放电-熔盐电池','放电-铅酸电池']


N=[4,4,4,4]# number of colors  to extract from each cmap, sum(N)=len(classes)
base_cmaps = ['Reds_r','Greens_r','Blues_r','Purples_r']

n_base = len(base_cmaps)

colors = np.concatenate([plt.get_cmap(name)(np.linspace(0.4,0.85,N[i])) for i,name in zip(range(n_base),base_cmaps)])
cmap = ListedColormap(colors)

Figt2, axt2 = plt.subplots(1, 3, figsize = (15, 4.5), sharey = True)
plt.tight_layout()
for i in range (0,len_scen):
    FE_t.filter(scenario=ss1[i]).plot.bar(width=1,
        x="timeslice",stacked=True,title=S_name[i],ax = axt2[i],legend=False,cmap=cmap,bars_order=storage_order)
    axt2[i].set_xticklabels(timeshort,rotation=0)
    axt2[i].set_xlabel('小时')
    axt2[i].set_ylabel('储能充放电负荷（GW）')
axt2[1].legend(l,ncol=8,loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False,fontsize=12,columnspacing=1)


# plt.savefig(ImgPath+"chap3_TS_ELC_STG"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# STG_Normal=Data_subannual.filter(year=2060,variable="Storage|Electricity*",scenario='ndc-ts_2c-cn60').filter(keep=False,timeslice="*H*").convert_unit('TWh','GWh')
# STG_Normal=normal_to_24hours_consistent(STG_Normal)
# storage_order=[
#  'Storage|Electricity|Charge|CAES',
#  'Storage|Electricity|Charge|Pumped Hydro',
#  'Storage|Electricity|Charge|Superconducting',
#  'Storage|Electricity|Charge|Fly-wheels',
#  'Storage|Electricity|Charge|Li-ion',
#  'Storage|Electricity|Charge|Flow-cell',
#  'Storage|Electricity|Charge|Molten Salt',
#  'Storage|Electricity|Charge|Pb-Acid',

#  'Storage|Electricity|Discharge|CAES',
#  'Storage|Electricity|Discharge|Pumped Hydro',
#  'Storage|Electricity|Discharge|Superconducting',
#  'Storage|Electricity|Discharge|Fly-wheels',
#  'Storage|Electricity|Discharge|Li-ion',
#  'Storage|Electricity|Discharge|Flow-cell',
#  'Storage|Electricity|Discharge|Molten Salt',
#  'Storage|Electricity|Discharge|Pb-Acid',
# ]

# L=['充电-压缩空气','充电-抽水蓄能','充电-超导储能','充电-飞轮储能','充电-锂电池','充电-液流电池','充电-熔盐电池','充电-铅酸电池',
#    '放电-压缩空气','放电-抽水蓄能','放电-超导储能','放电-飞轮储能','放电-锂电池','放电-液流电池','放电-熔盐电池','放电-铅酸电池']
# L.reverse()
# N=[4,4,4,4]# number of colors  to extract from each cmap, sum(N)=len(classes)
# base_cmaps = ['Reds_r','Greens_r','Blues_r','Purples_r']
# n_base = len(base_cmaps)
# colors = np.concatenate([plt.get_cmap(name)(np.linspace(0.4,0.85,N[i])) for i,name in zip(range(n_base),base_cmaps)])
# cmap = ListedColormap(colors)


# Figt2, axt2 = plt.subplots(1, 1, figsize = (12, 4), sharey = True)

# STG_Normal.plot.bar(width=1,x="timeslice",stacked=True,title=False,ax=axt2,cmap=cmap,bars_order=storage_order)
# axt2.xaxis.set_ticks(ticks=np.arange(0,192,24),labels=['   春季周末','  春季工作日','   夏季周末','  夏季工作日','   秋季周末','  秋季工作日','   冬季周末','  冬季工作日'],rotation=0,ha='left')
# axt2.xaxis.set_ticks(ticks=np.arange(0,192,1),minor=True)

# handles,l= axt2.get_legend_handles_labels()
# axt2.legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.5),frameon=False,fontsize=10,columnspacing=1)

# axt2.set_xlabel('小时')
# axt2.set_ylabel('储能充放电负荷（GW）')
# axt2.xaxis.grid(True, which='major')

# plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TS_ELC_STG_normal-CN60"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()


In [ ]:
# STG_Normal=Data_subannual.filter(year=2060,variable="Storage|Electricity*",scenario='ndc-ts_2c-cn60-lm').filter(keep=False,timeslice="*H*").convert_unit('TWh','GWh')
# STG_Normal=normal_to_24hours_consistent(STG_Normal)
# storage_order=[
#  'Storage|Electricity|Charge|CAES',
#  'Storage|Electricity|Charge|Pumped Hydro',
#  'Storage|Electricity|Charge|Superconducting',
#  'Storage|Electricity|Charge|Fly-wheels',
#  'Storage|Electricity|Charge|Li-ion',
#  'Storage|Electricity|Charge|Flow-cell',
#  'Storage|Electricity|Charge|Molten Salt',
#  'Storage|Electricity|Charge|Pb-Acid',

#  'Storage|Electricity|Discharge|CAES',
#  'Storage|Electricity|Discharge|Pumped Hydro',
#  'Storage|Electricity|Discharge|Superconducting',
#  'Storage|Electricity|Discharge|Fly-wheels',
#  'Storage|Electricity|Discharge|Li-ion',
#  'Storage|Electricity|Discharge|Flow-cell',
#  'Storage|Electricity|Discharge|Molten Salt',
#  'Storage|Electricity|Discharge|Pb-Acid',
# ]

# L=['充电-压缩空气','充电-抽水蓄能','充电-超导储能','充电-飞轮储能','充电-锂电池','充电-液流电池','充电-熔盐电池','充电-铅酸电池',
#    '放电-压缩空气','放电-抽水蓄能','放电-超导储能','放电-飞轮储能','放电-锂电池','放电-液流电池','放电-熔盐电池','放电-铅酸电池']
# L.reverse()
# N=[4,4,4,4]# number of colors  to extract from each cmap, sum(N)=len(classes)
# base_cmaps = ['Reds_r','Greens_r','Blues_r','Purples_r']
# n_base = len(base_cmaps)
# colors = np.concatenate([plt.get_cmap(name)(np.linspace(0.4,0.85,N[i])) for i,name in zip(range(n_base),base_cmaps)])
# cmap = ListedColormap(colors)


# Figt2, axt2 = plt.subplots(1, 1, figsize = (12, 4), sharey = True)

# STG_Normal.plot.bar(width=1,x="timeslice",stacked=True,title=False,ax=axt2,cmap=cmap,bars_order=storage_order)
# axt2.xaxis.set_ticks(ticks=np.arange(0,192,24),labels=['   春季周末','  春季工作日','   夏季周末','  夏季工作日','   秋季周末','  秋季工作日','   冬季周末','  冬季工作日'],rotation=0,ha='left')
# axt2.xaxis.set_ticks(ticks=np.arange(0,192,1),minor=True)

# handles,l= axt2.get_legend_handles_labels()
# axt2.legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.5),frameon=False,fontsize=10,columnspacing=1)
# axt2.set_xlabel('小时')
# axt2.set_ylabel('储能充放电负荷（GW）')
# axt2.xaxis.grid(True, which='major')

# plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TS_ELC_STG_normal-CN60-lm"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()


In [ ]:

# Figt2, axt2 = plt.subplots(1, 3, figsize = (15, 5), sharey = True)

# #Load shifting -tra NDC
# FE_t1=Data_subannual.filter(variable=["Load Management|TRA_F","Load Management|TRA_S"],year=2060,scenario='ndc-ts',timeslice="*H*").convert_unit('TWh','GWh')
# FE_t1.plot.bar(width=1,stacked=True,ax=axt2[0],
#         x="timeslice",y='value',bars="variable",title="REF",cmap=sns.color_palette("tab20",5,as_cmap=True))

# axt2[0].set_xticklabels(timeshort,rotation=0)
# axt2[0].legend(["快速充电","慢速充电"],frameon=False,loc='lower left',columnspacing=1)
# axt2[0].set_xlabel('小时')
# axt2[0].set_ylabel('交通部门耗电量（GW）')
# plt.xticks(rotation=0)
# plt.tight_layout()

# #Load shifting -tra CN60
# FE_t1=Data_subannual.filter(variable=["Load Management|TRA_F","Load Management|TRA_S"],year=2060,scenario='ndc-ts_2c-cn60',timeslice="*H*").convert_unit('TWh','GWh')
# FE_t1.plot.bar(width=1,stacked=True,ax=axt2[1],
#         x="timeslice",y='value',bars="variable",title="CN60",cmap=sns.color_palette("tab20",5,as_cmap=True))

# axt2[1].set_xticklabels(timeshort,rotation=0)
# axt2[1].legend(["快速充电","慢速充电"],frameon=False,loc='lower left',columnspacing=1)
# axt2[1].set_xlabel('小时')
# axt2[1].set_ylabel('交通部门耗电量（GW）')
# plt.xticks(rotation=0)
# plt.tight_layout()



# #Load shifting -tra CN60-LM
# FE_t0=Data_subannual.filter(variable="Load Management|TRA|V2G*",year=2060,scenario='ndc-ts_2c-cn60-lm',timeslice="*H*").convert_unit('TWh','GWh')
# FE_t0.multiply("Load Management|TRA|V2G|Charge",-1,'V2G-Charge',append=True)
# FE_t0.multiply("Load Management|TRA|V2G|Discharge",-1,'V2G-Discharge',append=True)
# FE_t1=Data_subannual.filter(variable=["Load Management|TRA_F","Load Management|TRA_S"],year=2060,scenario='ndc-ts_2c-cn60-lm',timeslice="*H*").convert_unit('TWh','GWh')
# FE_t1=FE_t1.append(FE_t0.filter(variable="V2G-*"))
# FE_t2=Data_subannual.filter(variable="Load Management|TRA_Shift*",year=2060,scenario='ndc-ts_2c-cn60-lm',timeslice="*H*").convert_unit('TWh','GWh').aggregate('Load Management|TRA_Shift')
# FE_t2=FE_t2.append(FE_t1)
# FE_t2.plot.bar(width=1,stacked=True,ax=axt2[2],
#         x="timeslice",y='value',bars="variable",title="CN60-LM",cmap=sns.color_palette("tab20",5,as_cmap=True))
# plt.xticks(np.arange(0,24,1),timeshort)
# plt.legend(["快速充电","慢速充电",'负荷时移','V2G储能','V2G放能'],frameon=False,ncol=2,columnspacing=1)
# plt.xlabel('小时')
# plt.ylabel('交通部门耗电量（GW）')
# plt.xticks(rotation=0)
# plt.tight_layout()

# plt.savefig(ImgPath+"chap3_TS_TRA_V2G"+".pdf",bbox_inches='tight',dpi=300)


# plt.show()

In [ ]:
# # V2G tech normal day

# #Load shifting -tra
# FE_t1=Data_subannual.filter(variable=["Load Management|TRA_F","Load Management|TRA_S"],year=2060,scenario='ndc-ts_2c-cn60-lm').filter(keep=False,timeslice="*H*").convert_unit('TWh','GWh')
# FE_t1=normal_to_24hours_consistent_TRA(FE_t1)
# FE_t2=Data_subannual.filter(variable=["Load Management|TRA_Shift*"],year=2060,scenario='ndc-ts_2c-cn60-lm').filter(keep=False,timeslice="*H*").convert_unit('TWh','GWh').aggregate("Load Management|TRA_Shift")
# FE_t2=normal_to_24hours_consistent(FE_t2)
# FE_t0=Data_subannual.filter(variable="Load Management|TRA|V2G*",year=2060,scenario='ndc-ts_2c-cn60-lm').filter(keep=False,timeslice="*H*").convert_unit('TWh','GWh')
# FE_t0.multiply("Load Management|TRA|V2G|Charge",-1,'V2G-Charge',append=True)
# FE_t0.multiply("Load Management|TRA|V2G|Discharge",-1,'V2G-Discharge',append=True)
# FE_t0=FE_t0.filter(variable="V2G-*")
# FE_t0=normal_to_24hours_consistent(FE_t0)

# FE_t=FE_t1.append(FE_t2)
# FE_t=FE_t.append(FE_t0)

# Figt2, axt2 = plt.subplots(1, 1, figsize = (12, 4), sharey = True)
# Figt2=FE_t.plot.bar(width=1,ax=axt2,stacked=True,
#         x="timeslice",y='value',bars="variable",title=None,cmap=cbm.pal("npg").as_cmap)
# axt2.xaxis.set_ticks(ticks=np.arange(0,192,24),labels=['   春季周末','  春季工作日','   夏季周末','  夏季工作日','   秋季周末','  秋季工作日','   冬季周末','  冬季工作日'],rotation=0,ha='left')
# axt2.xaxis.set_ticks(ticks=np.arange(0,192,1),minor=True)
# plt.legend(["快速充电","慢速充电",'负荷时移',"V2G充电","V2G放电"],frameon=False,loc=1,ncol=5,fontsize=12,columnspacing=0.4)
# plt.xlabel('小时')
# plt.ylabel('交通部门耗电量（GW）')
# plt.xticks(rotation=0)

# axt2.xaxis.grid(True, which='major')

# plt.tight_layout()

# plt.savefig(ImgPath+"chap3_TS_TRA_V2G_normal"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:
# Hydrogen Generation - 2060 Timeslice hourly
H2N_GEN=Data_subannual.filter(year=2060,timeslice="*H*",variable='Secondary Energy|Hydrogen*')
H2N_GEN=H2N_GEN.append(Data_subannual.filter(variable="Storage|Hydrogen*",timeslice="*H*",year=2060)).convert_unit('TWh','GWh')
L=['煤','煤CCS','油', '油CCS', '气','气CCS','生物质','生物质CCS','网电','陆上风电','海上风电','太阳能','液氢储能','液氢放能']
H2N_GEN_order=[
    'Storage|Hydrogen|Discharge',
    'Storage|Hydrogen|Charge',
    'Secondary Energy|Hydrogen|Distribute|Solar',
    'Secondary Energy|Hydrogen|Distribute|Onshore Wind',
    'Secondary Energy|Hydrogen|Distribute|Offshore Wind',
    'Secondary Energy|Hydrogen|Electricity',
    'Secondary Energy|Hydrogen|Biomass|w/ CCS',
    'Secondary Energy|Hydrogen|Biomass|w/o CCS',
    'Secondary Energy|Hydrogen|Natural gas|w/ CCS',
    'Secondary Energy|Hydrogen|Natural gas|w/o CCS',
    'Secondary Energy|Hydrogen|Oil|w/ CCS',
    'Secondary Energy|Hydrogen|Oil|w/o CCS',
    'Secondary Energy|Hydrogen|Coal|w/ CCS',
    'Secondary Energy|Hydrogen|Coal|w/o CCS']
H2N_GEN_order.reverse()
L.reverse()
Scenario_ts=["ndc-ts_2c-cn60"]
S_name=['CN60']
Figt2, axt2 = plt.subplots(1, 1, figsize = (6, 5))
H2N_GEN.filter(scenario=Scenario_ts).plot.bar(width=0.8,bars_order=H2N_GEN_order,
        x="timeslice",stacked=True,title=False,ax = axt2,legend=False,cmap=sns.color_palette("tab20c_r",as_cmap=True))
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.5),frameon=False,fontsize=12,columnspacing=1)

axt2.set_xticklabels(timeshort,rotation=0)
axt2.set_xlabel('小时')
axt2.set_ylabel('氢能生产量（GW）')

# plt.savefig(ImgPath+"chap3_TS_H2N_Supply_Hourly"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Hydrogen Generation - 2060 Timeslice hourly
H2N_DEM=Data_subannual.filter(year=2060,timeslice="*H*",variable='Hydrogen Consumption|*').convert_unit('TWh','GWh')
L=['工业部门','交通部门','电力部门', '建筑部门']
L.reverse()
H2N_DEM_order=['Hydrogen Consumption|BLD',
               'Hydrogen Consumption|ELC',
               'Hydrogen Consumption|TRA',    
               'Hydrogen Consumption|IND']
H2N_DEM_order.reverse()
Scenario_ts=["ndc-ts_2c-cn60"]
S_name=['CN60']
Figt2, axt2 = plt.subplots(1, 1, figsize = (6, 5))
H2N_DEM.filter(scenario=Scenario_ts).plot.bar(width=0.8,bars_order=H2N_DEM_order,
        x="timeslice",stacked=True,title=False,ax = axt2,legend=False,cmap=sns.color_palette("tab20c_r",as_cmap=True))
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.5),frameon=False,fontsize=12,columnspacing=1)
axt2.set_xticklabels(timeshort,rotation=0)
axt2.set_xlabel('小时')
axt2.set_ylabel('氢能消费量（GW）')

# plt.savefig(ImgPath+"chap3_TS_H2N_Demand_Hourly"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# H2N Storage energy flow - normal
H2N_STG_FLO=Data_subannual.filter(variable="Storage|Hydrogen*",year=2060,scenario='ndc-ts_2c-cn60').filter(keep=False,timeslice="*H*").convert_unit('TWh','GWh')
H2N_STG_FLO_normal=normal_to_24hours_consistent(H2N_STG_FLO)

Figt2, axt2 = plt.subplots(1, 1, figsize = (12, 4), sharey = True)

H2N_STG_FLO_normal.plot.bar(width=1,x="timeslice",stacked=True,title=False,ax=axt2)
axt2.xaxis.set_ticks(ticks=np.arange(0,192,24),labels=['   春季周末','  春季工作日','   夏季周末','  夏季工作日','   秋季周末','  秋季工作日','   冬季周末','  冬季工作日'],rotation=0,ha='left')
axt2.xaxis.set_ticks(ticks=np.arange(0,192,1),minor=True)

axt2.legend(["液氢储能","液氢放能"],loc='lower left',bbox_to_anchor=(0.365,0),frameon=False)
axt2.set_xlabel('小时')
axt2.set_ylabel('日内液氢储能运行模式（GW）')
axt2.xaxis.grid(True, which='major')

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TS_H2N_STG_normal"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
# Electricity Generation - 2060 Timeslice seasonal
FE_t=Data_season.filter(variable="Secondary Energy|Electricity*",year=2060)
L=['煤','煤CCS','油', '油CCS', '气','气CCS','核电','地热','生物质-煤','生物质-煤CCS','生物质','生物质CCS','水电','海洋能','风能','太阳能','储能放电','储能充电','V2G放电','V2G充电']
L.reverse()
ELC_gen_order=['Load Management|TRA|V2G|Charge','Load Management|TRA|V2G|Discharge','Storage|Electricity|Charge','Storage|Electricity|Discharge','Secondary Energy|Electricity|Solar', 'Secondary Energy|Electricity|Wind', 'Secondary Energy|Electricity|Ocean','Secondary Energy|Electricity|Hydro', 'Secondary Energy|Electricity|Biomass|w/ CCS','Secondary Energy|Electricity|Biomass|w/o CCS','Secondary Energy|Electricity|Biomass-Coal|w/ CCS', 'Secondary Energy|Electricity|Biomass-Coal|w/o CCS','Secondary Energy|Electricity|Geothermal','Secondary Energy|Electricity|Nuclear',  'Secondary Energy|Electricity|Gas|w/ CCS', 'Secondary Energy|Electricity|Gas|w/o CCS', 'Secondary Energy|Electricity|Oil|w/ CCS','Secondary Energy|Electricity|Oil|w/o CCS','Secondary Energy|Electricity|Coal|w/ CCS', 'Secondary Energy|Electricity|Coal|w/o CCS']
ELC_gen_order.reverse()
Scenario_ts=["ndc-ts","ndc-ts_2c-cn60","ndc-ts_2c-cn60-lm"]
S_name=["REF",'CN60','CN60-LM']
FE_t=FE_t.append(Data_season.filter(variable='Storage|Electricity*',year=2060).aggregate(["Storage|Electricity|Charge","Storage|Electricity|Discharge"])).convert_unit('PJ','TWh')
FE_t=FE_t.append(Data_season.filter(variable="Load Management|TRA|V2G*",year=2060))

Figt1, axt1 = plt.subplots(1, 3, figsize = (6, 4.5), sharey = True)
for i in range (0,len_scen):
    Figt1=FE_t.filter(scenario=Scenario_ts[i]).plot.bar(width=0.8,
        stacked=True,x="timeslice",y='value',bars="variable",title=S_name[i],ax = axt1[i],legend=False,bars_order=ELC_gen_order,
        cmap=sns.color_palette("tab20c_r",as_cmap=True))    
    axt1[i].set_xticklabels(['春','夏','秋','冬'],rotation=0)
    axt1[i].set_xlabel('季节')
    axt1[i].set_ylabel('发电负荷（TWh/季度）')
    handles,l= axt1[i].get_legend_handles_labels()
    axt1[2].legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.45),frameon=False,fontsize=10,columnspacing=1)
    add_net_values_to_bar_plot(axt1[i]) 
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TS_ELC_Supply_Season"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Electricity Consumption 2060 Timeslice season
FE_t=Data_season.filter(variable="Electricity Consumption*",year=2060).convert_unit('PJ','TWh')
L=['农业','建筑-公共','建筑-农村','建筑-城镇','供水','工业-非金属','工业-化工','工业-钢铁','工业-造纸','工业-有色','工业-其他','交通','上游']
L.reverse()
Scenario_ts=["ndc-ts","ndc-ts_2c-cn60","ndc-ts_2c-cn60-lm"]
S_name=["REF",'CN60','CN60-LM']
Figt1, axt1 = plt.subplots(1, 3, figsize = (6, 4.5), sharey = True)
for i in range (0,len_scen):
    Figt1=FE_t.filter(scenario=Scenario_ts[i]).plot.bar(width=0.8,
        stacked=True,x="timeslice",title=S_name[i],ax = axt1[i],legend=False,
        cmap=sns.color_palette("tab20",as_cmap=True))
    axt1[i].set_xticklabels(['春','夏','秋','冬'],rotation=0)
    axt1[i].set_xlabel('季节')
    axt1[i].set_ylabel('用电负荷（TWh/季度）')
handles,l= axt1[2].get_legend_handles_labels()
axt1[2].legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.5),frameon=False,fontsize=10,columnspacing=1)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TS_ELC_Demand_Season"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Hydrogen Consumption - 2060 Timeslice season
H2N_DEM=Data_season.filter(year=2060,variable='Hydrogen Consumption|*')
L=['工业部门','交通部门','电力部门', '建筑部门']
L.reverse()
H2N_DEM_order=['Hydrogen Consumption|BLD',
               'Hydrogen Consumption|ELC',
               'Hydrogen Consumption|TRA',    
               'Hydrogen Consumption|IND']
H2N_DEM_order.reverse()
Scenario_ts=["ndc-ts_2c-cn60"]
S_name=['CN60']
Figt2, axt2 = plt.subplots(1, 1, figsize = (1, 4))
H2N_DEM.filter(scenario=Scenario_ts).plot.bar(width=0.8,bars_order=H2N_DEM_order,
        x="timeslice",stacked=True,title=False,ax = axt2,legend=False,cmap=sns.color_palette("tab20c_r",as_cmap=True))
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.5),frameon=False,fontsize=12,columnspacing=1)
axt2.set_xticklabels(['春','夏','秋','冬'],rotation=0)
axt2.set_xlabel('季节')
axt2.set_ylabel('氢能消费（TWh/季度）')

# plt.savefig(ImgPath+"chap3_TS_H2N_Demand_Season"+".pdf",bbox_inches='tight',dpi=300)
plt.show()




In [ ]:
# Hydrogen Generation - 2060 Timeslice seasonal
H2N_GEN_Season=Data_season.filter(variable=['Secondary Energy|Hydrogen*','Storage|Hydrogen*'],year=2060)

L=['煤','煤CCS','油', '油CCS', '气','气CCS','生物质','生物质CCS','网电','陆上风电','海上风电','太阳能','液氢储能','液氢放能']
L.reverse()
H2N_GEN_order=[
    'Storage|Hydrogen|Discharge',
    'Storage|Hydrogen|Charge',
    'Secondary Energy|Hydrogen|Distribute|Solar',
    'Secondary Energy|Hydrogen|Distribute|Onshore Wind',
    'Secondary Energy|Hydrogen|Distribute|Offshore Wind',
    'Secondary Energy|Hydrogen|Electricity',
    'Secondary Energy|Hydrogen|Biomass|w/ CCS',
    'Secondary Energy|Hydrogen|Biomass|w/o CCS',
    'Secondary Energy|Hydrogen|Natural gas|w/ CCS',
    'Secondary Energy|Hydrogen|Natural gas|w/o CCS',
    'Secondary Energy|Hydrogen|Oil|w/ CCS',
    'Secondary Energy|Hydrogen|Oil|w/o CCS',
    'Secondary Energy|Hydrogen|Coal|w/ CCS',
    'Secondary Energy|Hydrogen|Coal|w/o CCS']
H2N_GEN_order.reverse()
Scenario_ts=["ndc-ts_2c-cn60"]
S_name=['CN60']

Figt2,axt2 = plt.subplots(1, 1, figsize = (1, 4))

Figt2=H2N_GEN_Season.filter(scenario=Scenario_ts).plot.bar(width=0.8,
        stacked=True,x="timeslice",y='value',bars="variable",title=False,legend=False,bars_order=H2N_GEN_order,ax=axt2,
        cmap=sns.color_palette("tab20c_r",as_cmap=True))    
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.5),frameon=False,fontsize=12,columnspacing=1)

axt2.set_xticklabels(['春','夏','秋','冬'],rotation=0)
axt2.set_xlabel('季节')
axt2.set_ylabel('氢能生产 （TWh/季度）')
# plt.savefig(ImgPath+"chap3_TS_H2N_Supply_Season"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Hydrogen Storage - 2060 Timeslice seasonal
H2N_STG_FLO_season=Data_season.filter(variable=['Storage|Hydrogen*'],year=2060)

L=['液氢储能','液氢放能']
H2N_STG_FLO_order=[
    'Storage|Hydrogen|Discharge',
    'Storage|Hydrogen|Charge']
H2N_STG_FLO_order.reverse()
Scenario_ts=["ndc-ts_2c-cn60"]
S_name=['CN60']

Figt2,axt2 = plt.subplots(1, 1, figsize = (1, 4))

H2N_STG_FLO_season.filter(scenario=Scenario_ts).plot.bar(width=0.8,
        stacked=True,x="timeslice",y='value',bars="variable",title=False,legend=False,ax=axt2,
        cmap=sns.color_palette("tab20c_r",as_cmap=True))    
axt2.set_xticklabels(['春','夏','秋','冬'],rotation=0)
axt2.set_xlabel('季节')
axt2.set_ylabel('跨季节液氢储能运行模式（TWh/季度）')
plt.legend(L,ncol=1,loc='center left',bbox_to_anchor=(1,0.5),frameon=False,fontsize=12,columnspacing=1)
 

# plt.savefig(ImgPath+"chap3_TS_H2N_STG_Season"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
storage_cap=Data.filter(variable="Capacity|Storage*",year=range(2019,2065)).filter(variable=["Capacity|Storage*-E","Capacity|Storage*-P"],keep=False).filter(variable="*-E*")
ss1=['ndc-ts','ndc-ts_2c-cn60','ndc-ts_2c-cn60-lm']
S_name=["REF",'CN60','CN60-LM']
L=["熔盐电池","铅酸电池","锂电池","液流电池"]
L.reverse()
Figt2, axt2 = plt.subplots(1, 3, figsize = (12, 4), sharey = True)

for i in range (0,3):
    storage_cap.filter(scenario=ss1[i]).filter(year=[2023],keep=False).plot.stack(legend=False,title=S_name[i],ax = axt2[i],cmap=sns.color_palette("Set3",as_cmap=True))
    axt2[i].set_xlabel('时间')
    axt2[i].set_ylabel('电池储能装机容量（GWh）')
    handles,l= axt2[i].get_legend_handles_labels()
    axt2[i].legend(handles=handles,labels=L,loc=2,frameon=False)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TS_ELC_STGCAP(ENERGY)"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
storage_cap=Data.filter(variable="Capacity|Storage*",year=range(2019,2065)).filter(variable=["Capacity|Storage*-E","Capacity|Storage*-P"],keep=False).filter(variable="*-P*")
ss1=['ndc-ts','ndc-ts_2c-cn60','ndc-ts_2c-cn60-lm']
S_name=["REF",'CN60','CN60-LM']
L=["超导储能","抽水蓄能","飞轮储能","压缩空气"]
L.reverse()

Figt2, axt2 = plt.subplots(1, 3, figsize = (12, 4), sharey = True)
plt.tight_layout()
for i in range (0,3):
    storage_cap.filter(scenario=ss1[i]).filter(year=2050,keep=False).plot.stack(legend=True,title=S_name[i],ax = axt2[i],cmap=sns.color_palette("Set2_r",as_cmap=True))
    axt2[i].set_xlabel('时间')
    axt2[i].set_ylabel('机械和电磁储能装机容量（GW）')
    handles,l= axt2[i].get_legend_handles_labels()
    axt2[i].legend(handles=handles,labels=L,loc=2,frameon=False)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TS_ELC_STGCAP(Power)"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
#Total Cost
Cost_tot=Data.filter(variable="Cost|CAPEX*")
Cost_tot.aggregate('Cost|CAPEX|Building*',append=True)
Cost_tot.aggregate('Cost|CAPEX|Transport*',append=True)
Cost_tot.aggregate('Cost|CAPEX|Industry*',append=True)
Cost_tot.aggregate('Cost|CAPEX|Upstream*',append=True)
Cost_tot.aggregate('Cost|CAPEX|Electricity*',append=True)
Cost_tot.aggregate('Cost|CAPEX|Hydrogen*',append=True)
Cost_tot.aggregate('Cost|CAPEX|H2O*',append=True)
Cost_tot.aggregate('Cost|CAPEX|Storage*',append=True)

Cost_tot_order=['Cost|CAPEX|Building*',
                'Cost|CAPEX|Transport*',
                'Cost|CAPEX|Industry*',
                'Cost|CAPEX|Upstream*',
                'Cost|CAPEX|Electricity*',
                'Cost|CAPEX|Storage*',
                'Cost|CAPEX|Hydrogen*',
                'Cost|CAPEX|H2O*']
L=["建筑","交通","工业","上游",'电力','储能','氢能','供水']
L.reverse()

scen=['ndc-ts','ndc-ts_2c-cn60']
S_name=['REF','CN60']

Cost_tot=Cost_tot.filter(variable="Cost|CAPEX*",level=1)
Cost_tot.convert_unit('million USD','万亿元',1/1000000*6.9,inplace=True)
Cost_tot_order.reverse()

Figt2, axt2 = plt.subplots(1, 2, figsize = (7, 3), sharey = True)

for i in range (0,2):
    Cost_tot.filter(scenario=scen[i]).filter(keep=False,year=[2019,2031,2032]).plot.stack(
        legend=False,title=S_name[i],ax = axt2[i],order=Cost_tot_order,
        cmap=sns.color_palette("Set2_r",as_cmap=True))
    axt2[i].set_xlabel('时间')
    axt2[i].set_ylabel('')
axt2[0].set_ylabel('总投资（万亿元）')
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Invest-area"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

Y=[2035,2050,2060,2085,2100]
scen_sdg=['ndc-ts','ndc-ts_2c-cn60']
S_name=['REF','CN60']
Cost_tot_order.reverse()
Figt2, axt2 = plt.subplots(figsize=(4,6.5))
Cost_tot.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True).filter(year=Y).plot.bar(
    stacked=True,bars_order=Cost_tot_order,x='year',width=0.8,legend=False,
ax=axt2,title=False,cmap=sns.color_palette("Set2",as_cmap=True))
axt2.set_xticklabels(Y,rotation=0)
axt2.set_xlabel("时间")
axt2.set_ylabel("总投资增加额（万亿元）")
# handles,l= axt2.get_legend_handles_labels()
# axt2.legend(handles=reversed(handles),labels=L,loc=2,fontsize=12)
add_net_values_to_bar_plot(axt2)

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_INVEST_DELTA"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

L=["建筑","交通","工业","上游",'电力','储能','氢能','供水']

Y=[2035,2050,2060,2085,2100]



Fig6, ax6 = plt.subplots(1, 5, figsize = (7, 2.5))
Fig6=plt.figure(figsize = (7, 2.5))
grid = plt.GridSpec(1,10,wspace=0,hspace=0)

for i in range (0,5):
    ax6[i]=Fig6.add_subplot(grid[0,((i)*2):(2*(i+1))],sharey=ax6[0])
    
for i in range (0,5):
    Cost_tot.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',bars_order=Cost_tot_order,
        title=Y[i],ax = ax6[i],legend=False,cmap=sns.color_palette("Set2",10,as_cmap=True),order=scen_sdg);
    

for j in range (0,5):
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("总投资（万亿元）")
plt.legend(L,ncol=4,bbox_to_anchor=(0.35, -0.15),frameon=False,fontsize=12,columnspacing=1)
# plt.savefig(ImgPath+"chap3_INVEST"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Building Area investment increase
L=['公建新建',"农村新建","城镇新建",'公建改造',"农村改造","城镇改造",'城镇化']
area_order=['Cost|CAPEX|Building|Area|Commercial|New',
            'Cost|CAPEX|Building|Area|Rural|New',
            'Cost|CAPEX|Building|Area|Urban|New',
            'Cost|CAPEX|Building|Area|Commercial|Retrofit', 
            'Cost|CAPEX|Building|Area|Rural|Retrofit', 
            'Cost|CAPEX|Building|Area|Urban|Retrofit',
            'Cost|CAPEX|Building|Area|Urban|Transfer']
Cost_AREA=Data.filter(variable=area_order).filter(keep=False,variable='*Lumpy*').convert_unit('million USD','亿元',1/100*6.9)
Y=[2035,2050,2060,2085,2100]
L.reverse()
Figt2, axt2 = plt.subplots(figsize=(4,6))
Cost_AREA.filter(year=Y).subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True,).plot.bar(stacked=True,bars_order=area_order,width=0.8,
ax=axt2,title=False,cmap=sns.color_palette("tab20c",as_cmap=True))
axt2.set_xticklabels(Y,rotation=0)
axt2.set_xlabel("时间")
axt2.set_ylabel("建筑围护结构投资增加额（亿元）")
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc=1,frameon=False,fontsize=10)

add_net_values_to_bar_plot(axt2)

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_Building_Area"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
#Buidling Technology investment increase

BLD_TECH_order=['Cost|CAPEX|Building|Tech|Electricity',
                'Cost|CAPEX|Building|Tech|Renewables',
                'Cost|CAPEX|Building|Tech|Fossil Fuel']
L=["电力",
   "可再生",
   "化石燃料"]
L.reverse()
Y=[2035,2050,2060,2085,2100]
Cost_Building=Data.filter(variable=BLD_TECH_order).filter(keep=False,variable='*Lumpy*').convert_unit('million USD','亿元',1/100*6.9)
Figt2, axt2 = plt.subplots(figsize=(4,6))
Cost_Building.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True,).filter(year=Y).plot.bar(width=0.8,stacked=True,bars_order=BLD_TECH_order,
ax=axt2,title=False,cmap=sns.color_palette("tab20c",3,as_cmap=True))
axt2.set_xticklabels(Y,rotation=0)
axt2.set_xlabel("时间")
axt2.set_ylabel("建筑用能技术投资增加额（亿元）")
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc=1,frameon=False,fontsize=10)
add_net_values_to_bar_plot(axt2)

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_Building_Tech"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
# Transport freight investment increase

TRA_F_TECH_order=['Cost|CAPEX|Transport|Freight|Electricity',
                  'Cost|CAPEX|Transport|Freight|Hydrogen',
                  'Cost|CAPEX|Transport|Freight|Biomass',
                  'Cost|CAPEX|Transport|Freight|Fossil Fuel']
L=["电力",
   "氢能",
   "生物质",
   "化石燃料"]
L.reverse()
Y=[2035,2050,2060,2085,2100]

TRA_F_TECH=Data.filter(variable=TRA_F_TECH_order).convert_unit('million USD','亿元',1/100*6.9)
Figt2, axt2 = plt.subplots(figsize=(4,6))
TRA_F_TECH.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True).filter(year=Y).plot.bar(stacked=True, bars_order=TRA_F_TECH_order,x='year',width=0.8,
ax=axt2,title=False,cmap=sns.color_palette("Set2",4,as_cmap=True))
axt2.set_xticklabels(Y,rotation=0)
axt2.set_xlabel("时间")
axt2.set_ylabel("货运交通投资增加额（亿元）")
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc=3,frameon=False,ncol=2,columnspacing=0.1,fontsize=10,bbox_to_anchor=(-0.03, -0.02))
add_net_values_to_bar_plot(axt2)

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_Transport_F"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Industry technology investment increase

Cost_IND_order=['Cost|CAPEX|Industry|CCS',
 'Cost|CAPEX|Industry|Hydrogen',
 'Cost|CAPEX|Industry|Others']
L=["CCS","氢能","其他"]
Y=[2035,2050,2060,2085,2100]
L.reverse()
Cost_IND=Data.filter(variable=Cost_IND_order).convert_unit('million USD','亿元',1/100*6.9)
Figt2, axt2 = plt.subplots(figsize=(4,6))
Cost_IND.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True).filter(year=Y).plot.bar(stacked=True,bars_order=Cost_IND_order,x='year',width=0.8,
ax=axt2,title=False,cmap=sns.color_palette("Set2",as_cmap=True))
axt2.set_xticklabels(Y,rotation=0)
axt2.set_xlabel("时间")
axt2.set_ylabel("工业部门用能技术投资增加额（亿元）")
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc=4,frameon=False,fontsize=10,columnspacing=1)
add_net_values_to_bar_plot(axt2)

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_IND"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Storage facility investment increase

STORAGE_order=['Cost|CAPEX|Storage|Electricity-E|Flow',
 'Cost|CAPEX|Storage|Electricity-E|Li-ion',
 'Cost|CAPEX|Storage|Electricity-E|PBAC',
 'Cost|CAPEX|Storage|Electricity-E|Salt',
 'Cost|CAPEX|Storage|Electricity-P|PSHP',
 'Cost|CAPEX|Storage|Electricity-P|CAES',
 'Cost|CAPEX|Storage|Electricity-P|FLYW',
 
 'Cost|CAPEX|Storage|Electricity-P|SMES',
 'Cost|CAPEX|Storage|Hydrogen|P2X',]
L=["液流电池","锂电池","铅蓄电池","熔盐电池",
   "抽水蓄能","压缩空气","飞轮储能","超导储能",'氢储能']
Y=[2035,2050,2060,2085,2100]
L.reverse()
STORAGE=Data.filter(variable=STORAGE_order).convert_unit('million USD','亿元',1/100*6.9)
Figt2, axt2 = plt.subplots(figsize=(4,6))
STORAGE.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True).filter(year=Y).plot.bar(stacked=True,bars_order=STORAGE_order,x='year',width=0.8,
ax=axt2,title=False,cmap=sns.color_palette("tab20c_r",as_cmap=True))
axt2.set_xticklabels(Y,rotation=0)
axt2.set_xlabel("时间")
axt2.set_ylabel("储能设施投资增加额（亿元）")
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc=2,frameon=False,fontsize=10,labelspacing=0.2)
add_net_values_to_bar_plot(axt2)

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_Storage"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Hydrogen production investment increase

Cost_H2N_order=['Cost|CAPEX|Hydrogen|Biomass',
                  'Cost|CAPEX|Hydrogen|Biomass CCS',
                  'Cost|CAPEX|Hydrogen|Electricity-ALK',
                  'Cost|CAPEX|Hydrogen|Electricity-PEM',
                  'Cost|CAPEX|Hydrogen|Electricity-SOE',
                  'Cost|CAPEX|Hydrogen|Offshore Wind',
                  'Cost|CAPEX|Hydrogen|Coal',
                  'Cost|CAPEX|Hydrogen|Coal CCS',
                  'Cost|CAPEX|Hydrogen|Onshore Wind',
                  'Cost|CAPEX|Hydrogen|Solar',
                  'Cost|CAPEX|Hydrogen|Natural Gas',
                  'Cost|CAPEX|Hydrogen|Natural Gas CCS',
                  'Cost|CAPEX|Hydrogen|Oil',
                  'Cost|CAPEX|Hydrogen|Oil CCS']

Cost_H2N=Data.filter(variable=Cost_H2N_order).convert_unit('million USD','亿元',1/100*6.9)
A1=['Cost|CAPEX|Hydrogen|Electricity-ALK', 'Cost|CAPEX|Hydrogen|Electricity-PEM','Cost|CAPEX|Hydrogen|Electricity-SOE']
A2=['Cost|CAPEX|Hydrogen|Offshore Wind','Cost|CAPEX|Hydrogen|Onshore Wind','Cost|CAPEX|Hydrogen|Solar']
A3=['Cost|CAPEX|Hydrogen|Coal','Cost|CAPEX|Hydrogen|Oil','Cost|CAPEX|Hydrogen|Natural Gas']
A4=['Cost|CAPEX|Hydrogen|Coal CCS','Cost|CAPEX|Hydrogen|Oil CCS','Cost|CAPEX|Hydrogen|Natural Gas CCS']

Cost_H2N.aggregate("Cost|CAPEX|Hydrogen|Electricity*",A1,append=True)
Cost_H2N.aggregate("Cost|CAPEX|Hydrogen|Renewable",A2,append=True)
# Cost_H2N.aggregate("Cost|CAPEX|Hydrogen|Fossil",A3,append=True)
# Cost_H2N.aggregate("Cost|CAPEX|Hydrogen|FossilCCS",A4,append=True)
Cost_H2N_order=['Cost|CAPEX|Hydrogen|Biomass',
                'Cost|CAPEX|Hydrogen|Biomass CCS',
               "Cost|CAPEX|Hydrogen|Electricity*",
               "Cost|CAPEX|Hydrogen|Renewable"]

L=["生物质",
   "生物质CCS",
   "网电",
   "风电光伏"]
L.reverse()
Y=[2035,2050,2060,2085,2100]

Figt2, axt2 = plt.subplots(figsize=(4,6))
Cost_H2N.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True).filter(year=Y).plot.bar(stacked=True,bars_order=Cost_H2N_order,x='year',width=0.8,
ax=axt2,title=False,cmap=sns.color_palette("Set2_r",as_cmap=True))
axt2.set_xticklabels(Y,rotation=0)
axt2.set_xlabel("时间")
axt2.set_ylabel("网电制氢和绿氢生产投资增加额（亿元）")
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc=2,frameon=False,fontsize=10,columnspacing=1)
add_net_values_to_bar_plot(axt2)

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_H2N"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Electricity production investment increase

Cost_ELC_order=['Cost|CAPEX|Electricity|Coal|w/o CCS',
                'Cost|CAPEX|Electricity|Oil|w/o CCS',               
                'Cost|CAPEX|Electricity|Gas|w/o CCS',
                'Cost|CAPEX|Electricity|Coal|w/ CCS',
                'Cost|CAPEX|Electricity|Oil|w/ CCS',
                'Cost|CAPEX|Electricity|Gas|w/ CCS',               
                'Cost|CAPEX|Electricity|Biomass-Coal|w/o CCS',
                'Cost|CAPEX|Electricity|Biomass-Coal|w/ CCS',
                'Cost|CAPEX|Electricity|Biomass|w/o CCS',
                'Cost|CAPEX|Electricity|Biomass|w/ CCS',            
                'Cost|CAPEX|Electricity|Nuclear',
                'Cost|CAPEX|Electricity|Geothermal',
                'Cost|CAPEX|Electricity|Hydro',
                'Cost|CAPEX|Electricity|Ocean',
                'Cost|CAPEX|Electricity|Wind',
                'Cost|CAPEX|Electricity|Solar']
A1=['Cost|CAPEX|Electricity|Coal|w/o CCS',
                'Cost|CAPEX|Electricity|Oil|w/o CCS',               
                'Cost|CAPEX|Electricity|Gas|w/o CCS']
A2=['Cost|CAPEX|Electricity|Coal|w/ CCS',
                'Cost|CAPEX|Electricity|Oil|w/ CCS',
                'Cost|CAPEX|Electricity|Gas|w/ CCS']
A3=['Cost|CAPEX|Electricity|Biomass-Coal|w/o CCS','Cost|CAPEX|Electricity|Biomass|w/o CCS']
A4=['Cost|CAPEX|Electricity|Biomass-Coal|w/ CCS','Cost|CAPEX|Electricity|Biomass|w/ CCS']
A5=['Cost|CAPEX|Electricity|Geothermal','Cost|CAPEX|Electricity|Hydro','Cost|CAPEX|Electricity|Ocean']
Cost_ELC=Data.filter(variable=Cost_ELC_order).convert_unit('million USD','亿元',1/100*6.9)
Cost_ELC.aggregate("Cost|CAPEX|Electricity|Fossil",A1,append=True)
Cost_ELC.aggregate("Cost|CAPEX|Electricity|FossilCCS",A2,append=True)
Cost_ELC.aggregate("Cost|CAPEX|Electricity|Biomass",A3,append=True)
Cost_ELC.aggregate("Cost|CAPEX|Electricity|BiomassCCS",A4,append=True)
Cost_ELC.aggregate("Cost|CAPEX|Electricity|Renewable",A5,append=True)
Cost_ELC_order=['Cost|CAPEX|Electricity|Wind',
                'Cost|CAPEX|Electricity|Solar',
                "Cost|CAPEX|Electricity|Renewable",
                'Cost|CAPEX|Electricity|Nuclear',
                'Cost|CAPEX|Electricity|Biomass',
                'Cost|CAPEX|Electricity|BiomassCCS',
               "Cost|CAPEX|Electricity|Fossil",
               "Cost|CAPEX|Electricity|FossilCCS"]


L=['风能','太阳能','其他可再生','核电','生物质','生物质CCS','化石燃料','化石燃料CCS']
Y=[2035,2050,2060,2085,2100]
L.reverse()

Figt2, axt2 = plt.subplots(figsize=(4,6))
Cost_ELC.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True).filter(year=Y).plot.bar(width=0.8,stacked=True,bars_order=Cost_ELC_order,x='year',
ax=axt2,title=False,cmap=sns.color_palette("RdYlGn_r",as_cmap=True))
axt2.set_xticklabels(Y,rotation=0)
axt2.set_xlabel("时间")
axt2.set_ylabel("电力生产投资增加额（亿元）")
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc=2,frameon=False,fontsize=10,bbox_to_anchor=(-0.02, 1.025),labelspacing=0)
add_net_values_to_bar_plot(axt2)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_ELC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Upstream technology investment increase

Cost_UPS_order=[
    'Cost|CAPEX|Upstream|DAC', 
    'Cost|CAPEX|Upstream|Fossil Fuel CCS',
    'Cost|CAPEX|Upstream|Fossil Fuel',
    'Cost|CAPEX|Upstream|Biomass',]
L=["直接空气捕集","化石燃料CCS","化石燃料","生物质"]
Y=[2035,2050,2060,2085,2100]
L.reverse()
Cost_UPS=Data.filter(variable=Cost_UPS_order).convert_unit('million USD','亿元',1/100*6.9)
Figt2, axt2 = plt.subplots(figsize=(4,6))
Cost_UPS.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True).filter(year=Y).plot.bar(stacked=True,bars_order=Cost_UPS_order,x='year',width=0.8,
ax=axt2,title=False,cmap=sns.color_palette("tab20c_r",as_cmap=True))
axt2.set_xticklabels(Y,rotation=0)
axt2.set_xlabel("时间")
axt2.set_ylabel("燃料加工转换部门和直接空气捕集投资增加额（亿元）")
handles,l= axt2.get_legend_handles_labels()
axt2.legend(handles=reversed(handles),labels=L,loc=3,frameon=False,fontsize=10,columnspacing=1,)
add_net_values_to_bar_plot(axt2)

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_UPS"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# # Transport passenger investment increase

# TRA_P_TECH_order=['Cost|CAPEX|Transport|Passenger|Electricity',
#                   'Cost|CAPEX|Transport|Passenger|Hydrogen',
#                   'Cost|CAPEX|Transport|Passenger|Biomass',
#                   'Cost|CAPEX|Transport|Passenger|Fossil Fuel']
# L=["电力",
#    "氢能",
#    "生物质",
#    "化石燃料"]
# L.reverse()
# Y=[2035,2050,2060,2085,2100]

# TRA_P_TECH=Data.filter(variable=TRA_P_TECH_order).convert_unit('million USD','亿元',1/100*6.9)
# Figt2, axt2 = plt.subplots(figsize=(4,6))
# TRA_P_TECH.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True).filter(year=Y).plot.bar(stacked=True,bars_order=TRA_P_TECH_order,x='year',width=0.8,
# ax=axt2,title=False,cmap=sns.color_palette("tab20c",4,as_cmap=True))
# axt2.set_xticklabels(Y,rotation=0)
# axt2.set_xlabel("时间")
# axt2.set_ylabel("客运交通投资增加额（亿元）")
# handles,l= axt2.get_legend_handles_labels()
# axt2.legend(handles=reversed(handles),labels=L,loc=1,frameon=False,fontsize=10)
# add_net_values_to_bar_plot(axt2)

# plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_Transport_P"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:
# # H2O investment increase

# Cost_UPS_order=[
#                 'Cost|CAPEX|H2O|Groundwater',
#                 'Cost|CAPEX|H2O|Surfacewater',
#                 'Cost|CAPEX|H2O|Desalination',
#                 'Cost|CAPEX|H2O|Waste treatment']
# L=["地下水","地表水","海水淡化","污水处理"]
# Y=[2035,2050,2060,2085,2100]

# Cost_UPS=Data.filter(variable=Cost_UPS_order).convert_unit('million USD','亿元',1/100*6.9)
# Figt2, axt2 = plt.subplots(figsize=(4,6))
# Cost_UPS.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True).filter(year=Y).plot.bar(stacked=True,bars_order=Cost_UPS_order,x='year',width=0.8,
# ax=axt2,title=False,cmap=sns.color_palette("Set2",as_cmap=True))
# axt2.set_xticklabels(Y,rotation=0)
# axt2.set_xlabel("时间")
# axt2.set_ylabel("水资源供应部门投资增加额（亿元）")
# handles,l= axt2.get_legend_handles_labels()
# axt2.legend(handles=handles,labels=L,loc=3,frameon=False,fontsize=10,columnspacing=1)
# add_net_values_to_bar_plot(axt2)

# plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_H2O"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:
Area1=Cost_AREA.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True,)
Area2=Cost_AREA.subtract('ndc-ts_2c-cn60-noCF','ndc-ts','CN60-noCF',axis='scenario',ignore_units=True,)
Area3=Cost_AREA.subtract('ndc-ts_2c-cn60-lm','ndc-ts','CN60-LM',axis='scenario',ignore_units=True,)

BLD1=Cost_Building.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True,)
BLD2=Cost_Building.subtract('ndc-ts_2c-cn60-noCF','ndc-ts','CN60-noCF',axis='scenario',ignore_units=True,)
BLD3=Cost_Building.subtract('ndc-ts_2c-cn60-lm','ndc-ts','CN60-LM',axis='scenario',ignore_units=True,)

STG1=STORAGE.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True)
STG2=STORAGE.subtract('ndc-ts_2c-cn60-noCF','ndc-ts','CN60-noCF',axis='scenario',ignore_units=True)
STG3=STORAGE.subtract('ndc-ts_2c-cn60-lm','ndc-ts','CN60-LM',axis='scenario',ignore_units=True)

ELC1=Cost_ELC.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True)
ELC2=Cost_ELC.subtract('ndc-ts_2c-cn60-noCF','ndc-ts','CN60-noCF',axis='scenario',ignore_units=True)
ELC3=Cost_ELC.subtract('ndc-ts_2c-cn60-lm','ndc-ts','CN60-LM',axis='scenario',ignore_units=True)

CAL=Area1.append(Area2).append(Area3).append(BLD1).append(BLD2).append(BLD3).append(STG1).append(STG2).append(STG3).append(ELC1).append(ELC2).append(ELC3)
CAL.aggregate('Cost|CAPEX|Building|Area*',append=True)
CAL.aggregate('Cost|CAPEX|Building|Tech*',append=True)
CAL.aggregate('Cost|CAPEX|Electricity*',append=True)
CAL.aggregate('Cost|CAPEX|Storage*',append=True)

V=['Cost|CAPEX|Building|Area*','Cost|CAPEX|Building|Tech*','Cost|CAPEX|Electricity*','Cost|CAPEX|Storage*']
S=['CN60','CN60-noCF','CN60-LM']
# LM and noCF
Y=[2035,2050,2060,2100]
Fig7, ax7 = plt.subplots(4, 4, figsize = (12, 3.5))
Fig7=plt.figure(figsize = (12, 3.5))
grid = plt.GridSpec(4,4,wspace=0.1,hspace=0)

for j in range (0,4):
    for i in range (0,4):  
        ax7[j][i]=Fig7.add_subplot(grid[((j)):((j+1)),i],sharex=ax7[0][i],sharey=ax7[j][0])
        CAL.filter(variable=V).filter(variable=V[i],year=Y[j]).plot.bar(y='value',x='variable',bars='scenario',bars_order=['CN60','CN60-noCF','CN60-LM'],width=0.9,
                                                                            orient='h',legend=False,ax=ax7[j][i],title=False,cmap=sns.color_palette("Set2",as_cmap=True))

ax7[3][0].set_xlabel("建筑围护结构（亿元）")
ax7[3][1].set_xlabel("建筑用能技术（亿元）")
ax7[3][2].set_xlabel("电源机组建设（亿元）")
ax7[3][3].set_xlabel("储能设施建设（亿元）")
ax7[0][0].set_ylabel("2035年")
ax7[1][0].set_ylabel("2050年")
ax7[2][0].set_ylabel("2060年")
ax7[3][0].set_ylabel("2100年")
ax7[0][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[1][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[2][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[3][0].yaxis.set_minor_locator(plt.NullLocator())
ax7[0][0].yaxis.set_major_locator(plt.NullLocator())
ax7[3][0].yaxis.set_major_locator(plt.NullLocator())
ax7[1][0].yaxis.set_major_locator(plt.NullLocator())
ax7[2][0].yaxis.set_major_locator(plt.NullLocator())
ax7[0][0].set_yticklabels("")
ax7[1][0].set_yticklabels("")
ax7[2][0].set_yticklabels("")
ax7[3][0].set_yticklabels("")
ax7[0][0].tick_params(left=False,right=False)
ax7[1][0].tick_params(left=False,right=False)
ax7[2][0].tick_params(left=False,right=False)
ax7[0][1].tick_params(left=False,right=False)
ax7[1][1].tick_params(left=False,right=False)
ax7[2][1].tick_params(left=False,right=False)
ax7[0][2].tick_params(left=False,right=False)
ax7[1][2].tick_params(left=False,right=False)
ax7[2][2].tick_params(left=False,right=False)
ax7[0][3].tick_params(left=False,right=False)
ax7[1][3].tick_params(left=False,right=False)
ax7[2][3].tick_params(left=False,right=False)
plt.legend(ncol=4,bbox_to_anchor=(-0.2, -0.65),frameon=False,fontsize=12,columnspacing=1)
# plt.savefig(ImgPath+"chap3_CostnoCF"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
#MAC

scen_sdg=['ndc-ts_2c-cn60','ndc-ts_2c-cn60-lm','ndc-ts_2c-cn60-noCF']
S_name=['CN60','CN60-LM','CN60-noCF']
Y=[2035,2060,2085,2100]
mac=Data.filter(variable="Cost|Marginal|CO2")

Fig6, ax6 = plt.subplots(1, 4, figsize = (20, 4))
Fig6=plt.figure(figsize = (20, 4))
grid = plt.GridSpec(1,18,wspace=0,hspace=0)

for i in range (0,4):
    ax6[i]=Fig6.add_subplot(grid[0,((i)*3):(3*(i+1))],sharey=ax6[0])
    
for i in range (0,4):
    mac.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',
        title=Y[i],ax = ax6[i],legend=False,cmap=sns.color_palette("Paired",10,as_cmap=True),order=scen_sdg);
    

for j in range (0,4):
    ax6[j].set_xticklabels(S_name,rotation=0)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("边际减排成本（美元/吨CO$_2$）")

# plt.savefig(ImgPath+"chap3_MAC-CN60"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# scen_sdg=['ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
# S_name=['REF-SDG','CN60','CN60-SDG']
# Y=[2035,2060,2085,2100]
# Fig6, ax6 = plt.subplots(1, 6, figsize = (20, 4))
# Fig6=plt.figure(figsize = (20, 4))
# grid = plt.GridSpec(1,12,wspace=0,hspace=0)

# for i in range (0,4):
#     ax6[i]=Fig6.add_subplot(grid[0,((i)*2):(2*(i+1))],sharey=ax6[0])
    
# for i in range (0,4):
#     mac.filter(scenario=scen_sdg).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',
#         title=Y[i],ax = ax6[i],legend=False,cmap=sns.color_palette("Paired",10,as_cmap=True),order=scen_sdg);
    

# for j in range (0,4):
#     ax6[j].set_xticklabels(S_name,rotation=0)
#     ax6[j].set_xlabel("")
#     ax6[j].set_ylabel("边际减排成本（美元/吨CO2）")

# plt.savefig(ImgPath+"chap3_MAC"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()


In [ ]:
#total discounted cost

Cost_order=['Cost|Total|CAPEX','Cost|Total|Fixed OMEX','Cost|Total|Variable OMEX','Cost|Total|Elasticity']
L=["初期投资成本","固定运维成本","可变运维成本","福利损失"]

scen_sdg=["ndc-ts",'ndc-ts_2c-cn60','ndc-ts_2c-cn60-lm','ndc-ts_2c-cn60-noCF']
S_name=['REF','CN60','CN60-LM','CN60-noCF']
Figt2, axt2 = plt.subplots(figsize = (3,4))
Data.filter(variable=Cost_order,year=2019).convert_unit('million USD','万亿元',1/1000000*6.9).plot.bar(width=0.8,stacked=True,x='scenario',bars_order=Cost_order,legend=False,
                                                                                                         order=scen_sdg,ax = axt2,title=False,cmap=sns.color_palette("Set2",4,as_cmap=True))
axt2.set_xticklabels(S_name,rotation=45)
axt2.set_xlabel("")
axt2.set_ylabel("折现系统总成本（万亿元）")
# plt.legend(L,ncol=4,bbox_to_anchor=(1, -0.05)) 
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_Total"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
# Cost annual
Cost_annual=Data.filter(variable='Cost|Annual*').convert_unit('million USD','万亿元',1/1000000*6.9)
Cost_annual_order=['Cost|Annual|CAPEX','Cost|Annual|Fixed OMEX','Cost|Annual|Variable OMEX','Cost|Annual|Elasticity']
L=["初期投资成本","固定运维成本","可变运维成本","福利损失"]
Cost_annual_order.reverse()

ss1=["ndc-ts",'ndc-ts_2c-cn60','ndc-ts_2c-cn60-lm','ndc-ts_2c-cn60-noCF']
S_name=['REF','CN60','CN60-LM','CN60-noCF']

Figt2, axt2 = plt.subplots(1, 4, figsize = (12, 4), sharey = True)

for i in range (0,4):
    Cost_annual.filter(scenario=ss1[i]).filter(keep=False,year=[2019,2031,2032]).plot.stack(legend=False,title=S_name[i],ax = axt2[i],order=Cost_annual_order,cmap=sns.color_palette("Set2_r",as_cmap=True))
    axt2[i].set_xlabel('时间')
    axt2[i].set_ylabel('')
axt2[0].set_ylabel('系统总成本（万亿元）')
plt.tight_layout()
plt.legend(L,ncol=1,loc=4,fontsize=12,columnspacing=1,bbox_to_anchor=(0.98, 0.05))
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_Annual"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
#Cost total waterfall
Cost_order=['Cost|Total|CAPEX','Cost|Total|Fixed OMEX','Cost|Total|Variable OMEX','Cost|Total|Elasticity']
L=["初期投资成本","固定运维成本","可变运维成本","福利损失"]

Cost_Total=Data.filter(scenario=['ndc-ts','ndc-ts_2c-cn60'],variable=Cost_order,year=2019).convert_unit('million USD','万亿元',1/1000000*6.9)
Cost_Total.aggregate('*',append=True)
Cost_Total.subtract('ndc-ts_2c-cn60','ndc-ts','delta',axis='scenario',ignore_units=True,append=True)
T=Cost_Total.filter(variable="*").filter(scenario=['ndc-ts','ndc-ts_2c-cn60']).as_pandas().value
T=list(T)
M=Cost_Total.filter(variable="*",keep=False).filter(scenario='delta').as_pandas().value
M=list(M)

C=[0,0,0,0,0]
C[0]=T[0]
C[1]=T[0]+M[0]
C[2]=C[1]+M[2]
C[3]=C[2]+M[3]
C[4]=C[3]+M[1]
Fig6, ax6 = plt.subplots(1, 1, figsize = (5,5))
# 标签
metric_name = '折现系统总成本（万亿元）'
step_names = ['REF', '投资支出', '固定运维', '可变运维', '福利损失']
last_step_label = 'CN60'

# 风格


# 绘制瀑布
waterfall = WaterfallChart(
    C, 
    step_names=step_names, 
    metric_name=metric_name, 
    last_step_label=last_step_label,
    
)
wf_ax = waterfall.plot_waterfall(
    title="",
    ax=ax6
)
plt.xticks(rotation=30)
plt.ylim(1030,1100)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_Total-waterfall"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Upstream technology investment increase

Cost_UPS_order=['Cost|Fuel|Coal|Domestic','Cost|Fuel|Oil|Domestic','Cost|Fuel|Natural Gas|Domestic',
                'Cost|Fuel|Coal|Import','Cost|Fuel|Oil|Import','Cost|Fuel|Natural Gas|Import','Cost|Fuel|LPG|Import' ]
L=["煤炭开采","石油开采","天然气开采","煤炭进口","石油进口","天然气进口",'LPG进口']
Y=[2035,2040,2045,2050,2060,2100]

Cost_UPS=Data.filter(variable=Cost_UPS_order).convert_unit('million USD','万亿元',1/1000000*6.9)
Figt2, axt2 = plt.subplots(figsize=(5,5))
Cost_UPS.subtract('ndc-ts_2c-cn60','ndc-ts','CN60',axis='scenario',ignore_units=True).filter(year=Y).plot.bar(stacked=True,bars_order=Cost_UPS_order,x='year',width=0.8,
ax=axt2,title=False,cmap=sns.color_palette("tab20c",as_cmap=True))
axt2.set_xticklabels(Y,rotation=0)
axt2.set_xlabel("时间")
axt2.set_ylabel("燃料成本增加额（万亿元）")
plt.legend(L,ncol=1,loc=3,fontsize=12,labelspacing=0.1)
add_net_values_to_bar_plot(axt2)

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_Fuel"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
#Walfare loss

scen_sdg=['ndc-ts_2c-cn60']
S_name=['CN60']
Y=[2035,2040,2045,2050,2055,2060,2070,2085,2100]
Figt2, axt2 = plt.subplots(figsize=(5,5))
Data.filter(variable='Cost|Annual|Elasticity',scenario=scen_sdg,year=Y).convert_unit('million USD','万亿元',1/1000000*6.9).plot.bar(x='year',width=0.8,legend=False,
ax=axt2,title=False,cmap=sns.color_palette("Set2",as_cmap=True))

axt2.set_xticklabels(Y,rotation=0)
axt2.set_xlabel("时间")
axt2.set_ylabel("福利损失（万亿元）")
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_Cost_Walfare"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
Y=[2035,2060,2100]
scen_sdg=['ndc-ts_2c-cn60','ndc-ts_2c-cn60-lm']
S_name=['CN60','CN60-LM']
ELC_Price=Data_subannual.filter(year=Y,variable='Cost|Marginal|Electricity',timeslice='*H*').filter(keep=False,year=2100,scenario='ndc-ts_2c-cn60',timeslice='SHdH21').convert_unit('Million USD/PJ','元/千瓦时',1/1000*3.6*6.9)
Figt2, axt2 = plt.subplots(1, 2, figsize = (11, 5), sharey = True,)

for i in range (0,2):
    ELC_Price.filter(scenario=scen_sdg[i]).plot(x="timeslice",ax = axt2[i],title=S_name[i],linewidth=3)
    axt2[i].xaxis.set_ticks(np.arange(0,24,1),timeshort)
    axt2[i].set_xlabel('小时')
    axt2[i].set_ylabel('边际供电成本（元/千瓦时）')
    axt2[i].legend(frameon=False)
plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TS_Price_ELC"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
Y=[2035,2060,2100]
scen_sdg=['ndc-ts_2c-cn60','ndc-ts_2c-cn60-lm']
S_name=['CN60','CN60-LM']
ELC_Price=Data_subannual.filter(year=Y,variable='Cost|Marginal|Hydrogen',timeslice='*H*').convert_unit('Million USD/PJ','元/千克',1/0.7025/10*6.9)
Figt2, axt2 = plt.subplots(1, 2, figsize = (11, 5), sharey = True,)

for i in range (0,2):
    ELC_Price.filter(scenario=scen_sdg[i]).plot(x="timeslice",ax = axt2[i],title=S_name[i],linewidth=3)
    axt2[i].xaxis.set_ticks(np.arange(0,24,1),timeshort)
    axt2[i].set_xlabel('小时')
    axt2[i].set_ylabel('边际供氢成本（元/千克）')
    axt2[i].legend(frameon=False)

plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TS_Price_H2N"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Y=[2035,2060,2100]
# scen_sdg=['ndc-ts','ndc-ts_2c-cn60','ndc-ts_2c-cn60-lm']
# S_name=['REF','CN60','CN60-LM']
# ELC_Price=Data_subannual.filter(year=Y,variable='Cost|Marginal|Electricity').filter(keep=False,timeslice='*H*').convert_unit('Million USD/PJ','元/千瓦时',1/1000*3.6*6.9)
# ELC_Price=normal_to_24hours_marginal(ELC_Price)
# Figt2, axt2 = plt.subplots(1, 1, figsize = (12, 4), sharey = True,)
# ELC_Price.filter(scenario=scen_sdg[1],year=Y[1]).plot.bar(width=1,x="timeslice",title=False,ax=axt2,legend=False)
# axt2.xaxis.set_ticks(ticks=np.arange(0,192,24),labels=['   春季周末','  春季工作日','   夏季周末','  夏季工作日','   秋季周末','  秋季工作日','   冬季周末','  冬季工作日'],rotation=0,ha='left')
# axt2.set_xlabel('小时')
# axt2.set_ylabel('边际电价（元/千瓦时）')

# axt2.legend(L,ncol=1,bbox_to_anchor=(1 ,1))
# plt.tight_layout()
# plt.savefig(ImgPath+"chap3_TS_ELC_Demand_normal"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()


In [ ]:
# LUCF total
LUCF=Data.filter(variable="Land-use*")
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
Y=[2020,2035,2060,2100]
L=['能源作物','耕地','森林','管理林','非管理林','植树造林']
LUCF_order=['Land-use|Crop|Energy Crop',
 'Land-use|Crop',
 'Land-use|Forest',
 'Land-use|Forest|Managed Forest Area',
'Land-use|Forest|Unmanaged Forest Area',
'Land-use|Forest|Afforested and Reforested']
g=LUCF.filter(year=Y,variable=LUCF_order).as_pandas()
g.replace(LUCF_order,L,inplace=True)
g.replace(scen_sdg,S_name,inplace=True)
g.rename(columns={'scenario':'情景'},inplace=True)

#土地面积
fig=sns.catplot(g,kind='bar',col='year',x='variable',y='value',hue='情景',palette="YlGn",height=4,aspect=1.35,sharex=False,sharey=False,dodge=True,order=L,hue_order=S_name,col_wrap=2,legend=S_name)
fig.refline(y=120,linestyle=':')
fig.set_ylabels("面积（万平方千米）")
fig.set_xlabels("")
fig.set_titles("{col_name}",size=20)
fig.tight_layout()
# fig.savefig(ImgPath+"chap4_LUCF_Area-Total"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# agricultural production and demand

L=['供应-能源作物','供应-非能源作物','供应-牲畜','食物-作物','食物-牲畜','饲料-作物','其他-作物','其他-牲畜','第一代能源作物','第二代能源作物',]
L.reverse()
LUCF_AGR_order=['LUCF|AGR|Production|Energy crop',
                'LUCF|AGR|Production|Non-energy crop',
 'LUCF|AGR|Production|Livestock',
 'LUCF|AGR|Demand|Food|Crop',
 'LUCF|AGR|Demand|Food|Livestock',
 'LUCF|AGR|Demand|Feed|Crop',
 'LUCF|AGR|Demand|Industrial use|Crop',
 'LUCF|AGR|Demand|Industrial use|Livestock',
 'LUCF|AGR|Demand|Bioenergy|First-generation',
 'LUCF|AGR|Demand|Bioenergy|Second-generation', ]
Y=[2020,2035,2050,2060,2100]
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']

LUCF_AGR=Data.filter(variable=LUCF_AGR_order,scenario=scen_sdg)
LUCF_AGR1=LUCF_AGR.filter(variable='*Demand*').convert_unit('Mt','亿吨',0.01)
LUCF_AGR2=LUCF_AGR.filter(variable='*Production*').convert_unit('Mt','亿吨',-0.01)
LUCF_AGR=pyam.concat([LUCF_AGR1,LUCF_AGR2])


Fig6, ax6 = plt.subplots(1, 5, figsize = (8, 2.5), sharey = True)
Fig6=plt.figure(figsize = (8, 2.5))
grid = plt.GridSpec(1,17,wspace=0,hspace=0)

ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

LUCF_AGR.filter(scenario='ndc-ts').filter(year=Y[0]).filter(keep=False,year=[2031,2032]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=LUCF_AGR_order,cmap=sns.color_palette("Paired",10,as_cmap=True));
plt.xticks(label="2020年",rotation=45)
ax6[0].set_xlabel("")

add_net_values_to_bar_plot(ax6[0]) 
ax6[0].set_ylabel("农业产品消费/供应（亿吨）")
for i in range (1,5):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*4):(1+4*i)],sharey=ax6[0])
    
for i in range (1,5):
    LUCF_AGR.filter(scenario=scen_sdg).filter(year=Y[i]).filter(keep=False,year=[2031,2032]).plot.bar(stacked=True,width=0.8,x='scenario',
        title=Y[i],ax = ax6[i],legend=False,bars_order=LUCF_AGR_order,cmap=sns.color_palette("Paired",10,as_cmap=True),order=scen_sdg);
    
handles,l= ax6[4].get_legend_handles_labels()
ax6[4].legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.5),frameon=False,fontsize=12)

for j in [1,2,3,4]:
    ax6[j].set_xticklabels(S_name,rotation=45)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("农业产品消费/供应（亿吨）")
    add_net_values_to_bar_plot(ax6[j]) 

# plt.savefig(ImgPath+"chap4_LUCF_AGR"+".pdf",bbox_inches='tight',dpi=300)
plt.show()



In [ ]:
# Crop production and demand

L=['食物','饲料','其他','供应']
LUCF_AGR_order=[ 
 
 'LUCF|AGR|Demand|Food|Crop',
 'LUCF|AGR|Demand|Feed|Crop',
 'LUCF|AGR|Demand|Industrial use|Crop',
 'LUCF|AGR|Production|Non-energy crop']
Y=[2020,2035,2060,2100]
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']

LUCF_AGR=Data.filter(variable=LUCF_AGR_order,scenario=scen_sdg)
LUCF_AGR1=LUCF_AGR.filter(variable='*Demand*').convert_unit('Mt','亿吨',0.01)
LUCF_AGR2=LUCF_AGR.filter(variable='*Production*').convert_unit('Mt','亿吨',-0.01)
LUCF_AGR=pyam.concat([LUCF_AGR1,LUCF_AGR2])


Fig6, ax6 = plt.subplots(1, 4, figsize = (6, 2.5), sharey = True)
Fig6=plt.figure(figsize = (6, 2.5))
grid = plt.GridSpec(1,13,wspace=0,hspace=0)

ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

LUCF_AGR.filter(scenario='ndc-ts').filter(year=Y[0]).filter(keep=False,year=[2031,2032]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=LUCF_AGR_order,cmap=sns.color_palette("Paired",10,as_cmap=True));
plt.xticks(label="2020年",rotation=45)
ax6[0].set_xlabel("")

add_net_values_to_bar_plot(ax6[0]) 
ax6[0].set_ylabel("作物消费/供应（亿吨）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*4):(1+4*i)],sharey=ax6[0])
    
for i in range (1,4):
    LUCF_AGR.filter(scenario=scen_sdg).filter(year=Y[i]).filter(keep=False,year=[2031,2032]).plot.bar(stacked=True,width=0.8,x='scenario',
        title=Y[i],ax = ax6[i],legend=False,bars_order=LUCF_AGR_order,cmap=sns.color_palette("Paired",10,as_cmap=True),order=scen_sdg);
    

for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=45)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("作物消费/供应（亿吨）")
    add_net_values_to_bar_plot(ax6[j]) 

# plt.savefig(ImgPath+"chap4_LUCF_AGR-Crop"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
# Livestock production and demand

L=['食物-牲畜','其他-牲畜','供应-牲畜']
LUCF_AGR_order=[ 
 'LUCF|AGR|Demand|Food|Livestock',
 'LUCF|AGR|Demand|Industrial use|Livestock', 
 'LUCF|AGR|Production|Livestock']
Y=[2020,2035,2060,2100]
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']

LUCF_AGR=Data.filter(variable=LUCF_AGR_order,scenario=scen_sdg)
LUCF_AGR1=LUCF_AGR.filter(variable='*Demand*').convert_unit('Mt','亿吨',0.01)
LUCF_AGR2=LUCF_AGR.filter(variable='*Production*').convert_unit('Mt','亿吨',-0.01)
LUCF_AGR=pyam.concat([LUCF_AGR1,LUCF_AGR2])


Fig6, ax6 = plt.subplots(1, 4, figsize = (6, 2.5), sharey = True)
Fig6=plt.figure(figsize = (6, 2.5))
grid = plt.GridSpec(1,13,wspace=0,hspace=0)

ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

LUCF_AGR.filter(scenario='ndc-ts').filter(year=Y[0]).filter(keep=False,year=[2031,2032]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=LUCF_AGR_order,cmap=sns.color_palette("Paired",10,as_cmap=True));
plt.xticks(label="2020年",rotation=45)
ax6[0].set_xlabel("")

add_net_values_to_bar_plot(ax6[0]) 
ax6[0].set_ylabel("牲畜消费/供应（亿吨）")
for i in range (1,4):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*4):(1+4*i)],sharey=ax6[0])
    
for i in range (1,4):
    LUCF_AGR.filter(scenario=scen_sdg).filter(year=Y[i]).filter(keep=False,year=[2031,2032]).plot.bar(stacked=True,width=0.8,x='scenario',
        title=Y[i],ax = ax6[i],legend=False,bars_order=LUCF_AGR_order,cmap=sns.color_palette("Paired",10,as_cmap=True),order=scen_sdg);
    

for j in [1,2,3]:
    ax6[j].set_xticklabels(S_name,rotation=45)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("牲畜消费/供应（亿吨）")
    add_net_values_to_bar_plot(ax6[j]) 

# plt.savefig(ImgPath+"chap4_LUCF_AGR-Livestock"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
# 生物质来源
Biomass=Data.filter(variable='Primary Energy|Biomass|*').filter(keep=False,variable='*CCS').convert_unit("PJ","亿吨标煤",1/0.293076/10000)
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
Y=[2020,2035,2050,2060,2100]
L=['农业剩余','薪材砍伐','圆木收割','采伐剩余','林业剩余','其他剩余','能源作物']
Biomass_order=['Primary Energy|Biomass|Energy crop',
         'Primary Energy|Biomass|Other Source',
         'Primary Energy|Biomass|Forest industry residues',
         'Primary Energy|Biomass|Logging residues',         
         'Primary Energy|Biomass|Roundwood harvest',
         'Primary Energy|Biomass|Fuelwood',
              'Primary Energy|Biomass|Agricultural residues']
Biomass_order.reverse()
L.reverse()
Fig6, ax6 = plt.subplots(1, 5, figsize = (8, 2.5), sharey = True)
Fig6=plt.figure(figsize = (8, 2.5))
grid = plt.GridSpec(1,17,wspace=0,hspace=0)

ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

Biomass.filter(scenario='ndc-ts').filter(year=Y[0]).filter(keep=False,year=[2031,2032]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=Biomass_order,cmap=sns.color_palette("Set2_r",as_cmap=True));
plt.xticks(label="2020年",rotation=45)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("生物质能源（亿吨标煤）")
for i in range (1,5):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*4):(1+4*i)],sharey=ax6[0])

for i in range (1,5):
    Biomass.filter(scenario=scen_sdg).filter(year=Y[i]).filter(keep=False,year=[2031,2032]).plot.bar(stacked=True,width=0.8,x='scenario',
        title=Y[i],ax = ax6[i],legend=False,bars_order=Biomass_order,cmap=sns.color_palette("Set2_r",as_cmap=True),order=scen_sdg);
for j in [1,2,3,4]:
    ax6[j].set_xticklabels(S_name,rotation=45)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("生物质能源（亿吨标煤）")
handles,l= ax6[4].get_legend_handles_labels()
ax6[4].legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.5),frameon=False,fontsize=12,labelspacing=0.3)

# plt.savefig(ImgPath+"chap4_LUCF_Biomass——total"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:

# 每日膳食量
bar_order=[
    'LUCF|FOOD|CROP',
    'LUCF|FOOD|LIVESTOCK']
L=['牲畜',"作物"]
bar_order.reverse()
FOOD=Data.filter(variable=bar_order)
scen=['ndc-ts','ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
Y=[2020,2035,2060,2085,2100]
Fig6, ax6 = plt.subplots(1, 5, figsize = (8, 2.5), sharey = True)
Fig6=plt.figure(figsize = (8, 2.5))
grid = plt.GridSpec(1,17,wspace=0,hspace=0)
ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])
L.reverse()
FOOD.filter(scenario='ndc-ts').filter(year=Y[0]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True));
plt.xticks(label="2020年",rotation=45)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("每日膳食量（千卡/人·日）")
for i in range (1,5):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*4):(1+4*i)],sharey=ax6[0])
    FOOD.filter(scenario=scen).filter(year=Y[i]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=bar_order,cmap=sns.color_palette("Set2",as_cmap=True),order=scen);
    
for j in [1,2,3,4]:
    ax6[j].set_xticklabels(S_name,rotation=45)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("每日膳食量（千卡/人·日）")

handles,l= ax6[4].get_legend_handles_labels()
ax6[4].legend(handles=reversed(handles),labels=L,loc="center left", bbox_to_anchor=(1, 0.5),frameon=False,fontsize=12,columnspacing=1)   
plt.tight_layout()
# plt.savefig(ImgPath+"chap4_FoodType"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# LUCF-price Food
LUCF=Data.filter(variable="Price*").filter(keep=False,variable="*Biomass")
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
Y=[2020,2025,2035,2050,2060]
g=LUCF.filter(scenario=scen_sdg,year=Y).normalize(year=2020).as_pandas()
g.replace(scen_sdg,S_name,inplace=True)
g.replace('Price|Crop','谷物',inplace=True)
g.replace('Price|Food','食物',inplace=True)
g.rename(columns={'scenario':'情景'},inplace=True)
g1=sns.catplot(g,kind='point',dodge=False,x='year',y='value',col='variable',height=3,aspect=1,hue="情景",palette='Paired',hue_order=S_name,legend=False);
g1.refline(y=1,linestyle=':')
g1.set_ylabels("比值")
g1.set_xlabels("")
g1.set_titles("{col_name}")
g1.tight_layout()
# g1.savefig(ImgPath+"chap4_LUCF_FoodPrice"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
# LUCF-price biomass
LUCF=Data.filter(variable="Price*").filter(variable="*Biomass")
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
Y=[2020,2025,2035,2050,2060]
g=LUCF.filter(scenario=scen_sdg,year=Y).filter(keep=False,year=[2031,2032]).as_pandas()
g.replace(scen_sdg,S_name,inplace=True)
g.rename(columns={'scenario':'情景'},inplace=True)
g1=sns.catplot(g,kind='point',dodge=True,x='year',y='value',height=3,aspect=1.1,hue="情景",palette='Paired',hue_order=S_name);
g1.set_ylabels("生物质价格（美元/GJ）")
g1.set_xlabels("")
g1.set_titles("")
g1.tight_layout()
# g1.savefig(ImgPath+"chap4_LUCF_BiomassPrice"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


In [ ]:
# Water -Demand withdrawal AGR

H2O_D3=Data.filter(variable="Water Demand|Water Withdrawal|AGR|*")
L=['漫灌','渠灌','管灌','喷灌','微灌','滴灌']
H2O_order=['Water Demand|Water Withdrawal|AGR|Drip',
           'Water Demand|Water Withdrawal|AGR|Micro',
    'Water Demand|Water Withdrawal|AGR|Sprinkler',
    'Water Demand|Water Withdrawal|AGR|Pipeline',
    'Water Demand|Water Withdrawal|AGR|Channel',
    'Water Demand|Water Withdrawal|AGR|Diffuse']


ss1=['ndc-ts',
 'ndc-ts-sdg',
 'ndc-ts_2c-cn60',
     'ndc-ts-sdg_2c-cn60',]
S_name=['REF','REF-SDG','CN60','CN60-SDG']

Figt2, axt2 = plt.subplots(1, 4, figsize = (8, 4), sharey = True)

for i in range (0,4):
    H2O_D3.filter(scenario=ss1[i]).filter(keep=False,year=[2045,2031,2032]).plot.stack(legend=False,title=S_name[i],ax = axt2[i],order=H2O_order,cmap=sns.color_palette("GnBu_d",6,as_cmap=True))
    axt2[i].set_xlabel('时间')
    axt2[i].set_ylabel('')
axt2[0].set_ylabel('灌溉量（亿立方米）')
plt.tight_layout()
# plt.savefig(ImgPath+"chap4_H2O_IRR"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


# Water -Demand Withdrawal AGR total
H2O_D3=Data.filter(variable="Water Demand|Water Withdrawal|AGR|*")
L=['漫灌','渠灌','管灌','喷灌','微灌','滴灌']
H2O_order=['Water Demand|Water Withdrawal|AGR|Drip',
    'Water Demand|Water Withdrawal|AGR|Micro',
    'Water Demand|Water Withdrawal|AGR|Sprinkler',
    'Water Demand|Water Withdrawal|AGR|Pipeline',
    'Water Demand|Water Withdrawal|AGR|Channel',
    'Water Demand|Water Withdrawal|AGR|Diffuse']

scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']

Y=[2020,2035,2050,2060,2085,2100]


Fig6, ax6 = plt.subplots(1, 6, figsize = (10, 4), sharey = True)
Fig6=plt.figure(figsize = (10, 4))
grid = plt.GridSpec(1,21,wspace=0,hspace=0)

ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

H2O_D3.filter(scenario='ndc-ts').filter(year=Y[0]).filter(keep=False,year=[2031,2032]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=H2O_order.reverse(),cmap=sns.color_palette("GnBu_r_d",6,as_cmap=True));
plt.xticks(label="2020年",rotation=45)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("灌溉量（亿立方米）")
for i in range (1,6):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*4):(1+4*i)],sharey=ax6[0])

for i in range (1,6):
    H2O_D3.filter(scenario=scen_sdg).filter(year=Y[i]).filter(keep=False,year=[2031,2032]).plot.bar(stacked=True,width=0.8,x='scenario',
        title=Y[i],ax = ax6[i],legend=False,bars_order=H2O_order.reverse(),cmap=sns.color_palette("GnBu_r_d",6,as_cmap=True),order=scen_sdg);
for j in [1,2,3,4,5]:
    ax6[j].set_xticklabels(S_name,rotation=45)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("灌溉量（亿立方米）")

plt.legend(L,ncol=6,bbox_to_anchor=(0.1, -0.2),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()

# plt.savefig(ImgPath+"chap4_H2O_IRR-Total"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Electricity capacity cooling type
ELC_cap_cooling=Data.filter(variable="Capacity|Electricity|*",level="1-").filter(variable=["*Once*","*Recir*","*Air*"])
ELC_cap_cooling_order=['Capacity|Electricity|Once-Through','Capacity|Electricity|Air-cooling','Capacity|Electricity|Recirculating']
ELC_cap_cooling_order.reverse()
L=["一次直流","空气冷却","循环水冷"]
L.reverse()
ss1=['ndc-ts',
 'ndc-ts-sdg',
 'ndc-ts_2c-cn60',
     'ndc-ts-sdg_2c-cn60',]
S_name=['REF','REF-SDG','CN60','CN60-SDG']

Figt2, axt2 = plt.subplots(1, 4, figsize = (8, 4), sharey = True)

for i in range (0,4):
    ELC_cap_cooling.filter(scenario=ss1[i]).filter(keep=False,year=[2031,2032]).plot.stack(legend=False,title=S_name[i],ax = axt2[i],order=ELC_cap_cooling_order,cmap=sns.color_palette("tab20c_r",as_cmap=True))
    axt2[i].set_xlabel('时间')
    axt2[i].set_ylabel('')
    
axt2[0].set_ylabel('电力装机容量（GW）')
# handles,l= axt2[3].get_legend_handles_labels()
# axt2[3].legend(handles=handles,labels=L,ncol=6,loc='upper center',bbox_to_anchor=(-1.5,-0.15),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()

# plt.savefig(ImgPath+"chap4_ELC_CAP_COOL"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# Electricity capacity cooling type -total
ELC_cap_cooling=Data.filter(variable="Capacity|Electricity|*",level="1-").filter(variable=["*Once*","*Recir*","*Air*"])
ELC_cap_cooling_order=['Capacity|Electricity|Once-Through','Capacity|Electricity|Air-cooling','Capacity|Electricity|Recirculating']
ELC_cap_cooling_order.reverse()
L=["一次直流","空气冷却","循环水冷"]
L.reverse()
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
Y=[2020,2035,2050,2060,2085,2100]

Fig6, ax6 = plt.subplots(1, 6, figsize = (10, 4), sharey = True)
Fig6=plt.figure(figsize = (10, 4))
grid = plt.GridSpec(1,21,wspace=0,hspace=0)

ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

ELC_cap_cooling.filter(scenario='ndc-ts').filter(year=Y[0]).filter(keep=False,year=[2031,2032]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=ELC_cap_cooling_order,cmap=sns.color_palette("Set2_r",as_cmap=True));
plt.xticks(label="2020年",rotation=45)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("火电和核电装机容量（GW）")
for i in range (1,6):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*4):(1+4*i)],sharey=ax6[0])
    ELC_cap_cooling.filter(scenario=scen_sdg).filter(year=Y[i]).filter(keep=False,year=[2031,2032]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=ELC_cap_cooling_order,cmap=sns.color_palette("Set2_r",as_cmap=True),order=scen_sdg);
for j in [1,2,3,4,5]:
    ax6[j].set_xticklabels(S_name,rotation=45)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("火电和核电装机容量（GW）")
plt.legend(L,ncol=6,bbox_to_anchor=(-0.3, -0.2),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()

# plt.savefig(ImgPath+"chap4_ELC_CAP_COOL-Total"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# Water -Demand
H2O_D=Data.filter(variable="Water Demand|Water Withdrawal|*")
L=["建筑","工业","电力","上游","农业","人工生态补水"]
H2O_order=['Water Demand|Water Withdrawal|UPS|Ecoflow','Water Demand|Water Withdrawal|AGR','Water Demand|Water Withdrawal|UPS|Energy','Water Demand|Water Withdrawal|ELC','Water Demand|Water Withdrawal|IND','Water Demand|Water Withdrawal|BLD']


ss1=['ndc-ts',
 'ndc-ts-sdg',
 'ndc-ts_2c-cn60',
     'ndc-ts-sdg_2c-cn60',]
S_name=['REF','REF-SDG','CN60','CN60-SDG']

Figt2, axt2 = plt.subplots(1, 4, figsize = (8, 4), sharey = True)

for i in range (0,4):
    H2O_D.filter(scenario=ss1[i]).filter(keep=False,year=[2031,2032]).plot.stack(legend=False,title=S_name[i],ax = axt2[i],order=H2O_order,cmap=sns.color_palette("Set2",as_cmap=True))
    axt2[i].set_xlabel('时间')
    axt2[i].set_ylabel('')
axt2[0].set_ylabel('取水量（亿立方米）')
plt.tight_layout()
# plt.savefig(ImgPath+"chap4_H2O_WW"+".pdf",bbox_inches='tight',dpi=300)
plt.show()


# Water -Demand Withdrawal with AGR -total
H2O_D=Data.filter(variable="Water Demand|Water Withdrawal|*")
L=["建筑","工业","电力","上游","农业","人工生态补水"]
H2O_order=['Water Demand|Water Withdrawal|UPS|Ecoflow','Water Demand|Water Withdrawal|AGR','Water Demand|Water Withdrawal|UPS|Energy','Water Demand|Water Withdrawal|ELC','Water Demand|Water Withdrawal|IND','Water Demand|Water Withdrawal|BLD']
H2O_order.reverse()
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
Y=[2020,2035,2050,2060,2085,2100]

Fig6, ax6 = plt.subplots(1, 6, figsize = (10, 4), sharey = True)
Fig6=plt.figure(figsize = (10, 4))
grid = plt.GridSpec(1,21,wspace=0,hspace=0)

ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

H2O_D.filter(scenario='ndc-ts').filter(year=Y[0]).filter(keep=False,year=[2031,2032]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=H2O_order,cmap=sns.color_palette("Set2_r",as_cmap=True));
plt.xticks(label="2020年",rotation=45)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("取水量（亿立方米）")
for i in range (1,6):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*4):(1+4*i)],sharey=ax6[0])
    H2O_D.filter(scenario=scen_sdg).filter(year=Y[i]).filter(keep=False,year=[2031,2032]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=H2O_order,cmap=sns.color_palette("Set2_r",as_cmap=True),order=scen_sdg);
for j in [1,2,3,4,5]:
    ax6[j].set_xticklabels(S_name,rotation=45)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("取水量（亿立方米）")
plt.legend(L,ncol=6,bbox_to_anchor=(0.3, -0.2),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()

# plt.savefig(ImgPath+"chap4_H2O_WW-Total"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# H2O demand
H2O_s=Data_subannual.filter(variable=['BLDWW','INDWW','ELCWW','UPSWW|Energy','AGRWW','UPSWW|Ecoflow']).filter(year=2035)
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
L=["建筑","工业","电力","上游","农业","人工生态补水"]
H2O_order=['BLDWW','INDWW','ELCWW','UPSWW|Energy','AGRWW','UPSWW|Ecoflow']

Figt1, axt1 = plt.subplots(1, 4, figsize = (6, 3), sharey = True)
for i in range (0,4):
    Figt1=H2O_s.filter(scenario=scen_sdg[i]).filter(keep=False,variable=["H2O*"]).plot.bar(width=0.8,
        stacked=True,x="timeslice",y='value',bars="variable",title=S_name[i],ax = axt1[i],legend=False,order=['R','S','F','W'],bars_order=H2O_order,
        cmap=sns.color_palette("Set2_r",as_cmap=True))
    axt1[i].set_xticklabels(['春','夏','秋','冬'],rotation=0)
    axt1[i].set_xlabel("季节")
    axt1[i].set_ylabel("取水量（亿立方米）")
axt1[1].legend(L,ncol=3,loc='upper center',bbox_to_anchor=(1, -0.2),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()

# plt.savefig(ImgPath+"chap4_H2O_WW-Season-2035"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# H2O demand
H2O_s=Data_subannual.filter(variable=['BLDWW','INDWW','ELCWW','UPSWW|Energy','AGRWW','UPSWW|Ecoflow']).filter(year=2060)
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
L=["建筑","工业","电力","上游","农业","人工生态补水"]
H2O_order=['BLDWW','INDWW','ELCWW','UPSWW|Energy','AGRWW','UPSWW|Ecoflow']

Figt1, axt1 = plt.subplots(1, 4, figsize = (6, 3), sharey = True)
for i in range (0,4):
    Figt1=H2O_s.filter(scenario=scen_sdg[i]).filter(keep=False,variable=["H2O*"]).plot.bar(width=0.8,
        stacked=True,x="timeslice",y='value',bars="variable",title=S_name[i],ax = axt1[i],legend=False,order=['R','S','F','W'],bars_order=H2O_order,
        cmap=sns.color_palette("Set2_r",as_cmap=True))
    axt1[i].set_xticklabels(['春','夏','秋','冬'],rotation=0)
    axt1[i].set_xlabel("季节")
    axt1[i].set_ylabel("取水量（亿立方米）")
axt1[1].legend(L,ncol=3,loc='upper center',bbox_to_anchor=(1, -0.2),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()

# plt.savefig(ImgPath+"chap4_H2O_WW-Season"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# SheetName=list(pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量持续曲线.xlsx',sheet_name=None).keys())

# Waterduration=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量持续曲线.xlsx',0,header=0)
# Waterduration=Waterduration.transpose()
# Waterduration.columns=range(2015,2101)
# Waterduration['model']=SheetName[0]
    
# for i in range(1,5):
#     Waterduration1=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量持续曲线.xlsx',i,header=0)
#     Waterduration1=Waterduration1.transpose()
#     Waterduration1.columns=range(2015,2101)
#     Waterduration1['model']=SheetName[i]
#     Waterduration=pd.concat([Waterduration,Waterduration1])
# Waterduration=pyam.IamDataFrame(Waterduration.reset_index(),scenario='总径流量',unit='10^8m3',region='China',variable='index')

# SheetName=list(pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量持续曲线排除生态径流.xlsx',sheet_name=None).keys())
# Waterduration2=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量持续曲线排除生态径流.xlsx',0,header=0)
# Waterduration2=Waterduration2.transpose()
# Waterduration2.columns=range(2015,2101)
# Waterduration2['model']=SheetName[0]

# for j in range(1,5):
#     Waterduration3=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量持续曲线排除生态径流.xlsx',j,header=0)
#     Waterduration3=Waterduration3.transpose()
#     Waterduration3.columns=range(2015,2101)
#     Waterduration3['model']=SheetName[j]
#     Waterduration2=pd.concat([Waterduration2,Waterduration3])
# Waterduration2=pyam.IamDataFrame(Waterduration2.reset_index(),scenario='水资源可获取量',unit='10^8m3',region='China',variable='index')
# Waterduration.append(Waterduration2,inplace=True)
# Waterduration.subtract('总径流量','水资源可获取量','生态径流',axis='scenario',ignore_units='10^8m3',append=True)

# #水资源量持续曲线图
# sns.lineplot(Waterduration.as_pandas(),x='variable',y='value',hue='scenario',errorbar='pi',legend='brief')
# plt.ylabel("水资源量（亿立方米）")
# plt.xlabel("月份（水量由大到小排列）")
# plt.legend(title='',frameon=False)
# plt.xticks(range(1,13))
# plt.tight_layout()
# # plt.savefig(ImgPath+"chap4_H2Oduration"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:
# SheetName=list(pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量.xlsx',sheet_name=None).keys())
# Waterduration=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量.xlsx',0,header=0)
# Waterduration['model']=SheetName[0]
    
# for i in range(1,5):
#     Waterduration1=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量.xlsx',i,header=0)
#     Waterduration1['model']=SheetName[i]
#     Waterduration=pd.concat([Waterduration,Waterduration1])
# Waterduration=pyam.IamDataFrame(Waterduration,scenario='总径流量',unit='亿立方米',region='China',variable='月份')

# SheetName=list(pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量排除生态径流.xlsx',sheet_name=None).keys())
# Waterduration2=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量排除生态径流.xlsx',0,header=0)
# Waterduration2['model']=SheetName[0]

# for j in range(1,5):
#     Waterduration3=pd.read_excel(r'C:\Users\44728\OneDrive\博五\博士论文\CWatM水资源总量排除生态径流.xlsx',j,header=0)
#     Waterduration3['model']=SheetName[j]
#     Waterduration2=pd.concat([Waterduration2,Waterduration3])
# Waterduration2=pyam.IamDataFrame(Waterduration2,scenario='水资源可获取量',unit='亿立方米',region='China',variable='月份')
# Waterduration.append(Waterduration2,inplace=True)
# Waterduration.subtract('总径流量','水资源可获取量','生态径流',axis='scenario',ignore_units='亿立方米',append=True)

# #水资源量图
# sns.lineplot(Waterduration.as_pandas(),x='variable',y='value',hue='scenario',errorbar='pi',legend='brief')
# plt.ylabel("水资源量（亿立方米）")
# plt.xlabel("月份")
# plt.legend(title='',frameon=False)
# plt.xticks(range(1,13))
# plt.tight_layout()
# # plt.savefig(ImgPath+"chap4_H2Omonthly"+".pdf",bbox_inches='tight',dpi=300)
# plt.show()


In [ ]:
# Water -Supply
H2O_S=Data.filter(variable="Water Supply|*",level=0)
H2O_order=['Water Supply|Waste Water','Water Supply|Salt Water','Water Supply|Surface Water','Water Supply|Ground Water']
L=["地下水","地表水","海水淡化","污水处理"]

ss1=['ndc-ts',
 'ndc-ts-sdg',
 'ndc-ts_2c-cn60',
     'ndc-ts-sdg_2c-cn60',]
S_name=['REF','REF-SDG','CN60','CN60-SDG']

Figt2, axt2 = plt.subplots(1, 4, figsize = (8, 4), sharey = True)

for i in range (0,4):
    H2O_S.filter(scenario=ss1[i]).filter(keep=False,year=[2031,2032]).plot.stack(legend=False,title=S_name[i],ax = axt2[i],order=H2O_order,cmap=sns.color_palette("tab20c_r",as_cmap=True))
    axt2[i].set_xlabel('时间')
    axt2[i].set_ylabel('')
axt2[0].set_ylabel('供水量（亿立方米）')
plt.tight_layout()
# plt.savefig(ImgPath+"chap4_H2O_Supply"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# Water Supply
H2O_S=Data.filter(variable="Water Supply|*",level=0)
H2O_order=['Water Supply|Ground Water','Water Supply|Surface Water','Water Supply|Salt Water','Water Supply|Waste Water']
L=["地下水","地表水","海水淡化","污水处理"]
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
Y=[2020,2035,2050,2060,2085,2100]

Fig6, ax6 = plt.subplots(1, 6, figsize = (10, 4), sharey = True)
Fig6=plt.figure(figsize = (10, 4))
grid = plt.GridSpec(1,21,wspace=0,hspace=0)

ax6[0]=Fig6.add_subplot(grid[0,0],sharey=ax6[0])

H2O_S.filter(scenario='ndc-ts').filter(year=Y[0]).filter(keep=False,year=[2031,2032]).rename(scenario={"ndc-ts":"2020年"}).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[0],ax = ax6[0],legend=False,bars_order=H2O_order,cmap=sns.color_palette("tab20c",as_cmap=True));
plt.xticks(label="2020年",rotation=45)
ax6[0].set_xlabel("")
ax6[0].set_ylabel("供水量（亿立方米）")

for i in range (1,6):
    ax6[i]=Fig6.add_subplot(grid[0,(1+(i-1)*4):(1+4*i)],sharey=ax6[0])
    H2O_S.filter(scenario=scen_sdg).filter(year=Y[i]).filter(keep=False,year=[2031,2032]).plot.bar(stacked=True,width=0.8,x='scenario',sharey=True,sharex=False,
        title=Y[i],ax = ax6[i],legend=False,bars_order=H2O_order,cmap=sns.color_palette("tab20c",as_cmap=True),order=scen_sdg);
for j in [1,2,3,4,5]:
    ax6[j].set_xticklabels(S_name,rotation=45)
    ax6[j].set_xlabel("")
    ax6[j].set_ylabel("供水量（亿立方米）")
plt.legend(L,ncol=4,bbox_to_anchor=(-0.25, -0.2),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()

# plt.savefig(ImgPath+"chap4_H2O_WaterSupply-Total"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
# H2O supply
H2O_s=Data_subannual.filter(variable=['H2OGFW','H2OSALT','H2OSFW','H2OWASTE'],year=2035)
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
H2O_order=['H2OGFW','H2OSFW','H2OSALT','H2OWASTE']
L=["地下水","地表水","海水淡化","污水处理"]
Figt1, axt1 = plt.subplots(1, 4, figsize = (6, 3), sharey = True)
for i in range (0,4):
    Figt1=H2O_s.filter(scenario=scen_sdg[i]).filter(keep=False,variable="H2OWW").plot.bar(width=0.8,
        stacked=True,x="timeslice",y='value',bars="variable",title=S_name[i],ax = axt1[i],legend=False,bars_order=H2O_order,order=['R','S','F','W'],
        cmap=sns.color_palette("tab20c",as_cmap=True))
    axt1[i].set_xticklabels(['春','夏','秋','冬'],rotation=0)
    axt1[i].set_xlabel("季节")
    axt1[i].set_ylabel("供水量（亿立方米）")
plt.legend(L,ncol=4,bbox_to_anchor=(1, -0.2),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()

# plt.savefig(ImgPath+"chap4_H2O_WaterSupply-Season-2035"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

# H2O supply
H2O_s=Data_subannual.filter(variable=['H2OGFW','H2OSALT','H2OSFW','H2OWASTE'],year=2060)
scen_sdg=["ndc-ts",'ndc-ts-sdg','ndc-ts_2c-cn60','ndc-ts-sdg_2c-cn60']
S_name=['REF','REF-SDG','CN60','CN60-SDG']
H2O_order=['H2OGFW','H2OSFW','H2OSALT','H2OWASTE']
L=["地下水","地表水","海水淡化","污水处理"]
Figt1, axt1 = plt.subplots(1, 4, figsize = (6, 3), sharey = True)
for i in range (0,4):
    Figt1=H2O_s.filter(scenario=scen_sdg[i]).filter(keep=False,variable="H2OWW").plot.bar(width=0.8,
        stacked=True,x="timeslice",y='value',bars="variable",title=S_name[i],ax = axt1[i],legend=False,bars_order=H2O_order,order=['R','S','F','W'],
        cmap=sns.color_palette("tab20c",as_cmap=True))
    axt1[i].set_xticklabels(['春','夏','秋','冬'],rotation=0)
    axt1[i].set_xlabel("季节")
    axt1[i].set_ylabel("供水量（亿立方米）")
plt.legend(L,ncol=4,bbox_to_anchor=(1, -0.2),frameon=False,fontsize=12,columnspacing=1)
plt.tight_layout()

# plt.savefig(ImgPath+"chap4_H2O_WaterSupply-Season"+".pdf",bbox_inches='tight',dpi=300)
plt.show()

In [ ]:
#排放量
ByRegion.replace({"污染物":{"SO2":"SO$\mathregular{_2}$","NOx":"NO$\mathregular{_X}$","PM10":"PM$\mathregular{_{10}}$","PM2.5":"PM$\mathregular{_{2.5}}$"}},inplace=True)
ByRegion.replace({"情景":{"NDC":"REF"}},inplace=True)

g=sns.FacetGrid(ByRegion,col="污染物",col_wrap=4,col_order=["SO$\mathregular{_2}$","NO$\mathregular{_X}$","PM$\mathregular{_{10}}$","PM$\mathregular{_{2.5}}$"],aspect=0.8,sharex=False)
g.map_dataframe(sns.pointplot, x='情景',y='value',hue='时间',dodge=False,palette="flare",markers='.').add_legend()
g.set_ylabels('排放量（百万吨）')
g.set_xlabels('')
g.set_titles("{col_name}")
g.tight_layout()
# g.savefig(ImgPath+"chap4_GAINSemissions"+".pdf",bbox_inches='tight',dpi=300)


In [ ]:
# temp=pyam.IamDataFrame(r'C:\Users\44728\OneDrive\博五\博士论文\gains\EMISSIONS\byBigSector.xlsx',region='China')
# temp.aggregate('交通部门',['客车','货车','工程车辆','车辆磨损'],append=True)
# temp.aggregate('工业部门',['工业过程排放','工业部门燃烧','废弃物','燃料生产和分配','燃料转换'],append=True)
# temp.aggregate('农业部门',['农业'],append=True)
# temp.aggregate('建筑部门',['居民部门燃烧'],append=True)
# temp.convert_unit('kt','Mt',inplace=True)
# #分部门排放
# Y=[2020,2030,2050]
# Fig7, ax7 = plt.subplots(3, 4, figsize = (14, 3))
# Fig7=plt.figure(figsize = (14, 3))
# grid = plt.GridSpec(7,4,wspace=0.1,hspace=0)
# E=['SO2','NOx','PM10','PM2.5']
# for i in range (0,4):
#     ax7[0][i]=Fig7.add_subplot(grid[0,i],sharex=ax7[0][i],sharey=ax7[0][0])
#     temp.filter(variable=['电力和热力','工业部门','建筑部门','交通部门']).filter(year=2020,scenario=['NDC']).filter(emissions=E[i]).rename(scenario={"NDC":"2020年水平"}).plot.bar(stacked =True,y='value',x='scenario',bars='variable',width=0.8,
#                                                 bars_order=['电力和热力','工业部门','建筑部门','交通部门'],orient='h',legend=False,ax=ax7[0][i],title=False,cmap=sns.color_palette("Set2_r",as_cmap=True))


# for j in range (1,3):
#     for i in range (0,4):  
#         ax7[j][i]=Fig7.add_subplot(grid[(1+3*(j-1)):(1+3*j),i],sharex=ax7[0][i],sharey=ax7[j][0])
#         temp.filter(variable=['电力和热力','工业部门','建筑部门','交通部门']).filter(keep=False,year=2020,scenario=['CN60','CN60-SDG']).rename(scenario={"NDC":"REF"}).filter(emissions=E[i],year=Y[j]).plot.bar(stacked =True,y='value',x='scenario',bars='variable',order=['REF','CN60','CN60-SDG'],width=0.8,
#                                                 bars_order=['电力和热力','工业部门','建筑部门','交通部门'],orient='h',legend=False,ax=ax7[j][i],title=False,cmap=sns.color_palette("Set2_r",as_cmap=True))

# ax7[2][0].set_xlabel("SO$\mathregular{_2}$排放量（百万吨）")
# ax7[2][1].set_xlabel("NO$\mathregular{_X}$排放量（百万吨）")
# ax7[2][2].set_xlabel("PM$\mathregular{_{10}}$排放量（百万吨）")
# ax7[2][3].set_xlabel("PM$\mathregular{_{2.5}}$排放量（百万吨）")
# ax7[0][0].set_ylabel("")
# ax7[1][0].set_ylabel("2030年")
# ax7[2][0].set_ylabel("2050年")
# ax7[0][0].yaxis.set_minor_locator(plt.NullLocator())
# ax7[1][0].yaxis.set_minor_locator(plt.NullLocator())
# ax7[2][0].yaxis.set_minor_locator(plt.NullLocator())


# plt.legend(ncol=4,bbox_to_anchor=(0, -0.5),frameon=False,fontsize=12,columnspacing=1)
# plt.savefig(ImgPath+"chap4_GAINSemissions_sector"+".pdf",bbox_inches='tight',dpi=300)


# plt.show()         

In [ ]:
# ByRegion=pd.read_excel(r"C:\Users\44728\OneDrive\博五\博士论文\gains\EMISSIONS\ByRegion.xlsx")
# ByRegion.drop(2015,axis=1,inplace=True)
# ByRegion=ByRegion[ByRegion['scenario']!='CN60-CLE']
# ByRegion.replace("CN50-MFR","CN60-SDG",inplace=True)
# ByRegion.replace("CN50-CLE","CN60",inplace=True)
# ByRegion=ByRegion.melt(id_vars=['scenario','Emissions(kt)','region'],var_name='时间',value_name='value')
# ByRegion.value=ByRegion.value/1000
# ByRegion=ByRegion[ByRegion['region']!='Sum']
# ggg=pyam.IamDataFrame(ByRegion,year='时间',model='C',variable='Emissions(kt)',unit='Mt')
# ggg.aggregate_region(['SO2','NOx','PM10','PM2.5'],'京津冀地区',['北京','天津','河北'],append=True)
# ggg.aggregate_region(['SO2','NOx','PM10','PM2.5'],'长三角地区',['上海','江苏','浙江','安徽'],append=True)
# ggg.aggregate_region(['SO2','NOx','PM10','PM2.5'],'成渝地区',['四川','重庆'],append=True)
# ggg.aggregate_region(['SO2','NOx','PM10','PM2.5'],'粤港澳大湾区',['广东','香港澳门'],append=True)
# gg=ggg.filter(region=['京津冀地区','长三角地区','成渝地区','粤港澳大湾区'],year=[2020,2030,2050])
# #分部门排放
# Y=[2020,2030,2050]
# Fig7, ax7 = plt.subplots(3, 4, figsize = (14, 4))
# Fig7=plt.figure(figsize = (14, 4))
# grid = plt.GridSpec(8,4,wspace=0.1,hspace=0)
# gg.rename(variable={"SO2":"SO$\mathregular{_2}$","NOx":"NO$\mathregular{_X}$","PM10":"PM$\mathregular{_{10}}$","PM2.5":"PM$\mathregular{_{2.5}}$"},inplace=True)
# E=["SO$\mathregular{_2}$","NO$\mathregular{_X}$","PM$\mathregular{_{10}}$","PM$\mathregular{_{2.5}}$"]
# R=['京津冀地区','长三角地区','粤港澳大湾区','成渝地区']
# for i in range (0,4):
#     ax7[0][i]=Fig7.add_subplot(grid[0:2,i],sharex=ax7[0][i],sharey=ax7[0][0])
#     gg.filter(year=2020,scenario=['NDC']).filter(region=R[i]).rename(scenario={"NDC":"2020年水平"}).plot.bar(y='value',x='variable',width=0.9,bars='scenario',order=E,
#                                                 orient='h',legend=False,ax=ax7[0][i],title=False,cmap=sns.color_palette("Set2_r",as_cmap=True))


# for j in range (1,3):
#     for i in range (0,4):  
#         ax7[j][i]=Fig7.add_subplot(grid[(2+3*(j-1)):(2+3*j),i],sharex=ax7[0][i],sharey=ax7[j][0])
#         gg.filter(keep=False,year=2020,scenario=['CN60','CN60-SDG']).filter(region=R[i],year=Y[j]).rename(scenario={"NDC":"REF"}).plot.bar(y='value',x='variable',bars='scenario',bars_order=['REF','CN60','CN60-SDG'],width=0.9,order=E,
#                                                                             orient='h',legend=False,ax=ax7[j][i],title=False,cmap=sns.color_palette("Set2_r",as_cmap=True))

# ax7[2][0].set_xlabel("京津冀地区（百万吨）")
# ax7[2][1].set_xlabel("长三角地区（百万吨）")
# ax7[2][2].set_xlabel("粤港澳大湾区（百万吨）")
# ax7[2][3].set_xlabel("成渝地区（百万吨）")
# ax7[0][0].set_ylabel("2020年")
# ax7[1][0].set_ylabel("2030年")
# ax7[2][0].set_ylabel("2050年")
# ax7[0][0].yaxis.set_minor_locator(plt.NullLocator())
# ax7[1][0].yaxis.set_minor_locator(plt.NullLocator())
# ax7[2][0].yaxis.set_minor_locator(plt.NullLocator())


# plt.legend(ncol=4,bbox_to_anchor=(-0.5, -0.6),frameon=False,fontsize=12,columnspacing=1)
# plt.savefig(ImgPath+"chap4_GAINSemissions_region"+".pdf",bbox_inches='tight',dpi=300)


# plt.show()

In [ ]:
#PM2.5分省浓度
PMconcentration=HealthImpact[HealthImpact['Variable']=='Mean PM2.5Concentration']
PMconcentration.replace({"情景":{"NDC":"REF"}},inplace=True)
fig=sns.catplot(PMconcentration,kind='strip',x='地区',row='情景',y='value',hue='时间',palette="Spectral",height=4,aspect=4,sharex=False,dodge=True,size=9,row_order=['REF','CN60','CN60-SDG'])
fig.refline(y=15,linestyle=':')
fig.refline(y=35,linestyle=':')
fig.set_ylabels("PM$\mathregular{_{2.5}}$浓度",size=18)
fig.set_xlabels("")
fig.set_xticklabels(size=16,rotation=30)
fig.set_yticklabels(size=16)
fig.set_titles("{row_name}",size=20)
# fig.savefig(ImgPath+"chap4_GAINSPM2.5"+".pdf",bbox_inches='tight',dpi=300)


#健康效应
Data_GAINS.replace({"情景":{"NDC":"REF"}},inplace=True)
fig=sns.catplot(Data_GAINS,kind='bar',x='地区',row='情景',y='value',hue='时间',palette="flare",height=4,aspect=4,sharex=False,dodge=True,row_order=['REF','CN60','CN60-SDG'],order=Province_order)
fig.set_ylabels("每万人死亡人数",size=18)
fig.set_xlabels("")
fig.set_xticklabels(size=16,rotation=30)
fig.set_yticklabels(size=16)
fig.set_titles("{row_name}",size=20)
fig.tight_layout()
# fig.savefig(ImgPath+"chap4_Death"+".pdf",bbox_inches='tight',dpi=300)

In [ ]:
#协同效应分析（全国）

figg, axg = plt.subplots()
figg=Cost.filter(scenario='协同效应').filter(variable=['净收益']).filter(region='Sum').filter(keep=False,year=2020).rename(variable={'收益':'损失减少'}).plot.bar(
    ax=axg,x='year',y='value',cmap='Set1_r',width=0.8,legend=False,title='温室气体减排协同效应')
axg.set_xticklabels([2025,2030,2035,2040,2045,2050],rotation=0)
axg.set_xlabel("")
axg.set_ylabel("净收益（十亿美元）")
plt.tight_layout()
# plt.savefig(ImgPath+"chap4_GAINS_cobenefit"+".pdf",bbox_inches='tight',dpi=300)

In [ ]:
#控制措施成本效益分析（全国）

figg, axg = plt.subplots()
figg=Cost.filter(scenario='控制措施').filter(year=2020,keep=False).filter(variable=['收益','成本增加']).filter(region='Sum').rename(variable={'收益':'损失减少'}).plot.bar(
    ax=axg,stacked=True,x='year',y='value',bars='variable',cmap='Set3',width=0.8,title="控制措施成本收益分析",legend=False)
add_net_values_to_bar_plot(axg)
axg.set_xlabel("")
axg.set_xticklabels([2025,2030,2035,2040,2045,2050],rotation=0)
axg.set_ylabel("收益/成本（十亿美元）")
plt.legend(loc="upper left",frameon=False)
plt.tight_layout()
# plt.savefig(ImgPath+"chap4_GAINS_cost"+".pdf",bbox_inches='tight',dpi=300)

In [ ]:
#协同效应分析（地区）
figg, axg = plt.subplots(1,6,figsize=(17,7),sharey=True)

Y=[2025,2030,2035,2040,2045,2050]
for i in range (0,6):
    Figg=Cost.filter(scenario='协同效应').filter(year=Y[i]).filter(variable=['人均净收益']).filter(keep=False,region=['Sum','香港澳门']).convert_unit('千美元/人','美元/人',1000).plot.bar(
    ax=axg[i],x='region',y='value',cmap='Set1_r',width=0.8,legend=False,title=False,orient='h',order=Province_order)
    axg[i].set_xlabel("人均净收益（美元/人）")
    axg[i].set_ylabel("")
    axg[i].set_title(Y[i])
plt.tight_layout()
# plt.savefig(ImgPath+"chap4_GAINS_cobenefit_regional"+".pdf",bbox_inches='tight',dpi=300)

In [ ]:
#控制措施成本效益分析（地区）
figg, axg = plt.subplots(1,6,figsize=(15,6),sharey=True)
for i in range (0,6):
    Figg=Cost.filter(scenario='控制措施').filter(year=Y[i]).filter(variable=['人均收益','人均成本增加']).filter(keep=False,region=['Sum','香港澳门']).rename(variable={'人均收益':'人均损失减少'}).convert_unit('千美元/人','美元/人',1000).plot.bar(
    ax=axg[i],stacked=True,x='region',y='value',bars='variable',cmap='Set3',width=0.8,title=False,legend=False,orient='h',order=Province_order)
    axg[i].set_xlabel("人均收益/成本（美元/人）")
    axg[i].set_ylabel("")
    axg[i].set_title(Y[i])
plt.legend(['成本增加','损失减少'],loc="upper right",frameon=False,labelspacing=0.3,bbox_to_anchor=(1, 0.98))
plt.tight_layout()
# plt.savefig(ImgPath+"chap4_GAINS_cost_regional"+".pdf",bbox_inches='tight',dpi=300)